In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# UAV CYBER-PHYSICAL PROJECT — INITIAL DATASET AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import shutil


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SOURCE_FILE = PROJECT_DIR / "Dataset_T-ITS.csv"

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
SRC_DIR = PROJECT_DIR / "src"
MODELS_DIR = PROJECT_DIR / "models"
FIGURES_DIR = PROJECT_DIR / "results" / "figures"
TABLES_DIR = PROJECT_DIR / "results" / "tables"

for folder in [
    RAW_DIR,
    PROCESSED_DIR,
    NOTEBOOK_DIR,
    SRC_DIR,
    MODELS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Project directories ready.")


# ============================================================
# 2. COPY DATASET INTO data/raw
# ============================================================

RAW_FILE = RAW_DIR / "Dataset_T-ITS.csv"

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{SOURCE_FILE}"
    )

if not RAW_FILE.exists():
    shutil.copy2(SOURCE_FILE, RAW_FILE)
    print("✅ Dataset copied to:")
    print(RAW_FILE)
else:
    print("✅ Dataset already exists in data/raw:")
    print(RAW_FILE)


# ============================================================
# 3. LOAD DATASET
# ============================================================

print("\n" + "=" * 100)
print("LOADING DATASET")
print("=" * 100)

df = pd.read_csv(
    RAW_FILE,
    low_memory=False
)

print("✅ Dataset loaded successfully.")


# ============================================================
# 4. BASIC SHAPE
# ============================================================

print("\n" + "=" * 100)
print("BASIC DATASET INFORMATION")
print("=" * 100)

print("Rows   :", df.shape[0])
print("Columns:", df.shape[1])

memory_mb = df.memory_usage(
    deep=True
).sum() / (1024 ** 2)

print(f"Memory : {memory_mb:.2f} MB")


# ============================================================
# 5. COLUMN NAMES
# ============================================================

print("\n" + "=" * 100)
print("COLUMN NAMES")
print("=" * 100)

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {repr(col)}")


# ============================================================
# 6. DATA TYPES
# ============================================================

print("\n" + "=" * 100)
print("DATA TYPES")
print("=" * 100)

dtype_table = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(x) for x in df.dtypes],
    "unique_values": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ]
})

print(dtype_table.to_string(index=False))


# ============================================================
# 7. MISSING VALUES
# ============================================================

print("\n" + "=" * 100)
print("MISSING VALUES")
print("=" * 100)

missing = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
})

missing["missing_percent"] = (
    missing["missing_count"]
    / len(df)
    * 100
)

missing = missing.sort_values(
    "missing_count",
    ascending=False
)

missing_nonzero = missing[
    missing["missing_count"] > 0
]

if len(missing_nonzero) == 0:
    print("✅ No missing values detected.")
else:
    print(
        missing_nonzero.to_string(
            index=False
        )
    )


# ============================================================
# 8. DUPLICATES
# ============================================================

print("\n" + "=" * 100)
print("DUPLICATE AUDIT")
print("=" * 100)

exact_duplicates = df.duplicated().sum()

print(
    "Exact duplicate rows:",
    exact_duplicates
)

print(
    "Exact duplicate percentage:",
    f"{exact_duplicates / len(df) * 100:.4f}%"
)


# ============================================================
# 9. INFINITE VALUES IN NUMERIC COLUMNS
# ============================================================

print("\n" + "=" * 100)
print("INFINITE VALUES")
print("=" * 100)

numeric_df = df.select_dtypes(
    include=[np.number]
)

if numeric_df.shape[1] > 0:

    inf_counts = np.isinf(
        numeric_df
    ).sum()

    inf_counts = inf_counts[
        inf_counts > 0
    ]

    if len(inf_counts) == 0:
        print("✅ No +/- infinity values detected.")
    else:
        print(inf_counts)

else:
    print("No numeric columns detected.")


# ============================================================
# 10. POSSIBLE LABEL / ATTACK / TYPE COLUMNS
# ============================================================

print("\n" + "=" * 100)
print("POSSIBLE LABEL / ATTACK COLUMNS")
print("=" * 100)

keywords = [
    "label",
    "class",
    "attack",
    "type",
    "category",
    "target",
    "state",
    "status",
]

candidate_label_columns = [
    col
    for col in df.columns
    if any(
        keyword in str(col).lower()
        for keyword in keywords
    )
]

if not candidate_label_columns:
    print(
        "⚠️ No obvious label column found by name."
    )

else:

    for col in candidate_label_columns:

        print("\nCOLUMN:", repr(col))
        print("-" * 70)

        print(
            df[col]
            .value_counts(
                dropna=False
            )
            .head(30)
        )


# ============================================================
# 11. LOW-CARDINALITY COLUMNS
# Useful for detecting hidden labels/type indicators
# ============================================================

print("\n" + "=" * 100)
print("LOW-CARDINALITY COLUMNS (<= 20 UNIQUE VALUES)")
print("=" * 100)

low_cardinality = []

for col in df.columns:

    n_unique = df[col].nunique(
        dropna=True
    )

    if n_unique <= 20:

        low_cardinality.append(
            (
                col,
                n_unique,
                df[col]
                .drop_duplicates()
                .head(20)
                .tolist()
            )
        )

for col, n_unique, values in low_cardinality:

    print(
        f"\n{repr(col)}"
        f" | unique={n_unique}"
    )

    print(values)


# ============================================================
# 12. CONSTANT COLUMNS
# ============================================================

print("\n" + "=" * 100)
print("CONSTANT COLUMNS")
print("=" * 100)

constant_cols = [
    col
    for col in df.columns
    if df[col].nunique(
        dropna=False
    ) <= 1
]

if constant_cols:
    for col in constant_cols:
        print("⚠️", repr(col))
else:
    print("✅ No constant columns.")


# ============================================================
# 13. SAMPLE ROWS
# ============================================================

print("\n" + "=" * 100)
print("FIRST 5 ROWS")
print("=" * 100)

pd.set_option(
    "display.max_columns",
    None
)

print(
    df.head().to_string()
)


# ============================================================
# 14. SAVE AUDIT SUMMARY
# ============================================================

audit_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "exact_duplicate_rows",
        "exact_duplicate_percent",
        "numeric_columns",
        "non_numeric_columns",
        "columns_with_missing_values",
        "constant_columns",
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        exact_duplicates,
        exact_duplicates / len(df) * 100,
        df.select_dtypes(include=[np.number]).shape[1],
        df.select_dtypes(exclude=[np.number]).shape[1],
        int((df.isna().sum() > 0).sum()),
        len(constant_cols),
    ]
})

AUDIT_FILE = (
    TABLES_DIR
    / "initial_dataset_audit.csv"
)

audit_summary.to_csv(
    AUDIT_FILE,
    index=False
)

print("\n" + "=" * 100)
print("AUDIT COMPLETE")
print("=" * 100)

print("✅ Audit summary saved:")
print(AUDIT_FILE)

In [ ]:
# ============================================================
# UAV DATASET — RAW STRUCTURE FORENSIC AUDIT
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import csv
import pandas as pd


# ============================================================
# 1. FILE PATH
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

RAW_FILE = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "Dataset_T-ITS.csv"
)

assert RAW_FILE.exists(), f"Dataset not found: {RAW_FILE}"

print("✅ Dataset found:")
print(RAW_FILE)


# ============================================================
# 2. TARGET STRINGS WE WANT TO LOCATE
# ============================================================

KNOWN_LABELS = {
    "benign",
    "DoS attack",
    "Replay",
    "evil_twin",
    "FDI",
}

HEADER_TOKENS = {
    "class",
    "timestamp_c",
    "frame.number",
    "wlan.fc.type",
    "wlan.fc.subtype",
}


# ============================================================
# 3. READ RAW CSV STRUCTURE
# ============================================================

field_count_distribution = Counter()

label_position_counts = defaultdict(Counter)
label_row_numbers = defaultdict(list)

header_like_rows = []

raw_examples_by_length = {}

total_rows = 0


with RAW_FILE.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    reader = csv.reader(f)

    for line_number, row in enumerate(reader, start=1):

        total_rows += 1

        # --------------------------------------------
        # Number of fields in this raw CSV row
        # --------------------------------------------

        n_fields = len(row)

        field_count_distribution[n_fields] += 1

        if n_fields not in raw_examples_by_length:
            raw_examples_by_length[n_fields] = (
                line_number,
                row[:10]
            )

        # --------------------------------------------
        # Strip values for exact matching
        # --------------------------------------------

        stripped_row = [
            str(value).strip()
            for value in row
        ]

        # --------------------------------------------
        # Locate attack labels anywhere in row
        # --------------------------------------------

        for position, value in enumerate(
            stripped_row,
            start=1
        ):

            if value in KNOWN_LABELS:

                label_position_counts[value][position] += 1

                label_row_numbers[value].append(
                    line_number
                )

        # --------------------------------------------
        # Detect repeated / embedded header rows
        # --------------------------------------------

        header_hits = [
            value
            for value in stripped_row
            if value in HEADER_TOKENS
        ]

        if len(header_hits) >= 2:

            header_like_rows.append({
                "line_number": line_number,
                "field_count": n_fields,
                "header_hits": header_hits,
                "first_12_values": stripped_row[:12],
            })


# ============================================================
# 4. FIELD-COUNT DISTRIBUTION
# ============================================================

print("\n" + "=" * 100)
print("RAW ROW LENGTH DISTRIBUTION")
print("=" * 100)

print("Total physical lines:", total_rows)

for n_fields, count in sorted(
    field_count_distribution.items()
):

    pct = (
        count
        / total_rows
        * 100
    )

    print(
        f"{n_fields:3d} fields : "
        f"{count:7d} rows "
        f"({pct:7.3f}%)"
    )


# ============================================================
# 5. EXAMPLE FOR EACH ROW LENGTH
# ============================================================

print("\n" + "=" * 100)
print("FIRST EXAMPLE OF EACH ROW LENGTH")
print("=" * 100)

for n_fields in sorted(
    raw_examples_by_length
):

    line_number, values = (
        raw_examples_by_length[n_fields]
    )

    print(
        f"\nField count : {n_fields}"
    )

    print(
        f"First line  : {line_number}"
    )

    print(
        "First values:",
        values
    )


# ============================================================
# 6. LABEL LOCATIONS
# ============================================================

print("\n" + "=" * 100)
print("ATTACK LABEL LOCATIONS")
print("=" * 100)

for label in sorted(KNOWN_LABELS):

    print(f"\nLABEL: {label}")
    print("-" * 80)

    positions = label_position_counts[label]

    if not positions:

        print("❌ Label not found.")
        continue

    total_label_rows = sum(
        positions.values()
    )

    print(
        "Total occurrences:",
        total_label_rows
    )

    print(
        "Column-position distribution:"
    )

    for position, count in sorted(
        positions.items()
    ):

        print(
            f"  column {position:02d}: "
            f"{count}"
        )

    rows = label_row_numbers[label]

    print(
        "First raw line:",
        min(rows)
    )

    print(
        "Last raw line :",
        max(rows)
    )


# ============================================================
# 7. REPEATED / EMBEDDED HEADER ROWS
# ============================================================

print("\n" + "=" * 100)
print("HEADER-LIKE ROWS")
print("=" * 100)

print(
    "Detected header-like rows:",
    len(header_like_rows)
)

for item in header_like_rows[:30]:

    print("\nLine:", item["line_number"])
    print(
        "Fields:",
        item["field_count"]
    )

    print(
        "Header hits:",
        item["header_hits"]
    )

    print(
        "Beginning:",
        item["first_12_values"]
    )


# ============================================================
# 8. PANDAS-LEVEL LABEL SEARCH ACROSS ALL COLUMNS
# ============================================================

print("\n" + "=" * 100)
print("LABELS FOUND BY PANDAS COLUMN")
print("=" * 100)

df = pd.read_csv(
    RAW_FILE,
    dtype=str,
    low_memory=False
)

label_locations_table = []

for label in sorted(KNOWN_LABELS):

    for col in df.columns:

        normalized = (
            df[col]
            .astype("string")
            .str.strip()
        )

        count = int(
            (normalized == label).sum()
        )

        if count > 0:

            label_locations_table.append({
                "label": label,
                "column": col,
                "count": count,
            })


label_locations_df = pd.DataFrame(
    label_locations_table
)

if label_locations_df.empty:

    print("No known labels located.")

else:

    print(
        label_locations_df
        .sort_values(
            ["label", "column"]
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 9. DETECT ROWS CONTAINING COLUMN NAMES AS DATA
# ============================================================

print("\n" + "=" * 100)
print("EMBEDDED HEADER VALUES INSIDE DATAFRAME")
print("=" * 100)

embedded_headers = []

for col in df.columns:

    values = (
        df[col]
        .astype("string")
        .str.strip()
    )

    for token in HEADER_TOKENS:

        mask = (
            values == token
        )

        count = int(
            mask.sum()
        )

        if count > 0:

            embedded_headers.append({
                "dataframe_column": col,
                "embedded_value": token,
                "count": count,
            })


embedded_headers_df = pd.DataFrame(
    embedded_headers
)

if embedded_headers_df.empty:

    print(
        "✅ No embedded header values detected."
    )

else:

    print(
        embedded_headers_df
        .sort_values(
            [
                "dataframe_column",
                "embedded_value"
            ]
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 10. SAVE STRUCTURAL AUDIT
# ============================================================

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


field_counts_df = pd.DataFrame(
    [
        {
            "field_count": n_fields,
            "row_count": count,
            "percent": count / total_rows * 100,
        }
        for n_fields, count
        in sorted(
            field_count_distribution.items()
        )
    ]
)

field_counts_df.to_csv(
    TABLES_DIR
    / "raw_row_length_distribution.csv",
    index=False
)

label_locations_df.to_csv(
    TABLES_DIR
    / "raw_label_locations.csv",
    index=False
)

embedded_headers_df.to_csv(
    TABLES_DIR
    / "embedded_header_audit.csv",
    index=False
)


print("\n" + "=" * 100)
print("STRUCTURAL AUDIT COMPLETE")
print("=" * 100)

print(
    "✅ Saved:",
    TABLES_DIR
    / "raw_row_length_distribution.csv"
)

print(
    "✅ Saved:",
    TABLES_DIR
    / "raw_label_locations.csv"
)

print(
    "✅ Saved:",
    TABLES_DIR
    / "embedded_header_audit.csv"
)

In [ ]:
# ============================================================
# UAV DATASET — IDENTIFY INTERNAL CYBER / PHYSICAL BOUNDARIES
# ============================================================

from pathlib import Path
from collections import Counter
import csv
import re


# ============================================================
# 1. FILE
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

RAW_FILE = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "Dataset_T-ITS.csv"
)

assert RAW_FILE.exists()

print("✅ Dataset:")
print(RAW_FILE)


# ============================================================
# 2. READ ALL RAW ROWS
# ============================================================

with RAW_FILE.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    rows = list(csv.reader(f))


print("\nTotal raw lines:", len(rows))


# ============================================================
# 3. CANDIDATE INTERNAL HEADER LINES
#
# Derived from the label-position counts found in the audit.
# We inspect them rather than assuming they are correct.
# ============================================================

candidate_internal_headers = [
    9427,   # benign: 9425 cyber rows before this
    25390,  # DoS: 11671 cyber rows before this
    38371,  # Replay: 12006 cyber rows before this
    45029,  # Evil Twin: 5683 rows before this
    53977,  # FDI: 3473 rows before this
]


# ============================================================
# 4. HELPER — PRINT FULL INDEXED ROW
# ============================================================

def print_indexed_row(line_number):

    row = rows[line_number - 1]

    print("\n" + "=" * 110)
    print(f"RAW LINE {line_number}")
    print("=" * 110)

    for i, value in enumerate(row, start=1):

        print(
            f"{i:02d}. {repr(value)}"
        )


# ============================================================
# 5. PRINT CANDIDATE INTERNAL HEADERS
# ============================================================

print("\n" + "=" * 110)
print("CANDIDATE INTERNAL HEADER ROWS")
print("=" * 110)

for line_number in candidate_internal_headers:
    print_indexed_row(line_number)


# ============================================================
# 6. PRINT ±2 ROWS AROUND EACH CANDIDATE
# ============================================================

print("\n" + "=" * 110)
print("ROWS AROUND EACH INTERNAL BOUNDARY")
print("=" * 110)

for center in candidate_internal_headers:

    print("\n" + "#" * 110)
    print(f"BOUNDARY AROUND RAW LINE {center}")
    print("#" * 110)

    for line_number in range(
        center - 2,
        center + 3
    ):

        row = rows[line_number - 1]

        nonempty = [
            x.strip()
            for x in row
            if str(x).strip()
        ]

        print(
            f"\nLINE {line_number}"
        )

        print(
            "First 12:",
            row[:12]
        )

        print(
            "Last 10 :",
            row[-10:]
        )

        print(
            "Non-empty fields:",
            len(nonempty)
        )


# ============================================================
# 7. SEARCH ALL ATTACK-LIKE TEXT TOKENS
#
# Important for finding the physical DoS label spelling.
# ============================================================

print("\n" + "=" * 110)
print("ALL ATTACK-LIKE TEXT VALUES")
print("=" * 110)

attack_pattern = re.compile(
    r"(benign|replay|evil|twin|fdi|dos|attack)",
    flags=re.IGNORECASE
)

attack_like_values = Counter()

attack_like_positions = Counter()

examples = {}


for line_number, row in enumerate(
    rows,
    start=1
):

    for position, value in enumerate(
        row,
        start=1
    ):

        value_clean = str(value).strip()

        if (
            value_clean
            and attack_pattern.search(value_clean)
        ):

            key = value_clean

            attack_like_values[key] += 1

            attack_like_positions[
                (key, position)
            ] += 1

            examples.setdefault(
                key,
                line_number
            )


print("\nVALUE COUNTS")
print("-" * 90)

for value, count in attack_like_values.most_common():

    print(
        f"{repr(value):30s}"
        f" count={count:7d}"
        f" first_line={examples[value]}"
    )


print("\n" + "=" * 110)
print("VALUE + COLUMN POSITION")
print("=" * 110)

for (value, position), count in sorted(
    attack_like_positions.items(),
    key=lambda x: (
        x[0][0].lower(),
        x[0][1]
    )
):

    print(
        f"{repr(value):30s}"
        f" column={position:02d}"
        f" count={count}"
    )


# ============================================================
# 8. CHECK MAIN BLOCK HEADERS TOO
# ============================================================

main_headers = [
    1,
    13718,
    26364,
    39345,
    50503,
]

print("\n" + "=" * 110)
print("TOP-LEVEL BLOCK HEADERS")
print("=" * 110)

for line_number in main_headers:
    print_indexed_row(line_number)


# ============================================================
# 9. DONE
# ============================================================

print("\n" + "=" * 110)
print("BOUNDARY AUDIT COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# UAV DATASET — SAFE MULTI-SCHEMA PARSER
# ============================================================

from pathlib import Path
import csv
import json
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

RAW_FILE = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "Dataset_T-ITS.csv"
)

SECTIONS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "sections"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

SECTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert RAW_FILE.exists(), RAW_FILE


# ============================================================
# 2. READ RAW CSV
# ============================================================

with RAW_FILE.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:
    raw_rows = list(csv.reader(f))

print("✅ Raw lines:", len(raw_rows))


# ============================================================
# 3. SECTION SPECIFICATION
#
# Raw line numbers are inclusive.
# These boundaries were established by the structural audit.
# ============================================================

sections = [

    # --------------------------------------------------------
    # BENIGN
    # --------------------------------------------------------
    {
        "name": "cyber_benign",
        "attack": "Benign",
        "modality": "cyber",
        "schema_id": "cyber_schema_A",
        "header_line": 1,
        "data_start": 2,
        "data_end": 9426,
        "expected_raw_label": "benign",
    },

    {
        "name": "physical_benign",
        "attack": "Benign",
        "modality": "physical",
        "schema_id": "physical_schema_A",
        "header_line": 9427,
        "data_start": 9428,
        "data_end": 13717,
        "expected_raw_label": "benign",
    },

    # --------------------------------------------------------
    # DOS
    # --------------------------------------------------------
    {
        "name": "cyber_dos",
        "attack": "DoS",
        "modality": "cyber",
        "schema_id": "cyber_schema_A",
        "header_line": 13718,
        "data_start": 13719,
        "data_end": 25389,
        "expected_raw_label": "DoS attack",
    },

    {
        "name": "physical_dos",
        "attack": "DoS",
        "modality": "physical",
        "schema_id": "physical_schema_A",
        "header_line": 25390,
        "data_start": 25391,
        "data_end": 26363,
        "expected_raw_label": "DoS",
    },

    # --------------------------------------------------------
    # REPLAY
    # --------------------------------------------------------
    {
        "name": "cyber_replay",
        "attack": "Replay",
        "modality": "cyber",
        "schema_id": "cyber_schema_A",
        "header_line": 26364,
        "data_start": 26365,
        "data_end": 38370,
        "expected_raw_label": "Replay",
    },

    {
        "name": "physical_replay",
        "attack": "Replay",
        "modality": "physical",
        "schema_id": "physical_schema_A",
        "header_line": 38371,
        "data_start": 38372,
        "data_end": 39344,
        "expected_raw_label": "Replay",
    },

    # --------------------------------------------------------
    # EVIL TWIN
    # --------------------------------------------------------
    {
        "name": "cyber_evil_twin",
        "attack": "Evil Twin",
        "modality": "cyber",
        "schema_id": "cyber_schema_B",
        "header_line": 39345,
        "data_start": 39346,
        "data_end": 45028,
        "expected_raw_label": "evil_twin",
    },

    {
        "name": "physical_evil_twin",
        "attack": "Evil Twin",
        "modality": "physical",
        "schema_id": "physical_schema_B",
        "header_line": 45029,
        "data_start": 45030,
        "data_end": 50502,
        "expected_raw_label": "evil_twin",
    },

    # --------------------------------------------------------
    # FDI
    # --------------------------------------------------------
    {
        "name": "cyber_fdi",
        "attack": "FDI",
        "modality": "cyber",
        "schema_id": "cyber_schema_B",
        "header_line": 50503,
        "data_start": 50504,
        "data_end": 53976,
        "expected_raw_label": "FDI",
    },

    {
        "name": "physical_fdi",
        "attack": "FDI",
        "modality": "physical",
        "schema_id": "physical_schema_C",
        "header_line": 53977,
        "data_start": 53978,
        "data_end": 54784,
        "expected_raw_label": "FDI",
    },
]


# ============================================================
# 4. HELPER — TRIM ONLY TRAILING EMPTY HEADER CELLS
# ============================================================

def trim_header(row):

    cleaned = [
        str(x).strip()
        for x in row
    ]

    while cleaned and cleaned[-1] == "":
        cleaned.pop()

    return cleaned


# ============================================================
# 5. PARSE ONE SECTION
# ============================================================

def parse_section(spec):

    header = trim_header(
        raw_rows[
            spec["header_line"] - 1
        ]
    )

    if "class" not in header:
        raise ValueError(
            f"No class column in {spec['name']}"
        )

    n_columns = len(header)

    parsed_rows = []
    source_lines = []

    for raw_line in range(
        spec["data_start"],
        spec["data_end"] + 1
    ):

        row = raw_rows[
            raw_line - 1
        ]

        values = [
            str(x).strip()
            for x in row[:n_columns]
        ]

        if len(values) != n_columns:
            raise ValueError(
                f"{spec['name']}: "
                f"line {raw_line} has "
                f"{len(values)} values; "
                f"expected {n_columns}"
            )

        parsed_rows.append(values)
        source_lines.append(raw_line)

    df_section = pd.DataFrame(
        parsed_rows,
        columns=header
    )

    df_section.insert(
        0,
        "source_line",
        source_lines
    )

    # --------------------------------------------------------
    # Validate raw labels
    # --------------------------------------------------------

    raw_labels = (
        df_section["class"]
        .astype(str)
        .str.strip()
    )

    observed_labels = sorted(
        raw_labels.unique().tolist()
    )

    expected = spec[
        "expected_raw_label"
    ]

    if observed_labels != [expected]:
        raise ValueError(
            f"\nLabel mismatch in {spec['name']}\n"
            f"Expected: {expected}\n"
            f"Observed: {observed_labels}"
        )

    # --------------------------------------------------------
    # Preserve original label + normalized target
    # --------------------------------------------------------

    df_section = df_section.rename(
        columns={
            "class": "raw_class"
        }
    )

    df_section["attack_class"] = (
        spec["attack"]
    )

    df_section["modality"] = (
        spec["modality"]
    )

    df_section["schema_id"] = (
        spec["schema_id"]
    )

    return df_section, header


# ============================================================
# 6. PARSE ALL 10 SECTIONS
# ============================================================

parsed = {}
manifest_rows = []

print("\n" + "=" * 100)
print("PARSING SECTIONS")
print("=" * 100)

for spec in sections:

    df_section, header = (
        parse_section(spec)
    )

    parsed[
        spec["name"]
    ] = df_section

    output_file = (
        SECTIONS_DIR
        / f"{spec['name']}.csv"
    )

    df_section.to_csv(
        output_file,
        index=False
    )

    feature_columns = [
        c
        for c in header
        if c != "class"
    ]

    duplicate_count = (
        df_section[
            feature_columns
            + ["raw_class"]
        ]
        .duplicated()
        .sum()
    )

    manifest_rows.append({
        "section": spec["name"],
        "attack_class": spec["attack"],
        "modality": spec["modality"],
        "schema_id": spec["schema_id"],
        "header_line": spec["header_line"],
        "data_start": spec["data_start"],
        "data_end": spec["data_end"],
        "rows": len(df_section),
        "raw_columns": len(header),
        "predictor_columns": len(
            feature_columns
        ),
        "exact_duplicates": int(
            duplicate_count
        ),
        "output_file": str(
            output_file.relative_to(
                PROJECT_DIR
            )
        ),
    })

    print(
        f"✅ {spec['name']:22s} "
        f"| rows={len(df_section):5d} "
        f"| cols={len(header):2d} "
        f"| schema={spec['schema_id']}"
    )


# ============================================================
# 7. SECTION MANIFEST
# ============================================================

manifest = pd.DataFrame(
    manifest_rows
)

manifest_file = (
    TABLES_DIR
    / "parsed_section_manifest.csv"
)

manifest.to_csv(
    manifest_file,
    index=False
)

print("\n" + "=" * 100)
print("SECTION MANIFEST")
print("=" * 100)

print(
    manifest[
        [
            "section",
            "attack_class",
            "modality",
            "schema_id",
            "rows",
            "predictor_columns",
            "exact_duplicates",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 8. VERIFY TOTAL DATA ROWS
# ============================================================

total_parsed_rows = sum(
    len(df)
    for df in parsed.values()
)

print("\n" + "=" * 100)
print("ROW ACCOUNTING")
print("=" * 100)

print(
    "Parsed data rows:",
    total_parsed_rows
)

print(
    "Section header rows:",
    len(sections)
)

print(
    "Expected total physical lines:",
    total_parsed_rows
    + len(sections)
)

print(
    "Actual total physical lines:",
    len(raw_rows)
)

assert (
    total_parsed_rows
    + len(sections)
    == len(raw_rows)
), (
    "Row accounting failed."
)

print(
    "✅ Every raw line accounted for exactly once."
)


# ============================================================
# 9. CYBER FEATURE INTERSECTION
# ============================================================

cyber_sections = [
    df
    for name, df in parsed.items()
    if name.startswith("cyber_")
]

physical_sections = [
    df
    for name, df in parsed.items()
    if name.startswith("physical_")
]


META_COLUMNS = {
    "source_line",
    "raw_class",
    "attack_class",
    "modality",
    "schema_id",
}


def predictor_set(df):

    return {
        col
        for col in df.columns
        if col not in META_COLUMNS
    }


cyber_common = set.intersection(
    *[
        predictor_set(df)
        for df in cyber_sections
    ]
)

physical_common = set.intersection(
    *[
        predictor_set(df)
        for df in physical_sections
    ]
)


print("\n" + "=" * 100)
print("COMMON CYBER FEATURES ACROSS ALL FIVE CLASSES")
print("=" * 100)

for col in sorted(cyber_common):
    print("✅", col)

print(
    "\nTotal common cyber features:",
    len(cyber_common)
)


print("\n" + "=" * 100)
print("COMMON PHYSICAL FEATURES ACROSS ALL FIVE CLASSES")
print("=" * 100)

for col in sorted(physical_common):
    print("✅", col)

print(
    "\nTotal common physical features:",
    len(physical_common)
)


# ============================================================
# 10. SCHEMA FEATURE MAP
# ============================================================

schema_feature_map = {}

for name, df in parsed.items():

    schema = (
        df["schema_id"].iloc[0]
    )

    schema_feature_map.setdefault(
        schema,
        sorted(
            predictor_set(df)
        )
    )


feature_map_file = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "schema_feature_map.json"
)

with feature_map_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "cyber_common_features":
                sorted(cyber_common),

            "physical_common_features":
                sorted(physical_common),

            "schemas":
                schema_feature_map,
        },
        f,
        indent=2
    )


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("PARSER COMPLETE")
print("=" * 100)

print(
    "✅ Parsed section files:",
    SECTIONS_DIR
)

print(
    "✅ Manifest:",
    manifest_file
)

print(
    "✅ Schema feature map:",
    feature_map_file
)

In [ ]:
# ============================================================
# UAV PROJECT — BUILD SCIENTIFIC EXPERIMENTS
# + FEATURE GOVERNANCE
# + DUPLICATE / LABEL-CONFLICT AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SECTIONS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "sections"
)

EXPERIMENTS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "experiments"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

EXPERIMENTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. LOAD PARSED SECTIONS
# ============================================================

section_names = [
    "cyber_benign",
    "physical_benign",
    "cyber_dos",
    "physical_dos",
    "cyber_replay",
    "physical_replay",
    "cyber_evil_twin",
    "physical_evil_twin",
    "cyber_fdi",
    "physical_fdi",
]

sections = {}

for name in section_names:

    path = (
        SECTIONS_DIR
        / f"{name}.csv"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing parsed section:\n{path}"
        )

    sections[name] = pd.read_csv(
        path,
        low_memory=False
    )

print("✅ All parsed sections loaded.")


# ============================================================
# 3. DEFINE SCIENTIFIC EXPERIMENTS
# ============================================================

EXPERIMENT_DEFINITIONS = {

    # --------------------------------------------------------
    # PRIMARY EXPERIMENT 1
    # Same cyber schema for all three classes
    # --------------------------------------------------------
    "cyber_schema_A_3class": {
        "sections": [
            "cyber_benign",
            "cyber_dos",
            "cyber_replay",
        ],
        "task": "multiclass",
        "role": "primary",
        "description": (
            "Cyber-domain classification of "
            "Benign, DoS, and Replay under "
            "the same cyber schema."
        ),
    },

    # --------------------------------------------------------
    # PRIMARY EXPERIMENT 2
    # Same physical schema for all three classes
    # --------------------------------------------------------
    "physical_schema_A_3class": {
        "sections": [
            "physical_benign",
            "physical_dos",
            "physical_replay",
        ],
        "task": "multiclass",
        "role": "primary",
        "description": (
            "Physical-domain classification of "
            "Benign, DoS, and Replay under "
            "the same physical schema."
        ),
    },

    # --------------------------------------------------------
    # SECONDARY EXPERIMENT
    # Both classes share cyber_schema_B
    # --------------------------------------------------------
    "cyber_schema_B_attack_type": {
        "sections": [
            "cyber_evil_twin",
            "cyber_fdi",
        ],
        "task": "binary_attack_type",
        "role": "secondary",
        "description": (
            "Cyber-domain discrimination between "
            "Evil Twin and FDI. "
            "This is not a benign-vs-attack detector."
        ),
    },
}


# ============================================================
# 4. METADATA THAT MUST NEVER ENTER THE MODEL
# ============================================================

META_COLUMNS = {
    "source_line",
    "raw_class",
    "attack_class",
    "modality",
    "schema_id",
}


# ============================================================
# 5. CONSERVATIVE LEAKAGE-PRONE FEATURE POLICY
#
# These variables may encode collection identity,
# packet identity, endpoint identity, or recording order.
# We exclude them BEFORE model development.
# ============================================================

CYBER_EXCLUDE_BY_DESIGN = {
    "timestamp_c",
    "frame.number",

    # Endpoint / hardware identity
    "wlan.ra",
    "wlan.ta",
    "wlan.da",
    "wlan.sa",
    "wlan.bssid",
    "ip.src",
    "ip.dst",

    # Packet/session identifiers
    "ip.id",
    "tcp.seq_raw",
    "tcp.ack_raw",
    "wlan.seq",

    # Raw packet contents / checksum-like identity
    "data.data",
    "wlan.fcs",
}


PHYSICAL_EXCLUDE_BY_DESIGN = {
    "timestamp_p",

    # Strong recording-progress variable.
    # Keep out initially to reduce temporal leakage.
    "flight_time",
}


# ============================================================
# 6. HELPER — COMBINE SECTIONS
# ============================================================

def build_experiment(exp_name, definition):

    dfs = [
        sections[name].copy()
        for name in definition["sections"]
    ]

    # Validate schema consistency within experiment
    schema_ids = sorted(
        set(
            str(df["schema_id"].iloc[0])
            for df in dfs
        )
    )

    if len(schema_ids) != 1:
        raise ValueError(
            f"{exp_name} mixes schemas: "
            f"{schema_ids}"
        )

    # Validate predictor columns are identical
    predictor_sets = []

    for df in dfs:

        predictors = [
            col
            for col in df.columns
            if col not in META_COLUMNS
        ]

        predictor_sets.append(
            predictors
        )

    reference = predictor_sets[0]

    for predictors in predictor_sets[1:]:

        if predictors != reference:
            raise ValueError(
                f"{exp_name}: predictor schemas differ."
            )

    combined = pd.concat(
        dfs,
        ignore_index=True
    )

    return combined, reference, schema_ids[0]


# ============================================================
# 7. FEATURE AUDIT FUNCTION
# ============================================================

def audit_features(
    df,
    predictor_columns,
    excluded_by_design
):

    rows = []

    for col in predictor_columns:

        series = df[col]

        nonmissing = (
            series
            .dropna()
        )

        n_nonmissing = len(
            nonmissing
        )

        unique_count = (
            nonmissing
            .nunique()
        )

        unique_ratio = (
            unique_count / n_nonmissing
            if n_nonmissing > 0
            else np.nan
        )

        numeric_version = pd.to_numeric(
            series,
            errors="coerce"
        )

        numeric_ratio = (
            numeric_version.notna().sum()
            / len(df)
        )

        missing_count = int(
            series.isna().sum()
        )

        missing_percent = (
            missing_count
            / len(df)
            * 100
        )

        rows.append({
            "feature": col,
            "rows": len(df),
            "nonmissing": n_nonmissing,
            "missing_count": missing_count,
            "missing_percent": missing_percent,
            "unique_values": unique_count,
            "unique_ratio": unique_ratio,
            "numeric_conversion_ratio": numeric_ratio,
            "excluded_by_design":
                col in excluded_by_design,
        })

    return pd.DataFrame(
        rows
    )


# ============================================================
# 8. FEATURE-VECTOR CONFLICT AUDIT
#
# Same selected predictors but different class labels
# would represent contradictory supervision.
# ============================================================

def conflict_audit(
    df,
    features
):

    work = (
        df[
            features
            + ["attack_class"]
        ]
        .copy()
    )

    # Normalize missing representation
    for col in features:
        work[col] = (
            work[col]
            .astype("string")
            .fillna("<MISSING>")
        )

    grouped = (
        work
        .groupby(
            features,
            dropna=False,
            sort=False
        )["attack_class"]
        .agg(
            label_count="nunique",
            row_count="size"
        )
        .reset_index()
    )

    conflicting_groups = (
        grouped[
            grouped["label_count"] > 1
        ]
    )

    n_conflicting_groups = len(
        conflicting_groups
    )

    rows_affected = int(
        conflicting_groups[
            "row_count"
        ].sum()
    )

    return (
        n_conflicting_groups,
        rows_affected
    )


# ============================================================
# 9. BUILD ALL EXPERIMENTS
# ============================================================

experiment_manifest = []
feature_config = {}

print("\n" + "=" * 105)
print("BUILDING EXPERIMENTS")
print("=" * 105)

for exp_name, definition in (
    EXPERIMENT_DEFINITIONS.items()
):

    df_exp, raw_predictors, schema_id = (
        build_experiment(
            exp_name,
            definition
        )
    )

    modality = str(
        df_exp["modality"].iloc[0]
    )

    if modality == "cyber":
        excluded = (
            CYBER_EXCLUDE_BY_DESIGN
            & set(raw_predictors)
        )

    else:
        excluded = (
            PHYSICAL_EXCLUDE_BY_DESIGN
            & set(raw_predictors)
        )

    candidate_features = [
        col
        for col in raw_predictors
        if col not in excluded
    ]

    # --------------------------------------------------------
    # Class counts
    # --------------------------------------------------------

    class_counts = (
        df_exp["attack_class"]
        .value_counts()
        .sort_index()
    )

    # --------------------------------------------------------
    # Duplicate audit using candidate predictors
    # --------------------------------------------------------

    duplicate_mask = (
        df_exp[
            candidate_features
            + ["attack_class"]
        ]
        .duplicated()
    )

    duplicate_rows = int(
        duplicate_mask.sum()
    )

    # --------------------------------------------------------
    # Contradictory feature vectors
    # --------------------------------------------------------

    (
        conflicting_groups,
        conflicting_rows,
    ) = conflict_audit(
        df_exp,
        candidate_features
    )

    # --------------------------------------------------------
    # Feature audit
    # --------------------------------------------------------

    audit = audit_features(
        df_exp,
        raw_predictors,
        excluded
    )

    audit_file = (
        TABLES_DIR
        / f"{exp_name}_feature_audit.csv"
    )

    audit.to_csv(
        audit_file,
        index=False
    )

    # --------------------------------------------------------
    # Save experiment with provenance retained
    #
    # source_line is retained for splitting/audit,
    # but NEVER becomes a model predictor.
    # --------------------------------------------------------

    keep_columns = (
        ["source_line"]
        + candidate_features
        + ["attack_class"]
    )

    experiment_df = (
        df_exp[
            keep_columns
        ]
        .copy()
    )

    experiment_file = (
        EXPERIMENTS_DIR
        / f"{exp_name}.csv"
    )

    experiment_df.to_csv(
        experiment_file,
        index=False
    )

    # --------------------------------------------------------
    # Save feature configuration
    # --------------------------------------------------------

    feature_config[
        exp_name
    ] = {
        "schema_id": schema_id,
        "modality": modality,
        "task": definition["task"],
        "role": definition["role"],
        "description":
            definition["description"],

        "raw_predictor_count":
            len(raw_predictors),

        "excluded_by_design":
            sorted(excluded),

        "candidate_features":
            candidate_features,

        "candidate_feature_count":
            len(candidate_features),
    }

    # --------------------------------------------------------
    # Manifest
    # --------------------------------------------------------

    experiment_manifest.append({
        "experiment": exp_name,
        "role": definition["role"],
        "task": definition["task"],
        "schema_id": schema_id,
        "modality": modality,
        "rows": len(df_exp),
        "classes": len(class_counts),
        "raw_predictors":
            len(raw_predictors),
        "excluded_predictors":
            len(excluded),
        "candidate_predictors":
            len(candidate_features),
        "duplicate_rows":
            duplicate_rows,
        "conflicting_feature_groups":
            conflicting_groups,
        "rows_in_conflicting_groups":
            conflicting_rows,
    })

    print(
        f"\n✅ {exp_name}"
    )

    print(
        f"   Schema              : {schema_id}"
    )

    print(
        f"   Rows                : {len(df_exp)}"
    )

    print(
        f"   Raw predictors      : {len(raw_predictors)}"
    )

    print(
        f"   Excluded by design  : {len(excluded)}"
    )

    print(
        f"   Candidate predictors: {len(candidate_features)}"
    )

    print(
        "   Class distribution:"
    )

    for label, count in (
        class_counts.items()
    ):

        print(
            f"      {label:12s}: {count}"
        )

    print(
        f"   Duplicate rows      : {duplicate_rows}"
    )

    print(
        f"   Conflicting groups  : {conflicting_groups}"
    )

    print(
        f"   Conflict rows       : {conflicting_rows}"
    )


# ============================================================
# 10. SAVE EXPERIMENT MANIFEST
# ============================================================

manifest_df = pd.DataFrame(
    experiment_manifest
)

manifest_file = (
    TABLES_DIR
    / "experiment_design_manifest.csv"
)

manifest_df.to_csv(
    manifest_file,
    index=False
)


# ============================================================
# 11. SAVE FEATURE CONFIGURATION
# ============================================================

feature_config_file = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "experiment_feature_configuration.json"
)

with feature_config_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_config,
        f,
        indent=2
    )


# ============================================================
# 12. HIGH-RISK REMAINING FEATURES
#
# Features with extremely high uniqueness may still
# behave like identifiers even if not excluded by name.
# ============================================================

print("\n" + "=" * 105)
print("HIGH-UNIQUENESS REMAINING FEATURE CHECK")
print("=" * 105)

for exp_name in (
    EXPERIMENT_DEFINITIONS
):

    audit = pd.read_csv(
        TABLES_DIR
        / f"{exp_name}_feature_audit.csv"
    )

    config = feature_config[
        exp_name
    ]

    candidates = set(
        config[
            "candidate_features"
        ]
    )

    high_unique = audit[
        (
            audit["feature"]
            .isin(candidates)
        )
        &
        (
            audit["unique_ratio"]
            >= 0.80
        )
    ]

    print(
        f"\n{exp_name}:"
    )

    if high_unique.empty:

        print(
            "✅ No candidate feature has "
            ">=80% unique values."
        )

    else:

        print(
            high_unique[
                [
                    "feature",
                    "unique_values",
                    "unique_ratio",
                    "numeric_conversion_ratio",
                ]
            ]
            .sort_values(
                "unique_ratio",
                ascending=False
            )
            .to_string(
                index=False
            )
        )


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 105)
print("EXPERIMENT DESIGN MANIFEST")
print("=" * 105)

print(
    manifest_df.to_string(
        index=False
    )
)

print("\n" + "=" * 105)
print("DONE")
print("=" * 105)

print(
    "✅ Experiments saved in:",
    EXPERIMENTS_DIR
)

print(
    "✅ Manifest:",
    manifest_file
)

print(
    "✅ Feature configuration:",
    feature_config_file
)

print(
    "\n⚠️ No model has been trained."
)

print(
    "Next step = finalize admissible features "
    "+ design leakage-resistant temporal splits."
)

In [ ]:
# ============================================================
# UAV PROJECT — TEMPORAL STRUCTURE AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SECTIONS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "sections"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. SECTIONS TO AUDIT
# ============================================================

TEMPORAL_SECTIONS = {
    "cyber_benign": {
        "timestamp": "timestamp_c",
        "class": "Benign"
    },
    "cyber_dos": {
        "timestamp": "timestamp_c",
        "class": "DoS"
    },
    "cyber_replay": {
        "timestamp": "timestamp_c",
        "class": "Replay"
    },

    "physical_benign": {
        "timestamp": "timestamp_p",
        "class": "Benign"
    },
    "physical_dos": {
        "timestamp": "timestamp_p",
        "class": "DoS"
    },
    "physical_replay": {
        "timestamp": "timestamp_p",
        "class": "Replay"
    },

    "cyber_evil_twin": {
        "timestamp": "timestamp_c",
        "class": "Evil Twin"
    },
    "cyber_fdi": {
        "timestamp": "timestamp_c",
        "class": "FDI"
    },
}


# ============================================================
# 3. AUDIT FUNCTION
# ============================================================

def temporal_audit(
    section_name,
    timestamp_col,
    class_name
):

    path = (
        SECTIONS_DIR
        / f"{section_name}.csv"
    )

    df = pd.read_csv(
        path,
        low_memory=False
    )

    if timestamp_col not in df.columns:
        raise ValueError(
            f"{timestamp_col} missing from {section_name}"
        )

    # --------------------------------------------------------
    # Convert timestamp
    # --------------------------------------------------------

    ts = pd.to_numeric(
        df[timestamp_col],
        errors="coerce"
    )

    valid = ts.notna()

    ts_valid = (
        ts[valid]
        .astype(float)
    )

    # --------------------------------------------------------
    # Basic temporal diagnostics
    # --------------------------------------------------------

    parse_ratio = (
        valid.mean()
    )

    if len(ts_valid) < 2:
        raise ValueError(
            f"Not enough valid timestamps in {section_name}"
        )

    raw_diff = (
        ts_valid
        .diff()
        .dropna()
    )

    negative_steps = int(
        (raw_diff < 0).sum()
    )

    zero_steps = int(
        (raw_diff == 0).sum()
    )

    positive_diff = (
        raw_diff[
            raw_diff > 0
        ]
    )

    duration = (
        ts_valid.iloc[-1]
        - ts_valid.iloc[0]
    )

    # --------------------------------------------------------
    # Estimate packets / observations per second
    # from elapsed duration
    # --------------------------------------------------------

    rows_per_second = (
        len(ts_valid) / duration
        if duration > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Candidate time-window counts
    # --------------------------------------------------------

    window_counts = {}

    for window_seconds in [
        0.5,
        1,
        2,
        5,
        10,
    ]:

        relative_time = (
            ts_valid
            - ts_valid.iloc[0]
        )

        window_id = np.floor(
            relative_time
            / window_seconds
        ).astype(int)

        counts = pd.Series(
            window_id
        ).value_counts()

        window_counts[
            window_seconds
        ] = {
            "n_windows": int(
                counts.size
            ),
            "median_rows_per_window": float(
                counts.median()
            ),
            "min_rows_per_window": int(
                counts.min()
            ),
            "max_rows_per_window": int(
                counts.max()
            ),
        }

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    summary = {
        "section": section_name,
        "class": class_name,
        "timestamp_column": timestamp_col,
        "rows": len(df),
        "valid_timestamp_rows": int(
            valid.sum()
        ),
        "timestamp_parse_ratio": float(
            parse_ratio
        ),
        "timestamp_start": float(
            ts_valid.iloc[0]
        ),
        "timestamp_end": float(
            ts_valid.iloc[-1]
        ),
        "duration_seconds": float(
            duration
        ),
        "negative_time_steps": negative_steps,
        "zero_time_steps": zero_steps,
        "median_positive_delta": (
            float(
                positive_diff.median()
            )
            if len(positive_diff)
            else np.nan
        ),
        "p95_positive_delta": (
            float(
                positive_diff.quantile(0.95)
            )
            if len(positive_diff)
            else np.nan
        ),
        "approx_rows_per_second": float(
            rows_per_second
        ),
    }

    return summary, window_counts


# ============================================================
# 4. RUN AUDIT
# ============================================================

all_summaries = []
all_window_rows = []

print("=" * 105)
print("TEMPORAL STRUCTURE AUDIT")
print("=" * 105)

for section_name, info in (
    TEMPORAL_SECTIONS.items()
):

    summary, windows = temporal_audit(
        section_name,
        info["timestamp"],
        info["class"]
    )

    all_summaries.append(
        summary
    )

    print(
        f"\n✅ {section_name}"
    )

    print(
        f"   Class               : {summary['class']}"
    )

    print(
        f"   Rows                : {summary['rows']}"
    )

    print(
        f"   Timestamp parse     : "
        f"{summary['timestamp_parse_ratio']:.4f}"
    )

    print(
        f"   Start               : "
        f"{summary['timestamp_start']}"
    )

    print(
        f"   End                 : "
        f"{summary['timestamp_end']}"
    )

    print(
        f"   Duration (sec)      : "
        f"{summary['duration_seconds']:.3f}"
    )

    print(
        f"   Negative time steps : "
        f"{summary['negative_time_steps']}"
    )

    print(
        f"   Zero time steps     : "
        f"{summary['zero_time_steps']}"
    )

    print(
        f"   Median +delta       : "
        f"{summary['median_positive_delta']}"
    )

    print(
        f"   P95 +delta          : "
        f"{summary['p95_positive_delta']}"
    )

    print(
        f"   Approx rows/sec     : "
        f"{summary['approx_rows_per_second']:.3f}"
    )

    print(
        "   Candidate windows:"
    )

    for window_seconds, stats in (
        windows.items()
    ):

        print(
            f"      {window_seconds:>4}s "
            f"→ windows={stats['n_windows']:5d}, "
            f"median rows={stats['median_rows_per_window']:6.1f}, "
            f"min={stats['min_rows_per_window']:4d}, "
            f"max={stats['max_rows_per_window']:5d}"
        )

        all_window_rows.append({
            "section": section_name,
            "class": info["class"],
            "window_seconds":
                window_seconds,
            **stats
        })


# ============================================================
# 5. SAVE RESULTS
# ============================================================

summary_df = pd.DataFrame(
    all_summaries
)

window_df = pd.DataFrame(
    all_window_rows
)

summary_file = (
    TABLES_DIR
    / "temporal_structure_audit.csv"
)

window_file = (
    TABLES_DIR
    / "candidate_window_audit.csv"
)

summary_df.to_csv(
    summary_file,
    index=False
)

window_df.to_csv(
    window_file,
    index=False
)


# ============================================================
# 6. FIXED-PACKET WINDOW AUDIT
#
# Alternative if timestamps are irregular.
# ============================================================

print("\n" + "=" * 105)
print("FIXED-PACKET / OBSERVATION WINDOW COUNTS")
print("=" * 105)

packet_window_rows = []

for section_name, info in (
    TEMPORAL_SECTIONS.items()
):

    path = (
        SECTIONS_DIR
        / f"{section_name}.csv"
    )

    df = pd.read_csv(
        path,
        low_memory=False
    )

    print(
        f"\n{section_name}"
    )

    for window_size in [
        10,
        20,
        50,
        100,
    ]:

        n_full_windows = (
            len(df)
            // window_size
        )

        remainder = (
            len(df)
            % window_size
        )

        print(
            f"   {window_size:3d} rows/window "
            f"→ full windows={n_full_windows:5d}, "
            f"remainder={remainder:3d}"
        )

        packet_window_rows.append({
            "section": section_name,
            "class": info["class"],
            "window_size_rows":
                window_size,
            "full_windows":
                n_full_windows,
            "remainder_rows":
                remainder,
        })


packet_window_df = pd.DataFrame(
    packet_window_rows
)

packet_window_file = (
    TABLES_DIR
    / "fixed_observation_window_audit.csv"
)

packet_window_df.to_csv(
    packet_window_file,
    index=False
)


# ============================================================
# 7. FINAL
# ============================================================

print("\n" + "=" * 105)
print("AUDIT COMPLETE")
print("=" * 105)

print(
    "✅ Temporal summary:",
    summary_file
)

print(
    "✅ Time-window audit:",
    window_file
)

print(
    "✅ Fixed-window audit:",
    packet_window_file
)

print(
    "\n⚠️ No split created yet."
)

print(
    "⚠️ No model trained yet."
)

In [ ]:
# ============================================================
# UAV PROJECT
# BUILD LEAKAGE-AWARE FIXED-OBSERVATION TEMPORAL WINDOWS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from collections import Counter


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SECTIONS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "sections"
)

WINDOWS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "windows"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

CONFIG_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "experiment_feature_configuration.json"
)

WINDOWS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert CONFIG_FILE.exists(), CONFIG_FILE


# ============================================================
# 2. LOAD FEATURE CONFIGURATION
# ============================================================

with CONFIG_FILE.open(
    "r",
    encoding="utf-8"
) as f:
    feature_config = json.load(f)


# ============================================================
# 3. WINDOW EXPERIMENT SPECIFICATION
# ============================================================

EXPERIMENTS = {

    "cyber_schema_A_3class": {
        "sections": [
            "cyber_benign",
            "cyber_dos",
            "cyber_replay",
        ],
        "timestamp": "timestamp_c",
        "window_size": 20,
    },

    "physical_schema_A_3class": {
        "sections": [
            "physical_benign",
            "physical_dos",
            "physical_replay",
        ],
        "timestamp": "timestamp_p",
        "window_size": 10,
    },

    "cyber_schema_B_attack_type": {
        "sections": [
            "cyber_evil_twin",
            "cyber_fdi",
        ],
        "timestamp": "timestamp_c",
        "window_size": 20,
    },
}


# ============================================================
# 4. LOAD ALL REQUIRED SECTIONS
# ============================================================

all_section_names = sorted(
    {
        section
        for exp in EXPERIMENTS.values()
        for section in exp["sections"]
    }
)

sections = {}

for section_name in all_section_names:

    path = (
        SECTIONS_DIR
        / f"{section_name}.csv"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing section: {path}"
        )

    sections[section_name] = pd.read_csv(
        path,
        low_memory=False
    )

print("✅ Required parsed sections loaded.")


# ============================================================
# 5. DETERMINE EXPERIMENT-LEVEL GAP THRESHOLD
#
# Rule:
# threshold = max(
#     2 seconds,
#     5 × maximum class-specific P95 positive delta
# )
#
# Same threshold is used for every class in the experiment.
# This avoids class-specific segmentation rules.
# ============================================================

def calculate_gap_threshold(
    section_names,
    timestamp_col
):

    p95_values = []

    for section_name in section_names:

        df = sections[
            section_name
        ]

        ts = pd.to_numeric(
            df[timestamp_col],
            errors="coerce"
        )

        diff = (
            ts.diff()
            .dropna()
        )

        positive = diff[
            diff > 0
        ]

        if len(positive):

            p95_values.append(
                float(
                    positive.quantile(
                        0.95
                    )
                )
            )

    if not p95_values:
        return 2.0

    threshold = max(
        2.0,
        5.0 * max(
            p95_values
        )
    )

    return float(
        threshold
    )


# ============================================================
# 6. DETECT TEMPORAL SEGMENTS
#
# New segment if:
# - timestamp decreases
# - timestamp gap exceeds experiment threshold
#
# Zero timestamp differences remain in the same segment.
# ============================================================

def assign_segments(
    df,
    timestamp_col,
    gap_threshold
):

    ts = pd.to_numeric(
        df[timestamp_col],
        errors="coerce"
    )

    if ts.isna().any():
        raise ValueError(
            f"Invalid timestamp detected in {timestamp_col}"
        )

    diff = ts.diff()

    boundary = (
        (diff < 0)
        |
        (diff > gap_threshold)
    )

    boundary.iloc[0] = True

    segment_id = (
        boundary.astype(int)
        .cumsum()
        - 1
    )

    return (
        ts.astype(float),
        segment_id.astype(int)
    )


# ============================================================
# 7. FEATURE TYPE INFERENCE
#
# Feature is treated as numeric if >=95% of nonmissing
# observations convert successfully to numbers.
# ============================================================

def infer_feature_types(
    experiment_sections,
    candidate_features
):

    combined = pd.concat(
        [
            sections[name][
                candidate_features
            ]
            for name in experiment_sections
        ],
        ignore_index=True
    )

    numeric_features = []
    categorical_features = []

    diagnostics = []

    for feature in candidate_features:

        series = combined[
            feature
        ]

        nonmissing = (
            series.notna()
        )

        n_nonmissing = int(
            nonmissing.sum()
        )

        numeric = pd.to_numeric(
            series,
            errors="coerce"
        )

        numeric_success = int(
            (
                numeric.notna()
                & nonmissing
            ).sum()
        )

        if n_nonmissing == 0:
            numeric_ratio = 0.0
        else:
            numeric_ratio = (
                numeric_success
                / n_nonmissing
            )

        if numeric_ratio >= 0.95:
            feature_type = "numeric"
            numeric_features.append(
                feature
            )
        else:
            feature_type = "categorical"
            categorical_features.append(
                feature
            )

        diagnostics.append({
            "feature": feature,
            "type": feature_type,
            "numeric_conversion_ratio":
                numeric_ratio,
        })

    return (
        numeric_features,
        categorical_features,
        pd.DataFrame(
            diagnostics
        )
    )


# ============================================================
# 8. MODE HELPER
# ============================================================

def safe_mode(series):

    values = (
        series
        .dropna()
        .astype(str)
    )

    if len(values) == 0:
        return "<MISSING>"

    counts = (
        values
        .value_counts()
    )

    return str(
        counts.index[0]
    )


# ============================================================
# 9. WINDOW AGGREGATION
# ============================================================

def aggregate_window(
    window_df,
    numeric_features,
    categorical_features
):

    output = {}

    # --------------------------------------------------------
    # Numeric features
    # --------------------------------------------------------

    for feature in numeric_features:

        x = pd.to_numeric(
            window_df[feature],
            errors="coerce"
        )

        valid = x.dropna()

        prefix = feature

        if len(valid) == 0:

            output[
                f"{prefix}__mean"
            ] = np.nan

            output[
                f"{prefix}__std"
            ] = np.nan

            output[
                f"{prefix}__min"
            ] = np.nan

            output[
                f"{prefix}__max"
            ] = np.nan

            output[
                f"{prefix}__median"
            ] = np.nan

        else:

            output[
                f"{prefix}__mean"
            ] = float(
                valid.mean()
            )

            output[
                f"{prefix}__std"
            ] = float(
                valid.std(
                    ddof=0
                )
            )

            output[
                f"{prefix}__min"
            ] = float(
                valid.min()
            )

            output[
                f"{prefix}__max"
            ] = float(
                valid.max()
            )

            output[
                f"{prefix}__median"
            ] = float(
                valid.median()
            )

        output[
            f"{prefix}__missing_ratio"
        ] = float(
            x.isna().mean()
        )

        output[
            f"{prefix}__nunique"
        ] = int(
            valid.nunique()
        )

    # --------------------------------------------------------
    # Categorical features
    # --------------------------------------------------------

    for feature in categorical_features:

        x = (
            window_df[feature]
        )

        nonmissing = (
            x.dropna()
            .astype(str)
        )

        mode_value = safe_mode(
            x
        )

        output[
            f"{feature}__mode"
        ] = mode_value

        output[
            f"{feature}__nunique"
        ] = int(
            nonmissing.nunique()
        )

        output[
            f"{feature}__missing_ratio"
        ] = float(
            x.isna().mean()
        )

        if len(nonmissing) == 0:

            output[
                f"{feature}__mode_ratio"
            ] = 0.0

        else:

            output[
                f"{feature}__mode_ratio"
            ] = float(
                (
                    nonmissing
                    == mode_value
                ).mean()
            )

    return output


# ============================================================
# 10. WINDOW CONSTRUCTION
# ============================================================

all_experiment_summaries = []
window_feature_configs = {}


print("\n" + "=" * 110)
print("BUILDING WINDOW-LEVEL DATASETS")
print("=" * 110)


for exp_name, spec in (
    EXPERIMENTS.items()
):

    print(
        "\n" + "=" * 110
    )

    print(
        exp_name
    )

    print(
        "=" * 110
    )

    candidate_features = (
        feature_config[
            exp_name
        ][
            "candidate_features"
        ]
    )

    timestamp_col = (
        spec["timestamp"]
    )

    window_size = int(
        spec["window_size"]
    )

    # --------------------------------------------------------
    # One experiment-wide gap threshold
    # --------------------------------------------------------

    gap_threshold = (
        calculate_gap_threshold(
            spec["sections"],
            timestamp_col
        )
    )

    print(
        f"Gap threshold      : "
        f"{gap_threshold:.6f} sec"
    )

    print(
        f"Window size        : "
        f"{window_size} observations"
    )

    # --------------------------------------------------------
    # Infer feature types
    # --------------------------------------------------------

    (
        numeric_features,
        categorical_features,
        type_diagnostics,
    ) = infer_feature_types(
        spec["sections"],
        candidate_features
    )

    print(
        f"Numeric features   : "
        f"{len(numeric_features)}"
    )

    print(
        f"Categorical features: "
        f"{len(categorical_features)}"
    )

    diagnostics_file = (
        TABLES_DIR
        / f"{exp_name}_window_feature_types.csv"
    )

    type_diagnostics.to_csv(
        diagnostics_file,
        index=False
    )

    # --------------------------------------------------------
    # Build windows section by section
    # --------------------------------------------------------

    window_rows = []

    section_stats = []

    global_window_id = 0

    for section_name in spec[
        "sections"
    ]:

        df = sections[
            section_name
        ].copy()

        attack_class = str(
            df["attack_class"]
            .iloc[0]
        )

        ts, segment_ids = (
            assign_segments(
                df,
                timestamp_col,
                gap_threshold
            )
        )

        df[
            "_timestamp_numeric"
        ] = ts.values

        df[
            "_segment_id"
        ] = segment_ids.values

        section_windows = 0
        dropped_rows = 0

        unique_segments = sorted(
            df[
                "_segment_id"
            ].unique()
        )

        for segment_id in (
            unique_segments
        ):

            segment = (
                df[
                    df[
                        "_segment_id"
                    ]
                    == segment_id
                ]
                .copy()
            )

            segment = (
                segment
                .sort_index()
            )

            n_full_windows = (
                len(segment)
                // window_size
            )

            used_rows = (
                n_full_windows
                * window_size
            )

            dropped_rows += (
                len(segment)
                - used_rows
            )

            for local_window in range(
                n_full_windows
            ):

                start = (
                    local_window
                    * window_size
                )

                end = (
                    start
                    + window_size
                )

                window = (
                    segment.iloc[
                        start:end
                    ]
                )

                features = (
                    aggregate_window(
                        window,
                        numeric_features,
                        categorical_features
                    )
                )

                record = {
                    "window_id":
                        global_window_id,

                    "section":
                        section_name,

                    "attack_class":
                        attack_class,

                    "segment_id":
                        int(
                            segment_id
                        ),

                    "window_index_in_segment":
                        local_window,

                    "window_size":
                        window_size,

                    "source_line_start":
                        int(
                            window[
                                "source_line"
                            ].iloc[0]
                        ),

                    "source_line_end":
                        int(
                            window[
                                "source_line"
                            ].iloc[-1]
                        ),

                    "timestamp_start":
                        float(
                            window[
                                "_timestamp_numeric"
                            ].iloc[0]
                        ),

                    "timestamp_end":
                        float(
                            window[
                                "_timestamp_numeric"
                            ].iloc[-1]
                        ),
                }

                record.update(
                    features
                )

                window_rows.append(
                    record
                )

                global_window_id += 1
                section_windows += 1

        section_stats.append({
            "experiment":
                exp_name,

            "section":
                section_name,

            "attack_class":
                attack_class,

            "rows":
                len(df),

            "segments":
                len(unique_segments),

            "windows":
                section_windows,

            "dropped_tail_rows":
                dropped_rows,
        })

    # --------------------------------------------------------
    # Build experiment dataframe
    # --------------------------------------------------------

    window_df = pd.DataFrame(
        window_rows
    )

    if window_df.empty:
        raise RuntimeError(
            f"No windows created for {exp_name}"
        )

    # --------------------------------------------------------
    # Feature columns
    # --------------------------------------------------------

    metadata_columns = {
        "window_id",
        "section",
        "attack_class",
        "segment_id",
        "window_index_in_segment",
        "window_size",
        "source_line_start",
        "source_line_end",
        "timestamp_start",
        "timestamp_end",
    }

    aggregated_features = [
        col
        for col in window_df.columns
        if col not in metadata_columns
    ]

    # --------------------------------------------------------
    # Duplicate window audit
    # --------------------------------------------------------

    duplicate_rows = int(
        window_df[
            aggregated_features
            + ["attack_class"]
        ]
        .duplicated()
        .sum()
    )

    # --------------------------------------------------------
    # Conflict audit
    #
    # Exact same aggregated feature vector
    # but more than one class.
    # --------------------------------------------------------

    conflict_work = (
        window_df[
            aggregated_features
            + ["attack_class"]
        ]
        .copy()
    )

    for col in aggregated_features:

        conflict_work[col] = (
            conflict_work[col]
            .astype("string")
            .fillna("<MISSING>")
        )

    grouped = (
        conflict_work
        .groupby(
            aggregated_features,
            dropna=False,
            sort=False
        )["attack_class"]
        .agg(
            label_count="nunique",
            row_count="size"
        )
        .reset_index()
    )

    conflicts = (
        grouped[
            grouped[
                "label_count"
            ] > 1
        ]
    )

    conflict_groups = len(
        conflicts
    )

    conflict_rows = int(
        conflicts[
            "row_count"
        ].sum()
    )

    # --------------------------------------------------------
    # Save window dataset
    # --------------------------------------------------------

    output_file = (
        WINDOWS_DIR
        / f"{exp_name}_windows.csv"
    )

    window_df.to_csv(
        output_file,
        index=False
    )

    # --------------------------------------------------------
    # Save section statistics
    # --------------------------------------------------------

    section_stats_df = pd.DataFrame(
        section_stats
    )

    section_stats_file = (
        TABLES_DIR
        / f"{exp_name}_window_section_stats.csv"
    )

    section_stats_df.to_csv(
        section_stats_file,
        index=False
    )

    # --------------------------------------------------------
    # Class distribution
    # --------------------------------------------------------

    class_counts = (
        window_df[
            "attack_class"
        ]
        .value_counts()
        .sort_index()
    )

    print(
        "\nSection statistics:"
    )

    print(
        section_stats_df[
            [
                "section",
                "attack_class",
                "segments",
                "windows",
                "dropped_tail_rows",
            ]
        ].to_string(
            index=False
        )
    )

    print(
        "\nWindow class distribution:"
    )

    for label, count in (
        class_counts.items()
    ):

        print(
            f"   {label:12s}: "
            f"{count}"
        )

    print(
        f"\nTotal windows       : "
        f"{len(window_df)}"
    )

    print(
        f"Aggregated features : "
        f"{len(aggregated_features)}"
    )

    print(
        f"Duplicate windows   : "
        f"{duplicate_rows}"
    )

    print(
        f"Conflicting groups  : "
        f"{conflict_groups}"
    )

    print(
        f"Rows in conflicts   : "
        f"{conflict_rows}"
    )

    # --------------------------------------------------------
    # Store configuration
    # --------------------------------------------------------

    window_feature_configs[
        exp_name
    ] = {
        "window_size":
            window_size,

        "gap_threshold_seconds":
            gap_threshold,

        "numeric_source_features":
            numeric_features,

        "categorical_source_features":
            categorical_features,

        "aggregated_features":
            aggregated_features,

        "aggregated_feature_count":
            len(
                aggregated_features
            ),

        "total_windows":
            len(
                window_df
            ),

        "duplicate_windows":
            duplicate_rows,

        "conflicting_feature_groups":
            conflict_groups,

        "rows_in_conflicting_groups":
            conflict_rows,
    }

    all_experiment_summaries.append({
        "experiment":
            exp_name,

        "window_size":
            window_size,

        "gap_threshold_seconds":
            gap_threshold,

        "windows":
            len(
                window_df
            ),

        "aggregated_features":
            len(
                aggregated_features
            ),

        "duplicate_windows":
            duplicate_rows,

        "conflicting_groups":
            conflict_groups,

        "rows_in_conflicting_groups":
            conflict_rows,
    })


# ============================================================
# 11. SAVE WINDOW CONFIGURATION
# ============================================================

window_config_file = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "window_feature_configuration.json"
)

with window_config_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        window_feature_configs,
        f,
        indent=2
    )


# ============================================================
# 12. SAVE OVERALL MANIFEST
# ============================================================

summary_df = pd.DataFrame(
    all_experiment_summaries
)

summary_file = (
    TABLES_DIR
    / "window_experiment_manifest.csv"
)

summary_df.to_csv(
    summary_file,
    index=False
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print(
    "\n" + "=" * 110
)

print(
    "WINDOW EXPERIMENT SUMMARY"
)

print(
    "=" * 110
)

print(
    summary_df.to_string(
        index=False
    )
)

print(
    "\n✅ Window datasets:",
    WINDOWS_DIR
)

print(
    "✅ Configuration:",
    window_config_file
)

print(
    "✅ Manifest:",
    summary_file
)

print(
    "\n⚠️ Timestamp and source_line are provenance only."
)

print(
    "⚠️ They will NOT enter the classifier."
)

print(
    "⚠️ No Train/Validation/Test split created yet."
)

print(
    "⚠️ No model trained yet."
)

In [ ]:
# ============================================================
# UAV PROJECT
# FROZEN CHRONOLOGICAL SEGMENT-GROUPED SPLIT
#
# Train / Validation / Locked Test ≈ 60 / 20 / 20
#
# IMPORTANT:
# - Entire temporal segments stay within one split.
# - Splitting is performed separately within each class.
# - Segment order is determined by original source-line order.
# - Test is assigned now but must remain LOCKED.
# - No feature preprocessing/model fitting is performed here.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

WINDOWS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "windows"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

SPLITS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. PRIMARY EXPERIMENTS ONLY
# ============================================================

PRIMARY_EXPERIMENTS = [
    "cyber_schema_A_3class",
    "physical_schema_A_3class",
]

TARGET_TRAIN = 0.60
TARGET_VALIDATION = 0.20
TARGET_TEST = 0.20


# ============================================================
# 3. METADATA — NEVER MODEL FEATURES
# ============================================================

METADATA_COLUMNS = {
    "window_id",
    "section",
    "attack_class",
    "segment_id",
    "window_index_in_segment",
    "window_size",
    "source_line_start",
    "source_line_end",
    "timestamp_start",
    "timestamp_end",
    "group_id",
    "split",
}


# ============================================================
# 4. CHOOSE CHRONOLOGICAL SPLIT BOUNDARIES
#
# For each class:
#
#   earliest segment block -> Train
#   middle segment block   -> Validation
#   latest segment block   -> Locked Test
#
# We search every valid pair of segment boundaries and select
# the pair whose WINDOW counts are closest to 60/20/20.
#
# At least one full segment is required in every partition.
# ============================================================

def choose_boundaries(group_table):

    group_table = (
        group_table
        .sort_values(
            "source_line_start"
        )
        .reset_index(drop=True)
    )

    n_groups = len(group_table)

    if n_groups < 3:
        raise ValueError(
            f"Need >=3 temporal segments; found {n_groups}"
        )

    counts = (
        group_table["windows"]
        .to_numpy()
    )

    total = counts.sum()

    best = None

    for i in range(
        1,
        n_groups - 1
    ):

        for j in range(
            i + 1,
            n_groups
        ):

            n_train = (
                counts[:i].sum()
            )

            n_val = (
                counts[i:j].sum()
            )

            n_test = (
                counts[j:].sum()
            )

            proportions = np.array([
                n_train / total,
                n_val / total,
                n_test / total,
            ])

            target = np.array([
                TARGET_TRAIN,
                TARGET_VALIDATION,
                TARGET_TEST,
            ])

            # Absolute deviation from desired window proportions
            score = float(
                np.abs(
                    proportions - target
                ).sum()
            )

            candidate = {
                "i": i,
                "j": j,
                "score": score,
                "train_windows":
                    int(n_train),
                "validation_windows":
                    int(n_val),
                "test_windows":
                    int(n_test),
                "train_ratio":
                    float(proportions[0]),
                "validation_ratio":
                    float(proportions[1]),
                "test_ratio":
                    float(proportions[2]),
            }

            if (
                best is None
                or candidate["score"]
                < best["score"]
            ):
                best = candidate

    return best


# ============================================================
# 5. ASSIGN SPLIT TO ONE EXPERIMENT
# ============================================================

def create_split(exp_name):

    window_file = (
        WINDOWS_DIR
        / f"{exp_name}_windows.csv"
    )

    if not window_file.exists():
        raise FileNotFoundError(
            window_file
        )

    df = pd.read_csv(
        window_file,
        low_memory=False
    )

    required = {
        "window_id",
        "section",
        "attack_class",
        "segment_id",
        "source_line_start",
        "source_line_end",
        "timestamp_start",
        "timestamp_end",
    }

    missing = (
        required
        - set(df.columns)
    )

    if missing:
        raise ValueError(
            f"{exp_name}: missing columns {missing}"
        )

    # --------------------------------------------------------
    # Globally unique group ID
    # segment_id alone is not globally unique.
    # --------------------------------------------------------

    df["group_id"] = (
        df["section"].astype(str)
        + "::segment_"
        + df["segment_id"].astype(str)
    )

    # --------------------------------------------------------
    # Build temporal-group table
    # --------------------------------------------------------

    group_table = (
        df
        .groupby(
            [
                "group_id",
                "section",
                "attack_class",
                "segment_id",
            ],
            as_index=False
        )
        .agg(
            windows=(
                "window_id",
                "size"
            ),
            source_line_start=(
                "source_line_start",
                "min"
            ),
            source_line_end=(
                "source_line_end",
                "max"
            ),
            timestamp_start=(
                "timestamp_start",
                "min"
            ),
            timestamp_end=(
                "timestamp_end",
                "max"
            ),
        )
    )

    split_map = {}
    boundary_records = []

    # --------------------------------------------------------
    # Split each class independently
    # --------------------------------------------------------

    for attack_class in sorted(
        group_table[
            "attack_class"
        ].unique()
    ):

        class_groups = (
            group_table[
                group_table[
                    "attack_class"
                ] == attack_class
            ]
            .sort_values(
                "source_line_start"
            )
            .reset_index(drop=True)
        )

        boundary = choose_boundaries(
            class_groups
        )

        i = boundary["i"]
        j = boundary["j"]

        train_groups = (
            class_groups.iloc[:i]
        )

        val_groups = (
            class_groups.iloc[i:j]
        )

        test_groups = (
            class_groups.iloc[j:]
        )

        for gid in (
            train_groups[
                "group_id"
            ]
        ):
            split_map[gid] = "train"

        for gid in (
            val_groups[
                "group_id"
            ]
        ):
            split_map[gid] = "validation"

        for gid in (
            test_groups[
                "group_id"
            ]
        ):
            split_map[gid] = "test"

        boundary_records.append({
            "experiment":
                exp_name,

            "attack_class":
                attack_class,

            "total_segments":
                len(class_groups),

            "train_segments":
                len(train_groups),

            "validation_segments":
                len(val_groups),

            "test_segments":
                len(test_groups),

            "total_windows":
                int(
                    class_groups[
                        "windows"
                    ].sum()
                ),

            **boundary,
        })

    # --------------------------------------------------------
    # Apply split map
    # --------------------------------------------------------

    df["split"] = (
        df["group_id"]
        .map(split_map)
    )

    if df["split"].isna().any():
        raise RuntimeError(
            f"{exp_name}: unassigned windows."
        )

    # --------------------------------------------------------
    # HARD LEAKAGE CHECK:
    # no temporal segment in multiple splits
    # --------------------------------------------------------

    leakage_check = (
        df
        .groupby(
            "group_id"
        )["split"]
        .nunique()
    )

    bad_groups = (
        leakage_check[
            leakage_check > 1
        ]
    )

    if len(bad_groups) > 0:
        raise RuntimeError(
            f"{exp_name}: segment leakage detected."
        )

    # --------------------------------------------------------
    # Each class must exist in every split
    # --------------------------------------------------------

    contingency = pd.crosstab(
        df["attack_class"],
        df["split"]
    )

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        if split_name not in contingency.columns:
            raise RuntimeError(
                f"{exp_name}: no {split_name} data."
            )

        if (
            contingency[
                split_name
            ] == 0
        ).any():

            raise RuntimeError(
                f"{exp_name}: a class is absent "
                f"from {split_name}."
            )

    # --------------------------------------------------------
    # Verify temporal ordering WITHIN EACH CLASS:
    #
    # max Train source line
    #       <
    # min Validation source line
    #       <
    # min Test source line
    # --------------------------------------------------------

    temporal_checks = []

    for attack_class in sorted(
        df[
            "attack_class"
        ].unique()
    ):

        class_df = (
            df[
                df[
                    "attack_class"
                ] == attack_class
            ]
        )

        train_max = (
            class_df[
                class_df["split"]
                == "train"
            ][
                "source_line_end"
            ].max()
        )

        val_min = (
            class_df[
                class_df["split"]
                == "validation"
            ][
                "source_line_start"
            ].min()
        )

        val_max = (
            class_df[
                class_df["split"]
                == "validation"
            ][
                "source_line_end"
            ].max()
        )

        test_min = (
            class_df[
                class_df["split"]
                == "test"
            ][
                "source_line_start"
            ].min()
        )

        valid_order = (
            train_max < val_min
            and
            val_max < test_min
        )

        temporal_checks.append({
            "experiment":
                exp_name,

            "attack_class":
                attack_class,

            "train_max_source_line":
                int(train_max),

            "validation_min_source_line":
                int(val_min),

            "validation_max_source_line":
                int(val_max),

            "test_min_source_line":
                int(test_min),

            "chronological_order_valid":
                bool(valid_order),
        })

        if not valid_order:
            raise RuntimeError(
                f"{exp_name}/{attack_class}: "
                f"chronological ordering failed."
            )

    # --------------------------------------------------------
    # Feature columns
    # --------------------------------------------------------

    feature_columns = [
        col
        for col in df.columns
        if col not in METADATA_COLUMNS
    ]

    # attack_class got excluded via metadata set.
    if "attack_class" in feature_columns:
        raise RuntimeError(
            "Target leakage: attack_class in features."
        )

    # --------------------------------------------------------
    # Save FULL split mapping only.
    #
    # We intentionally do NOT print Test rows/features.
    # --------------------------------------------------------

    split_map_file = (
        SPLITS_DIR
        / f"{exp_name}_window_split_map.csv"
    )

    df[
        [
            "window_id",
            "group_id",
            "section",
            "segment_id",
            "attack_class",
            "split",
            "source_line_start",
            "source_line_end",
            "timestamp_start",
            "timestamp_end",
        ]
    ].to_csv(
        split_map_file,
        index=False
    )

    # --------------------------------------------------------
    # Save train and validation tables for development.
    #
    # Do NOT write a separate test feature table.
    # Test stays accessible only through the original
    # windows file + frozen split map when final evaluation
    # is explicitly performed.
    # --------------------------------------------------------

    development_columns = (
        feature_columns
        + ["attack_class"]
    )

    train_df = (
        df[
            df["split"]
            == "train"
        ][
            development_columns
        ]
        .copy()
    )

    val_df = (
        df[
            df["split"]
            == "validation"
        ][
            development_columns
        ]
        .copy()
    )

    train_file = (
        SPLITS_DIR
        / f"{exp_name}_train.csv"
    )

    val_file = (
        SPLITS_DIR
        / f"{exp_name}_validation.csv"
    )

    train_df.to_csv(
        train_file,
        index=False
    )

    val_df.to_csv(
        val_file,
        index=False
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    split_summary = (
        df
        .groupby(
            [
                "split",
                "attack_class"
            ]
        )
        .agg(
            windows=(
                "window_id",
                "size"
            ),
            temporal_groups=(
                "group_id",
                "nunique"
            ),
        )
        .reset_index()
    )

    return {
        "dataframe":
            df,

        "group_table":
            group_table,

        "split_summary":
            split_summary,

        "boundary_records":
            pd.DataFrame(
                boundary_records
            ),

        "temporal_checks":
            pd.DataFrame(
                temporal_checks
            ),

        "feature_columns":
            feature_columns,

        "split_map_file":
            split_map_file,

        "train_file":
            train_file,

        "validation_file":
            val_file,
    }


# ============================================================
# 6. CREATE BOTH PRIMARY SPLITS
# ============================================================

results = {}

all_boundaries = []
all_temporal_checks = []
all_split_summaries = []

print(
    "=" * 110
)

print(
    "CREATING FROZEN SEGMENT-GROUPED SPLITS"
)

print(
    "=" * 110
)


for exp_name in (
    PRIMARY_EXPERIMENTS
):

    result = create_split(
        exp_name
    )

    results[
        exp_name
    ] = result

    boundary_df = (
        result[
            "boundary_records"
        ]
    )

    temporal_df = (
        result[
            "temporal_checks"
        ]
    )

    split_summary = (
        result[
            "split_summary"
        ]
        .copy()
    )

    split_summary.insert(
        0,
        "experiment",
        exp_name
    )

    all_boundaries.append(
        boundary_df
    )

    all_temporal_checks.append(
        temporal_df
    )

    all_split_summaries.append(
        split_summary
    )

    print(
        "\n" + "=" * 110
    )

    print(exp_name)

    print(
        "=" * 110
    )

    print(
        "\nClass-level boundary allocation:"
    )

    print(
        boundary_df[
            [
                "attack_class",
                "total_segments",
                "train_segments",
                "validation_segments",
                "test_segments",
                "total_windows",
                "train_windows",
                "validation_windows",
                "test_windows",
                "train_ratio",
                "validation_ratio",
                "test_ratio",
            ]
        ].to_string(
            index=False
        )
    )

    print(
        "\nSplit distribution:"
    )

    print(
        split_summary.to_string(
            index=False
        )
    )

    print(
        "\n✅ No temporal-group overlap."
    )

    print(
        "✅ Chronological order preserved "
        "within every class."
    )

    print(
        "✅ Train file created."
    )

    print(
        "✅ Validation file created."
    )

    print(
        "🔒 Locked Test assigned but "
        "NO standalone test feature file created."
    )


# ============================================================
# 7. SAVE AUDIT TABLES
# ============================================================

boundaries_df = pd.concat(
    all_boundaries,
    ignore_index=True
)

temporal_checks_df = pd.concat(
    all_temporal_checks,
    ignore_index=True
)

split_manifest_df = pd.concat(
    all_split_summaries,
    ignore_index=True
)


boundaries_file = (
    TABLES_DIR
    / "frozen_split_class_boundaries.csv"
)

temporal_checks_file = (
    TABLES_DIR
    / "frozen_split_temporal_checks.csv"
)

split_manifest_file = (
    TABLES_DIR
    / "frozen_split_manifest.csv"
)


boundaries_df.to_csv(
    boundaries_file,
    index=False
)

temporal_checks_df.to_csv(
    temporal_checks_file,
    index=False
)

split_manifest_df.to_csv(
    split_manifest_file,
    index=False
)


# ============================================================
# 8. SPLIT CONFIGURATION + HASH
# ============================================================

configuration = {
    "strategy": (
        "class-stratified chronological "
        "segment-grouped blocked split"
    ),

    "target_proportions": {
        "train": TARGET_TRAIN,
        "validation":
            TARGET_VALIDATION,
        "test": TARGET_TEST,
    },

    "group_definition": (
        "section + temporal segment_id"
    ),

    "ordering_variable": (
        "original source_line order"
    ),

    "test_policy": (
        "LOCKED — no model selection, "
        "feature selection, preprocessing fitting, "
        "calibration fitting, or hyperparameter tuning"
    ),

    "secondary_experiment_policy": (
        "cyber_schema_B_attack_type is excluded "
        "from the primary 3-way frozen split because "
        "FDI contains only two temporal segments."
    ),
}


config_json = json.dumps(
    configuration,
    sort_keys=True,
    indent=2
)

config_hash = hashlib.sha256(
    config_json.encode(
        "utf-8"
    )
).hexdigest()


configuration[
    "sha256"
] = config_hash


config_file = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "frozen_split_configuration.json"
)

with config_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        configuration,
        f,
        indent=2
    )


# ============================================================
# 9. FINAL GLOBAL QA
# ============================================================

print(
    "\n" + "=" * 110
)

print(
    "GLOBAL SPLIT QA"
)

print(
    "=" * 110
)


for exp_name, result in (
    results.items()
):

    df = result[
        "dataframe"
    ]

    train_groups = set(
        df.loc[
            df["split"]
            == "train",
            "group_id"
        ]
    )

    val_groups = set(
        df.loc[
            df["split"]
            == "validation",
            "group_id"
        ]
    )

    test_groups = set(
        df.loc[
            df["split"]
            == "test",
            "group_id"
        ]
    )

    tv = (
        train_groups
        & val_groups
    )

    tt = (
        train_groups
        & test_groups
    )

    vt = (
        val_groups
        & test_groups
    )

    print(
        f"\n{exp_name}"
    )

    print(
        "Train ∩ Validation:",
        len(tv)
    )

    print(
        "Train ∩ Test      :",
        len(tt)
    )

    print(
        "Validation ∩ Test :",
        len(vt)
    )

    assert len(tv) == 0
    assert len(tt) == 0
    assert len(vt) == 0


# ============================================================
# 10. FINAL
# ============================================================

print(
    "\n" + "=" * 110
)

print(
    "FROZEN SPLIT COMPLETE"
)

print(
    "=" * 110
)

print(
    "✅ Split manifest:",
    split_manifest_file
)

print(
    "✅ Boundary audit:",
    boundaries_file
)

print(
    "✅ Temporal QA:",
    temporal_checks_file
)

print(
    "✅ Configuration:",
    config_file
)

print(
    "✅ Configuration SHA256:",
    config_hash
)

print(
    "\n🔒 LOCKED TEST CREATED."
)

print(
    "🔒 Do not inspect test features, "
    "predictions, or metrics until the "
    "entire model pipeline is frozen."
)

print(
    "\nNext development data:"
)

for exp_name, result in (
    results.items()
):

    print(
        f"\n{exp_name}"
    )

    print(
        "Train      :",
        result[
            "train_file"
        ]
    )

    print(
        "Validation :",
        result[
            "validation_file"
        ]
    )

In [ ]:
# ============================================================
# UAV PROJECT
# DEVELOPMENT-ONLY BASELINE MODELING
#
# Models:
#   1. Logistic Regression
#   2. Random Forest
#   3. XGBoost
#
# IMPORTANT:
# - Train only on frozen TRAIN.
# - Evaluate only on frozen VALIDATION.
# - Locked Test is NEVER loaded in this cell.
# - No hyperparameter tuning is performed.
# ============================================================

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    classification_report,
    confusion_matrix,
)

from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier
import joblib


warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
    / "development"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. PRIMARY EXPERIMENTS
# ============================================================

EXPERIMENTS = [
    "cyber_schema_A_3class",
    "physical_schema_A_3class",
]


# ============================================================
# 3. MULTICLASS BRIER SCORE
#
# Mean, across samples, of:
#
# sum_k (p_k - y_k)^2
# ============================================================

def multiclass_brier_score(
    y_true_encoded,
    probabilities,
    n_classes
):

    y_onehot = np.eye(
        n_classes
    )[y_true_encoded]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - y_onehot
                ) ** 2,
                axis=1
            )
        )
    )


# ============================================================
# 4. EQUAL-WIDTH ECE
# ============================================================

def expected_calibration_error(
    y_true_encoded,
    probabilities,
    n_bins=15
):

    confidence = (
        probabilities.max(
            axis=1
        )
    )

    prediction = (
        probabilities.argmax(
            axis=1
        )
    )

    correctness = (
        prediction
        == y_true_encoded
    ).astype(float)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for i in range(
        n_bins
    ):

        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (confidence >= lower)
                &
                (confidence <= upper)
            )
        else:
            mask = (
                (confidence >= lower)
                &
                (confidence < upper)
            )

        if not np.any(mask):
            continue

        bin_accuracy = (
            correctness[mask]
            .mean()
        )

        bin_confidence = (
            confidence[mask]
            .mean()
        )

        bin_weight = (
            mask.mean()
        )

        ece += (
            bin_weight
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(ece)


# ============================================================
# 5. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model_name,
    experiment_name,
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    label_encoder,
    sample_weight=None,
):

    start = time.time()

    # --------------------------------------------------------
    # Fit
    # --------------------------------------------------------

    if sample_weight is None:

        model.fit(
            X_train,
            y_train
        )

    else:

        # Pipeline models require model__sample_weight
        if isinstance(
            model,
            Pipeline
        ):

            model.fit(
                X_train,
                y_train,
                model__sample_weight=
                    sample_weight
            )

        else:

            model.fit(
                X_train,
                y_train,
                sample_weight=
                    sample_weight
            )

    training_seconds = (
        time.time()
        - start
    )

    # --------------------------------------------------------
    # Validation inference
    # --------------------------------------------------------

    inference_start = time.time()

    pred = model.predict(
        X_val
    )

    proba = model.predict_proba(
        X_val
    )

    inference_seconds = (
        time.time()
        - inference_start
    )

    n_classes = len(
        label_encoder.classes_
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics = {
        "experiment":
            experiment_name,

        "model":
            model_name,

        "train_rows":
            len(X_train),

        "validation_rows":
            len(X_val),

        "features":
            X_train.shape[1],

        "accuracy":
            accuracy_score(
                y_val,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_val,
                pred
            ),

        "macro_precision":
            precision_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                y_val,
                pred,
                average="weighted",
                zero_division=0
            ),

        "log_loss":
            log_loss(
                y_val,
                proba,
                labels=np.arange(
                    n_classes
                )
            ),

        "multiclass_brier":
            multiclass_brier_score(
                y_val,
                proba,
                n_classes
            ),

        "ece_15":
            expected_calibration_error(
                y_val,
                proba,
                n_bins=15
            ),

        "mean_confidence":
            float(
                proba.max(
                    axis=1
                ).mean()
            ),

        "training_seconds":
            training_seconds,

        "inference_seconds":
            inference_seconds,

        "inference_ms_per_sample":
            (
                inference_seconds
                / len(X_val)
                * 1000
            ),
    }

    # --------------------------------------------------------
    # Per-class metrics
    # --------------------------------------------------------

    report = classification_report(
        y_val,
        pred,
        labels=np.arange(
            n_classes
        ),
        target_names=
            label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )

    per_class_rows = []

    for class_name in (
        label_encoder.classes_
    ):

        row = report[
            class_name
        ]

        per_class_rows.append({
            "experiment":
                experiment_name,

            "model":
                model_name,

            "class":
                class_name,

            "precision":
                row["precision"],

            "recall":
                row["recall"],

            "f1":
                row["f1-score"],

            "support":
                int(
                    row["support"]
                ),
        })

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_val,
        pred,
        labels=np.arange(
            n_classes
        )
    )

    cm_df = pd.DataFrame(
        cm,
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    )

    return (
        metrics,
        pd.DataFrame(
            per_class_rows
        ),
        cm_df,
        model
    )


# ============================================================
# 6. RESULTS CONTAINERS
# ============================================================

all_results = []
all_class_results = []


# ============================================================
# 7. RUN EACH PRIMARY EXPERIMENT
# ============================================================

for experiment_name in (
    EXPERIMENTS
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        experiment_name
    )

    print(
        "=" * 110
    )

    # --------------------------------------------------------
    # Load ONLY train + validation
    # --------------------------------------------------------

    train_file = (
        SPLITS_DIR
        / f"{experiment_name}_train.csv"
    )

    val_file = (
        SPLITS_DIR
        / f"{experiment_name}_validation.csv"
    )

    train_df = pd.read_csv(
        train_file,
        low_memory=False
    )

    val_df = pd.read_csv(
        val_file,
        low_memory=False
    )

    assert (
        "attack_class"
        in train_df.columns
    )

    assert (
        "attack_class"
        in val_df.columns
    )

    # --------------------------------------------------------
    # Predictor definition
    # --------------------------------------------------------

    feature_columns = [
        col
        for col in train_df.columns
        if col != "attack_class"
    ]

    assert set(
        feature_columns
    ) == set(
        col
        for col in val_df.columns
        if col != "attack_class"
    )

    X_train = (
        train_df[
            feature_columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    X_val = (
        val_df[
            feature_columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    # --------------------------------------------------------
    # Label encoding — FIT ON TRAIN ONLY
    # --------------------------------------------------------

    label_encoder = LabelEncoder()

    y_train = (
        label_encoder
        .fit_transform(
            train_df[
                "attack_class"
            ]
        )
    )

    unknown_val_labels = (
        set(
            val_df[
                "attack_class"
            ].unique()
        )
        -
        set(
            label_encoder.classes_
        )
    )

    if unknown_val_labels:

        raise RuntimeError(
            f"Unknown validation classes: "
            f"{unknown_val_labels}"
        )

    y_val = (
        label_encoder
        .transform(
            val_df[
                "attack_class"
            ]
        )
    )

    print(
        "Classes:",
        list(
            label_encoder.classes_
        )
    )

    print(
        "Train rows:",
        len(
            train_df
        )
    )

    print(
        "Validation rows:",
        len(
            val_df
        )
    )

    print(
        "Features:",
        len(
            feature_columns
        )
    )

    print(
        "\nTrain distribution:"
    )

    print(
        train_df[
            "attack_class"
        ]
        .value_counts()
        .sort_index()
    )

    print(
        "\nValidation distribution:"
    )

    print(
        val_df[
            "attack_class"
        ]
        .value_counts()
        .sort_index()
    )

    # --------------------------------------------------------
    # Balanced sample weights for XGBoost
    # --------------------------------------------------------

    balanced_weights = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train
        )
    )

    # ========================================================
    # MODEL 1 — LOGISTIC REGRESSION
    # ========================================================

    logistic_model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            )
        ),
    ])

    # ========================================================
    # MODEL 2 — RANDOM FOREST
    # ========================================================

    rf_model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            )
        ),
    ])

    # ========================================================
    # MODEL 3 — XGBOOST
    # ========================================================

    xgb_model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            XGBClassifier(
                objective="multi:softprob",
                num_class=len(
                    label_encoder.classes_
                ),
                n_estimators=400,
                learning_rate=0.05,
                max_depth=5,
                min_child_weight=1,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.0,
                reg_lambda=1.0,
                tree_method="hist",
                eval_metric="mlogloss",
                n_jobs=-1,
                random_state=42,
            )
        ),
    ])

    models = [
        (
            "Logistic Regression",
            logistic_model,
            None
        ),
        (
            "Random Forest",
            rf_model,
            None
        ),
        (
            "XGBoost",
            xgb_model,
            balanced_weights
        ),
    ]

    # --------------------------------------------------------
    # Train + evaluate
    # --------------------------------------------------------

    for (
        model_name,
        model,
        sample_weight
    ) in models:

        print(
            "\n"
            + "-" * 90
        )

        print(
            "Training:",
            model_name
        )

        (
            metrics,
            per_class,
            cm,
            fitted_model
        ) = evaluate_model(
            model_name=
                model_name,

            experiment_name=
                experiment_name,

            model=model,

            X_train=X_train,
            y_train=y_train,

            X_val=X_val,
            y_val=y_val,

            label_encoder=
                label_encoder,

            sample_weight=
                sample_weight,
        )

        all_results.append(
            metrics
        )

        all_class_results.append(
            per_class
        )

        print(
            f"Accuracy          : "
            f"{metrics['accuracy']:.4f}"
        )

        print(
            f"Balanced Accuracy : "
            f"{metrics['balanced_accuracy']:.4f}"
        )

        print(
            f"Macro F1          : "
            f"{metrics['macro_f1']:.4f}"
        )

        print(
            f"Log Loss          : "
            f"{metrics['log_loss']:.4f}"
        )

        print(
            f"Brier             : "
            f"{metrics['multiclass_brier']:.4f}"
        )

        print(
            f"ECE-15            : "
            f"{metrics['ece_15']:.4f}"
        )

        # ----------------------------------------------------
        # Save confusion matrix
        # ----------------------------------------------------

        safe_model_name = (
            model_name
            .lower()
            .replace(
                " ",
                "_"
            )
        )

        cm_file = (
            TABLES_DIR
            / (
                f"{experiment_name}_"
                f"{safe_model_name}_"
                f"validation_confusion_matrix.csv"
            )
        )

        cm.to_csv(
            cm_file
        )

        # ----------------------------------------------------
        # Save development model
        #
        # These are NOT final models yet.
        # ----------------------------------------------------

        model_file = (
            MODELS_DIR
            / (
                f"{experiment_name}_"
                f"{safe_model_name}.joblib"
            )
        )

        joblib.dump(
            fitted_model,
            model_file
        )

    # --------------------------------------------------------
    # Save label encoder
    # --------------------------------------------------------

    encoder_file = (
        MODELS_DIR
        / (
            f"{experiment_name}_"
            "label_encoder.joblib"
        )
    )

    joblib.dump(
        label_encoder,
        encoder_file
    )


# ============================================================
# 8. SAVE GLOBAL VALIDATION RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_results
)

results_file = (
    TABLES_DIR
    / "development_validation_model_results.csv"
)

results_df.to_csv(
    results_file,
    index=False
)


class_results_df = pd.concat(
    all_class_results,
    ignore_index=True
)

class_results_file = (
    TABLES_DIR
    / "development_validation_class_metrics.csv"
)

class_results_df.to_csv(
    class_results_file,
    index=False
)


# ============================================================
# 9. RANK WITHIN EACH EXPERIMENT
#
# Primary ranking metric = Macro F1
# ============================================================

ranked = (
    results_df
    .sort_values(
        [
            "experiment",
            "macro_f1",
            "log_loss",
        ],
        ascending=[
            True,
            False,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "VALIDATION MODEL COMPARISON"
)

print(
    "=" * 115
)

print(
    ranked[
        [
            "experiment",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "weighted_f1",
            "log_loss",
            "multiclass_brier",
            "ece_15",
            "training_seconds",
            "inference_ms_per_sample",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 10. BEST MODEL PER EXPERIMENT
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "BEST BASELINE PER EXPERIMENT — BY VALIDATION MACRO F1"
)

print(
    "=" * 115
)


for experiment_name in (
    EXPERIMENTS
):

    subset = (
        results_df[
            results_df[
                "experiment"
            ]
            == experiment_name
        ]
        .sort_values(
            [
                "macro_f1",
                "log_loss"
            ],
            ascending=[
                False,
                True
            ]
        )
    )

    best = (
        subset.iloc[0]
    )

    print(
        f"\n{experiment_name}"
    )

    print(
        f"Best model : "
        f"{best['model']}"
    )

    print(
        f"Macro F1   : "
        f"{best['macro_f1']:.6f}"
    )

    print(
        f"Balanced Acc: "
        f"{best['balanced_accuracy']:.6f}"
    )

    print(
        f"Log Loss   : "
        f"{best['log_loss']:.6f}"
    )


# ============================================================
# 11. FINAL
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "DEVELOPMENT BASELINE STAGE COMPLETE"
)

print(
    "=" * 115
)

print(
    "✅ Results:",
    results_file
)

print(
    "✅ Per-class metrics:",
    class_results_file
)

print(
    "✅ Development models:",
    MODELS_DIR
)

print(
    "\n🔒 LOCKED TEST WAS NOT LOADED."
)

print(
    "Next step = inspect validation behavior "
    "and decide whether feature reduction / "
    "calibration is justified."
)

In [ ]:
# ============================================================
# UAV PROJECT
# DEVELOPMENT DIAGNOSTIC:
# XGBOOST GAIN + SINGLE-SOURCE FEATURE TEST
# + TOP-FEATURE ABLATION
#
# LOCKED TEST IS NOT LOADED
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
    / "development"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


EXPERIMENTS = [
    "cyber_schema_A_3class",
    "physical_schema_A_3class",
]


# ============================================================
# 2. HELPERS
# ============================================================

def source_feature_name(
    aggregated_feature
):
    return aggregated_feature.split(
        "__",
        1
    )[0]


def evaluate_predictions(
    y_true,
    pred
):
    return {
        "accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                pred,
                average="macro",
                zero_division=0
            ),
    }


# ============================================================
# 3. GLOBAL RESULT CONTAINERS
# ============================================================

all_gain_rows = []
all_single_source_rows = []
all_ablation_rows = []


# ============================================================
# 4. LOOP OVER PRIMARY EXPERIMENTS
# ============================================================

for experiment_name in EXPERIMENTS:

    print(
        "\n"
        + "=" * 115
    )

    print(
        experiment_name
    )

    print(
        "=" * 115
    )

    # --------------------------------------------------------
    # Load train + validation only
    # --------------------------------------------------------

    train_df = pd.read_csv(
        SPLITS_DIR
        / f"{experiment_name}_train.csv",
        low_memory=False
    )

    val_df = pd.read_csv(
        SPLITS_DIR
        / f"{experiment_name}_validation.csv",
        low_memory=False
    )

    feature_columns = [
        c
        for c in train_df.columns
        if c != "attack_class"
    ]

    X_train = (
        train_df[
            feature_columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    X_val = (
        val_df[
            feature_columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    # --------------------------------------------------------
    # Load frozen development label encoder
    # --------------------------------------------------------

    encoder = joblib.load(
        MODELS_DIR
        / (
            f"{experiment_name}_"
            "label_encoder.joblib"
        )
    )

    y_train = encoder.transform(
        train_df[
            "attack_class"
        ]
    )

    y_val = encoder.transform(
        val_df[
            "attack_class"
        ]
    )

    # --------------------------------------------------------
    # Load fitted development XGBoost
    # --------------------------------------------------------

    xgb_pipeline = joblib.load(
        MODELS_DIR
        / (
            f"{experiment_name}_"
            "xgboost.joblib"
        )
    )

    xgb_model = (
        xgb_pipeline
        .named_steps[
            "model"
        ]
    )

    booster = (
        xgb_model
        .get_booster()
    )


    # ========================================================
    # 5. XGBOOST TOTAL GAIN
    # ========================================================

    raw_gain = booster.get_score(
        importance_type="total_gain"
    )

    gain_rows = []

    for key, gain in raw_gain.items():

        # XGBoost usually returns f0, f1, ...
        if key.startswith("f"):
            index = int(
                key[1:]
            )
        else:
            continue

        if index >= len(
            feature_columns
        ):
            continue

        aggregated_feature = (
            feature_columns[index]
        )

        source_feature = (
            source_feature_name(
                aggregated_feature
            )
        )

        gain_rows.append({
            "experiment":
                experiment_name,

            "aggregated_feature":
                aggregated_feature,

            "source_feature":
                source_feature,

            "total_gain":
                float(gain),
        })

    gain_df = pd.DataFrame(
        gain_rows
    )

    if gain_df.empty:
        raise RuntimeError(
            "Could not extract XGBoost gain."
        )

    total_gain_sum = (
        gain_df[
            "total_gain"
        ].sum()
    )

    gain_df[
        "gain_fraction"
    ] = (
        gain_df[
            "total_gain"
        ]
        / total_gain_sum
    )

    source_gain = (
        gain_df
        .groupby(
            [
                "experiment",
                "source_feature"
            ],
            as_index=False
        )
        .agg(
            total_gain=(
                "total_gain",
                "sum"
            )
        )
    )

    source_gain[
        "gain_fraction"
    ] = (
        source_gain[
            "total_gain"
        ]
        /
        source_gain[
            "total_gain"
        ].sum()
    )

    source_gain = (
        source_gain
        .sort_values(
            "gain_fraction",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    all_gain_rows.append(
        source_gain
    )

    print(
        "\nTOP SOURCE FEATURES BY XGBOOST TOTAL GAIN"
    )

    print(
        source_gain
        .head(12)
        .to_string(
            index=False
        )
    )


    # ========================================================
    # 6. SINGLE-SOURCE-FAMILY DIAGNOSTIC
    #
    # Each raw source feature contributes several
    # aggregated statistics:
    # mean/std/min/max/median/missing/nunique
    #
    # We test whether ONE source variable alone can nearly
    # solve the task using a shallow decision tree.
    # ========================================================

    source_groups = {}

    for col in feature_columns:

        source = (
            source_feature_name(
                col
            )
        )

        source_groups.setdefault(
            source,
            []
        ).append(
            col
        )

    single_source_rows = []

    for source, cols in (
        source_groups.items()
    ):

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                DecisionTreeClassifier(
                    max_depth=3,
                    min_samples_leaf=5,
                    class_weight="balanced",
                    random_state=42
                )
            ),
        ])

        model.fit(
            X_train[
                cols
            ],
            y_train
        )

        pred = model.predict(
            X_val[
                cols
            ]
        )

        scores = (
            evaluate_predictions(
                y_val,
                pred
            )
        )

        row = {
            "experiment":
                experiment_name,

            "source_feature":
                source,

            "aggregated_columns":
                len(cols),

            **scores
        }

        single_source_rows.append(
            row
        )

    single_source_df = (
        pd.DataFrame(
            single_source_rows
        )
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy"
            ],
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    all_single_source_rows.append(
        single_source_df
    )

    print(
        "\nTOP SINGLE-SOURCE-FAMILY RESULTS"
    )

    print(
        single_source_df
        .head(12)
        .to_string(
            index=False
        )
    )


    # ========================================================
    # 7. DIAGNOSTIC ABLATION
    #
    # Remove the top 1 and top 2 source features by XGB gain
    # and retrain the SAME fixed XGBoost configuration.
    #
    # This is not hyperparameter tuning.
    # ========================================================

    ranked_sources = (
        source_gain[
            "source_feature"
        ].tolist()
    )

    ablation_sets = {
        "all_features":
            [],

        "drop_top1":
            ranked_sources[:1],

        "drop_top2":
            ranked_sources[:2],
    }

    weights = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train
        )
    )

    for ablation_name, removed_sources in (
        ablation_sets.items()
    ):

        kept_features = [
            col
            for col in feature_columns
            if source_feature_name(
                col
            )
            not in removed_sources
        ]

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                XGBClassifier(
                    objective=
                        "multi:softprob",

                    num_class=
                        len(
                            encoder.classes_
                        ),

                    n_estimators=400,
                    learning_rate=0.05,
                    max_depth=5,
                    min_child_weight=1,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_alpha=0.0,
                    reg_lambda=1.0,
                    tree_method="hist",
                    eval_metric="mlogloss",
                    n_jobs=-1,
                    random_state=42,
                )
            ),
        ])

        model.fit(
            X_train[
                kept_features
            ],
            y_train,
            model__sample_weight=
                weights
        )

        pred = model.predict(
            X_val[
                kept_features
            ]
        )

        scores = (
            evaluate_predictions(
                y_val,
                pred
            )
        )

        row = {
            "experiment":
                experiment_name,

            "ablation":
                ablation_name,

            "removed_source_features":
                ",".join(
                    removed_sources
                ),

            "remaining_aggregated_features":
                len(
                    kept_features
                ),

            **scores
        }

        all_ablation_rows.append(
            row
        )

    ablation_df = pd.DataFrame(
        [
            row
            for row in all_ablation_rows
            if row[
                "experiment"
            ] == experiment_name
        ]
    )

    print(
        "\nXGBOOST TOP-FEATURE ABLATION"
    )

    print(
        ablation_df.to_string(
            index=False
        )
    )


# ============================================================
# 8. SAVE RESULTS
# ============================================================

gain_all_df = pd.concat(
    all_gain_rows,
    ignore_index=True
)

single_source_all_df = pd.concat(
    all_single_source_rows,
    ignore_index=True
)

ablation_all_df = pd.DataFrame(
    all_ablation_rows
)


gain_file = (
    TABLES_DIR
    / "development_xgboost_source_gain.csv"
)

single_source_file = (
    TABLES_DIR
    / "development_single_source_diagnostic.csv"
)

ablation_file = (
    TABLES_DIR
    / "development_xgboost_source_ablation.csv"
)


gain_all_df.to_csv(
    gain_file,
    index=False
)

single_source_all_df.to_csv(
    single_source_file,
    index=False
)

ablation_all_df.to_csv(
    ablation_file,
    index=False
)


# ============================================================
# 9. FINAL FLAGS
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "DIAGNOSTIC FLAGS"
)

print(
    "=" * 115
)


for experiment_name in EXPERIMENTS:

    subset = (
        single_source_all_df[
            single_source_all_df[
                "experiment"
            ]
            == experiment_name
        ]
        .sort_values(
            "macro_f1",
            ascending=False
        )
    )

    best = (
        subset.iloc[0]
    )

    print(
        f"\n{experiment_name}"
    )

    print(
        "Best single source:",
        best[
            "source_feature"
        ]
    )

    print(
        "Single-source Macro F1:",
        f"{best['macro_f1']:.6f}"
    )

    if best[
        "macro_f1"
    ] >= 0.95:

        print(
            "⚠️ STRONG CONFOUNDING / "
            "LEAKAGE-PROXY WARNING"
        )

    elif best[
        "macro_f1"
    ] >= 0.85:

        print(
            "⚠️ Very strong single-feature "
            "separability — investigate."
        )

    else:

        print(
            "✅ No single source feature "
            "nearly solves the task."
        )


print(
    "\n"
    + "=" * 115
)

print(
    "DIAGNOSTIC COMPLETE"
)

print(
    "=" * 115
)

print(
    "✅ Gain table:",
    gain_file
)

print(
    "✅ Single-source diagnostic:",
    single_source_file
)

print(
    "✅ Ablation:",
    ablation_file
)

print(
    "\n🔒 LOCKED TEST WAS NOT LOADED."
)

In [ ]:
# ============================================================
# UAV PROJECT
# PHYSICAL BAROMETER CONFOUNDING AUDIT
#
# DEVELOPMENT ONLY:
# - Train + Validation only
# - Locked Test is NOT loaded
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
)
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

WINDOWS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "windows"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
    / "development"
)

EXPERIMENT = "physical_schema_A_3class"


# ============================================================
# 2. LOAD TRAIN + VALIDATION ONLY
# ============================================================

train_df = pd.read_csv(
    SPLITS_DIR / f"{EXPERIMENT}_train.csv",
    low_memory=False
)

val_df = pd.read_csv(
    SPLITS_DIR / f"{EXPERIMENT}_validation.csv",
    low_memory=False
)

encoder = joblib.load(
    MODELS_DIR
    / f"{EXPERIMENT}_label_encoder.joblib"
)

y_train = encoder.transform(
    train_df["attack_class"]
)

y_val = encoder.transform(
    val_df["attack_class"]
)

feature_columns = [
    col
    for col in train_df.columns
    if col != "attack_class"
]

X_train = (
    train_df[feature_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

X_val = (
    val_df[feature_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


# ============================================================
# 3. IDENTIFY BAROMETER AGGREGATES
# ============================================================

barometer_features = [
    col
    for col in feature_columns
    if col.startswith("barometer__")
]

print("=" * 100)
print("BAROMETER AGGREGATED FEATURES")
print("=" * 100)

for col in barometer_features:
    print("✅", col)


# ============================================================
# 4. BAROMETER DISTRIBUTIONS BY CLASS
# ============================================================

print("\n" + "=" * 100)
print("BAROMETER DISTRIBUTION — TRAIN")
print("=" * 100)

distribution_rows = []

for split_name, df_part in [
    ("train", train_df),
    ("validation", val_df),
]:

    for class_name in sorted(
        df_part["attack_class"].unique()
    ):

        subset = df_part[
            df_part["attack_class"]
            == class_name
        ]

        for feature in barometer_features:

            x = pd.to_numeric(
                subset[feature],
                errors="coerce"
            ).dropna()

            if len(x) == 0:
                continue

            distribution_rows.append({
                "split": split_name,
                "attack_class": class_name,
                "feature": feature,
                "count": len(x),
                "mean": x.mean(),
                "std": x.std(ddof=0),
                "min": x.min(),
                "q25": x.quantile(0.25),
                "median": x.median(),
                "q75": x.quantile(0.75),
                "max": x.max(),
            })


distribution_df = pd.DataFrame(
    distribution_rows
)

main_barometer_stats = distribution_df[
    distribution_df["feature"].isin([
        "barometer__mean",
        "barometer__median",
        "barometer__min",
        "barometer__max",
        "barometer__std",
    ])
]

print(
    main_barometer_stats.to_string(
        index=False
    )
)


# ============================================================
# 5. RANGE OVERLAP AUDIT
#
# If class ranges barely overlap, barometer may identify
# recording/session conditions directly.
# ============================================================

print("\n" + "=" * 100)
print("BAROMETER RANGE OVERLAP — VALIDATION")
print("=" * 100)

for feature in [
    "barometer__mean",
    "barometer__median",
]:

    if feature not in val_df.columns:
        continue

    print(f"\nFEATURE: {feature}")

    ranges = {}

    for class_name in sorted(
        val_df["attack_class"].unique()
    ):

        x = pd.to_numeric(
            val_df.loc[
                val_df["attack_class"]
                == class_name,
                feature
            ],
            errors="coerce"
        ).dropna()

        ranges[class_name] = (
            float(x.min()),
            float(x.max())
        )

        print(
            f"{class_name:10s} "
            f"min={x.min():.6f} "
            f"max={x.max():.6f}"
        )

    classes = list(
        ranges.keys()
    )

    for i in range(len(classes)):
        for j in range(
            i + 1,
            len(classes)
        ):

            a = classes[i]
            b = classes[j]

            low = max(
                ranges[a][0],
                ranges[b][0]
            )

            high = min(
                ranges[a][1],
                ranges[b][1]
            )

            overlap = max(
                0.0,
                high - low
            )

            print(
                f"Overlap {a} vs {b}: "
                f"{overlap:.6f}"
            )


# ============================================================
# 6. FEATURE GROUPS
# ============================================================

LEVEL_SUFFIXES = {
    "__mean",
    "__median",
    "__min",
    "__max",
}

DYNAMIC_SUFFIXES = {
    "__std",
    "__nunique",
    "__missing_ratio",
}


def has_suffix(
    feature,
    suffix_set
):
    return any(
        feature.endswith(suffix)
        for suffix in suffix_set
    )


all_features = feature_columns

without_barometer = [
    f
    for f in all_features
    if not f.startswith(
        "barometer__"
    )
]

barometer_dynamic_only = [
    f
    for f in all_features
    if (
        not f.startswith(
            "barometer__"
        )
        or has_suffix(
            f,
            DYNAMIC_SUFFIXES
        )
    )
]

all_dynamic_only = [
    f
    for f in all_features
    if has_suffix(
        f,
        DYNAMIC_SUFFIXES
    )
]


# ============================================================
# 7. MODEL HELPER
# ============================================================

weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)


def run_xgb(
    name,
    selected_features
):

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            XGBClassifier(
                objective="multi:softprob",
                num_class=len(
                    encoder.classes_
                ),
                n_estimators=400,
                learning_rate=0.05,
                max_depth=5,
                min_child_weight=1,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.0,
                reg_lambda=1.0,
                tree_method="hist",
                eval_metric="mlogloss",
                n_jobs=-1,
                random_state=42,
            )
        ),
    ])

    model.fit(
        X_train[
            selected_features
        ],
        y_train,
        model__sample_weight=weights
    )

    pred = model.predict(
        X_val[
            selected_features
        ]
    )

    proba = model.predict_proba(
        X_val[
            selected_features
        ]
    )

    return {
        "configuration": name,
        "features": len(
            selected_features
        ),
        "accuracy":
            accuracy_score(
                y_val,
                pred
            ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_val,
                pred
            ),
        "macro_f1":
            f1_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            ),
        "log_loss":
            log_loss(
                y_val,
                proba
            ),
    }


# ============================================================
# 8. CONFOUND-RESISTANT SENSITIVITY TESTS
# ============================================================

configs = {
    "all_features":
        all_features,

    "remove_all_barometer":
        without_barometer,

    "remove_barometer_absolute_level":
        barometer_dynamic_only,

    "all_sources_dynamic_only":
        all_dynamic_only,
}


results = []

print("\n" + "=" * 100)
print("PHYSICAL CONFOUNDING SENSITIVITY")
print("=" * 100)

for name, features in configs.items():

    result = run_xgb(
        name,
        features
    )

    results.append(
        result
    )

    print(
        f"\n{name}"
    )

    print(
        "Features          :",
        result["features"]
    )

    print(
        "Accuracy          :",
        f"{result['accuracy']:.6f}"
    )

    print(
        "Balanced Accuracy :",
        f"{result['balanced_accuracy']:.6f}"
    )

    print(
        "Macro F1          :",
        f"{result['macro_f1']:.6f}"
    )

    print(
        "Log Loss          :",
        f"{result['log_loss']:.6f}"
    )


# ============================================================
# 9. SAVE
# ============================================================

results_df = pd.DataFrame(
    results
)

results_file = (
    TABLES_DIR
    / "physical_barometer_confounding_sensitivity.csv"
)

distribution_file = (
    TABLES_DIR
    / "physical_barometer_class_distributions.csv"
)

results_df.to_csv(
    results_file,
    index=False
)

distribution_df.to_csv(
    distribution_file,
    index=False
)


# ============================================================
# 10. FINAL INTERPRETATION FLAG
# ============================================================

print("\n" + "=" * 100)
print("INTERPRETATION FLAG")
print("=" * 100)

base_f1 = float(
    results_df.loc[
        results_df["configuration"]
        == "all_features",
        "macro_f1"
    ].iloc[0]
)

no_level_f1 = float(
    results_df.loc[
        results_df["configuration"]
        == "remove_barometer_absolute_level",
        "macro_f1"
    ].iloc[0]
)

no_barometer_f1 = float(
    results_df.loc[
        results_df["configuration"]
        == "remove_all_barometer",
        "macro_f1"
    ].iloc[0]
)

print(
    "Full model Macro F1:",
    f"{base_f1:.6f}"
)

print(
    "No absolute barometer level:",
    f"{no_level_f1:.6f}"
)

print(
    "No barometer at all:",
    f"{no_barometer_f1:.6f}"
)

if (
    base_f1 >= 0.95
    and no_level_f1 < 0.70
):

    print(
        "\n⚠️ STRONG EVIDENCE OF "
        "BAROMETER-BASELINE CONFOUNDING."
    )

    print(
        "Do NOT report the 1.00 result "
        "as general UAV attack detection."
    )

else:

    print(
        "\n✅ Physical performance remains "
        "substantial after removing absolute "
        "barometer level."
    )


print(
    "\n🔒 LOCKED TEST WAS NOT LOADED."
)

print(
    "✅ Saved:",
    results_file
)

print(
    "✅ Saved:",
    distribution_file
)

In [ ]:
# ============================================================
# UAV PROJECT
# CYBER PRIMARY MODEL — SOURCE-FAMILY FEATURE REDUCTION
#
# DEVELOPMENT ONLY:
# - TRAIN used for fitting
# - VALIDATION used for model selection
# - LOCKED TEST IS NEVER LOADED
#
# Selection rule:
# Smallest source-family subset achieving
# >= 99% of full-model validation Macro F1.
# ============================================================

from pathlib import Path
import json
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    confusion_matrix,
    classification_report,
)

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
    / "development"
)

CONFIG_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


EXPERIMENT = "cyber_schema_A_3class"


# ============================================================
# 2. LOAD TRAIN + VALIDATION ONLY
# ============================================================

train_df = pd.read_csv(
    SPLITS_DIR
    / f"{EXPERIMENT}_train.csv",
    low_memory=False
)

val_df = pd.read_csv(
    SPLITS_DIR
    / f"{EXPERIMENT}_validation.csv",
    low_memory=False
)

print("✅ Train rows     :", len(train_df))
print("✅ Validation rows:", len(val_df))


# ============================================================
# 3. LOAD LABEL ENCODER
# ============================================================

encoder = joblib.load(
    MODELS_DIR
    / f"{EXPERIMENT}_label_encoder.joblib"
)

y_train = encoder.transform(
    train_df["attack_class"]
)

y_val = encoder.transform(
    val_df["attack_class"]
)


# ============================================================
# 4. PREDICTORS
# ============================================================

all_features = [
    col
    for col in train_df.columns
    if col != "attack_class"
]

X_train_all = (
    train_df[all_features]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

X_val_all = (
    val_df[all_features]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


# ============================================================
# 5. LOAD EXISTING SOURCE-GAIN RANKING
# ============================================================

gain_file = (
    TABLES_DIR
    / "development_xgboost_source_gain.csv"
)

gain_df = pd.read_csv(
    gain_file
)

gain_df = (
    gain_df[
        gain_df["experiment"]
        == EXPERIMENT
    ]
    .sort_values(
        "gain_fraction",
        ascending=False
    )
    .reset_index(drop=True)
)

if gain_df.empty:
    raise RuntimeError(
        "Cyber source-gain ranking not found."
    )

ranked_sources = (
    gain_df[
        "source_feature"
    ]
    .tolist()
)

print(
    "\n"
    + "=" * 100
)

print(
    "SOURCE-FAMILY RANKING"
)

print(
    "=" * 100
)

print(
    gain_df[
        [
            "source_feature",
            "gain_fraction"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 6. MAP AGGREGATED FEATURE -> RAW SOURCE FAMILY
# ============================================================

def source_name(
    aggregated_feature
):
    return aggregated_feature.split(
        "__",
        1
    )[0]


source_to_features = {}

for feature in all_features:

    source = source_name(
        feature
    )

    source_to_features.setdefault(
        source,
        []
    ).append(
        feature
    )


# ============================================================
# 7. CALIBRATION METRICS
# ============================================================

def multiclass_brier(
    y_true,
    probabilities,
    n_classes
):

    onehot = np.eye(
        n_classes
    )[y_true]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - onehot
                ) ** 2,
                axis=1
            )
        )
    )


def ece_equal_width(
    y_true,
    probabilities,
    n_bins=15
):

    confidence = (
        probabilities.max(
            axis=1
        )
    )

    prediction = (
        probabilities.argmax(
            axis=1
        )
    )

    correct = (
        prediction
        == y_true
    ).astype(float)

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ece = 0.0

    for i in range(
        n_bins
    ):

        if i == n_bins - 1:

            mask = (
                (confidence >= edges[i])
                &
                (confidence <= edges[i + 1])
            )

        else:

            mask = (
                (confidence >= edges[i])
                &
                (confidence < edges[i + 1])
            )

        if not mask.any():
            continue

        ece += (
            mask.mean()
            * abs(
                correct[mask].mean()
                -
                confidence[mask].mean()
            )
        )

    return float(ece)


# ============================================================
# 8. FIXED XGBOOST CONFIGURATION
# ============================================================

def make_xgb():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            XGBClassifier(
                objective=
                    "multi:softprob",

                num_class=
                    len(
                        encoder.classes_
                    ),

                n_estimators=400,
                learning_rate=0.05,
                max_depth=5,
                min_child_weight=1,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.0,
                reg_lambda=1.0,
                tree_method="hist",
                eval_metric="mlogloss",
                n_jobs=-1,
                random_state=42,
            )
        ),
    ])


balanced_weights = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )
)


# ============================================================
# 9. PREDEFINED SUBSETS
# ============================================================

candidate_k = [
    1,
    2,
    3,
    4,
    5,
    6,
    8,
    10,
    12,
    len(ranked_sources),
]

candidate_k = sorted(
    set(
        min(
            k,
            len(ranked_sources)
        )
        for k in candidate_k
    )
)


# ============================================================
# 10. TRAIN EACH SUBSET
# ============================================================

results = []
fitted_models = {}
subset_features = {}


print(
    "\n"
    + "=" * 115
)

print(
    "SOURCE-FAMILY REDUCTION EXPERIMENT"
)

print(
    "=" * 115
)


for k in candidate_k:

    selected_sources = (
        ranked_sources[:k]
    )

    selected_features = [
        feature
        for feature in all_features
        if source_name(feature)
        in selected_sources
    ]

    subset_features[k] = (
        selected_features
    )

    model = make_xgb()

    model.fit(
        X_train_all[
            selected_features
        ],
        y_train,
        model__sample_weight=
            balanced_weights
    )

    pred = model.predict(
        X_val_all[
            selected_features
        ]
    )

    proba = model.predict_proba(
        X_val_all[
            selected_features
        ]
    )

    row = {
        "source_families":
            k,

        "aggregated_features":
            len(
                selected_features
            ),

        "selected_sources":
            ",".join(
                selected_sources
            ),

        "accuracy":
            accuracy_score(
                y_val,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_val,
                pred
            ),

        "macro_precision":
            precision_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                y_val,
                pred,
                average="weighted",
                zero_division=0
            ),

        "log_loss":
            log_loss(
                y_val,
                proba,
                labels=np.arange(
                    len(
                        encoder.classes_
                    )
                )
            ),

        "brier":
            multiclass_brier(
                y_val,
                proba,
                len(
                    encoder.classes_
                )
            ),

        "ece_15":
            ece_equal_width(
                y_val,
                proba,
                n_bins=15
            ),
    }

    results.append(
        row
    )

    fitted_models[k] = (
        model
    )

    print(
        f"\nTop-{k:02d} source families"
    )

    print(
        f"Aggregated features : "
        f"{len(selected_features)}"
    )

    print(
        f"Macro F1            : "
        f"{row['macro_f1']:.6f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{row['balanced_accuracy']:.6f}"
    )

    print(
        f"Log Loss            : "
        f"{row['log_loss']:.6f}"
    )

    print(
        f"ECE-15              : "
        f"{row['ece_15']:.6f}"
    )


# ============================================================
# 11. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)

full_k = max(
    candidate_k
)

full_row = (
    results_df[
        results_df[
            "source_families"
        ] == full_k
    ]
    .iloc[0]
)

full_macro_f1 = float(
    full_row[
        "macro_f1"
    ]
)

retention_target = (
    0.99
    * full_macro_f1
)

results_df[
    "macro_f1_retention"
] = (
    results_df[
        "macro_f1"
    ]
    / full_macro_f1
)


# ============================================================
# 12. PREDEFINED SELECTION RULE
#
# Smallest subset retaining >=99%
# of FULL validation Macro F1.
# ============================================================

eligible = (
    results_df[
        results_df[
            "macro_f1"
        ]
        >= retention_target
    ]
    .sort_values(
        "source_families"
    )
)

if eligible.empty:

    selected_row = (
        results_df[
            results_df[
                "source_families"
            ] == full_k
        ]
        .iloc[0]
    )

else:

    selected_row = (
        eligible.iloc[0]
    )


selected_k = int(
    selected_row[
        "source_families"
    ]
)

selected_sources = (
    ranked_sources[
        :selected_k
    ]
)

selected_features = (
    subset_features[
        selected_k
    ]
)


# ============================================================
# 13. FINAL SELECTED MODEL METRICS
# ============================================================

selected_model = (
    fitted_models[
        selected_k
    ]
)

selected_pred = (
    selected_model.predict(
        X_val_all[
            selected_features
        ]
    )
)

selected_proba = (
    selected_model.predict_proba(
        X_val_all[
            selected_features
        ]
    )
)


# ============================================================
# 14. PER-CLASS VALIDATION METRICS
# ============================================================

report = classification_report(
    y_val,
    selected_pred,
    labels=np.arange(
        len(
            encoder.classes_
        )
    ),
    target_names=
        encoder.classes_,
    output_dict=True,
    zero_division=0
)

class_rows = []

for class_name in (
    encoder.classes_
):

    info = report[
        class_name
    ]

    class_rows.append({
        "class":
            class_name,

        "precision":
            info["precision"],

        "recall":
            info["recall"],

        "f1":
            info["f1-score"],

        "support":
            int(
                info["support"]
            ),
    })

class_df = pd.DataFrame(
    class_rows
)


# ============================================================
# 15. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_val,
    selected_pred,
    labels=np.arange(
        len(
            encoder.classes_
        )
    )
)

cm_df = pd.DataFrame(
    cm,
    index=encoder.classes_,
    columns=encoder.classes_
)


# ============================================================
# 16. SAVE TABLES
# ============================================================

results_file = (
    TABLES_DIR
    / "cyber_source_family_reduction.csv"
)

class_file = (
    TABLES_DIR
    / "cyber_selected_validation_class_metrics.csv"
)

cm_file = (
    TABLES_DIR
    / "cyber_selected_validation_confusion_matrix.csv"
)

results_df.to_csv(
    results_file,
    index=False
)

class_df.to_csv(
    class_file,
    index=False
)

cm_df.to_csv(
    cm_file
)


# ============================================================
# 17. SAVE SELECTED FEATURE CONFIG
# ============================================================

selection_config = {
    "experiment":
        EXPERIMENT,

    "primary_metric":
        "validation_macro_f1",

    "selection_rule":
        (
            "smallest predefined source-family "
            "subset retaining at least 99% of "
            "full-model validation Macro F1"
        ),

    "full_source_family_count":
        full_k,

    "full_macro_f1":
        full_macro_f1,

    "retention_threshold":
        retention_target,

    "selected_source_family_count":
        selected_k,

    "selected_sources":
        selected_sources,

    "selected_aggregated_feature_count":
        len(
            selected_features
        ),

    "selected_features":
        selected_features,
}

config_file = (
    CONFIG_DIR
    / "cyber_selected_feature_configuration.json"
)

with config_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        selection_config,
        f,
        indent=2
    )


# ============================================================
# 18. SAVE SELECTED DEVELOPMENT MODEL
# ============================================================

selected_model_file = (
    MODELS_DIR
    / "cyber_schema_A_selected_xgboost.joblib"
)

joblib.dump(
    selected_model,
    selected_model_file
)


# ============================================================
# 19. PRINT COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "FEATURE REDUCTION COMPARISON"
)

print(
    "=" * 120
)

print(
    results_df[
        [
            "source_families",
            "aggregated_features",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "macro_f1_retention",
            "log_loss",
            "brier",
            "ece_15",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 20. SELECTED CONFIGURATION
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "SELECTED CYBER CONFIGURATION"
)

print(
    "=" * 120
)

print(
    "Full Macro F1           :",
    f"{full_macro_f1:.6f}"
)

print(
    "99% retention threshold :",
    f"{retention_target:.6f}"
)

print(
    "Selected source families:",
    selected_k
)

print(
    "Aggregated features     :",
    len(
        selected_features
    )
)

print(
    "\nSelected sources:"
)

for i, source in enumerate(
    selected_sources,
    start=1
):

    print(
        f"{i:02d}. {source}"
    )

print(
    "\nSelected validation Macro F1:",
    f"{selected_row['macro_f1']:.6f}"
)

print(
    "Selected Balanced Accuracy  :",
    f"{selected_row['balanced_accuracy']:.6f}"
)

print(
    "Selected Log Loss           :",
    f"{selected_row['log_loss']:.6f}"
)

print(
    "Selected ECE-15             :",
    f"{selected_row['ece_15']:.6f}"
)


# ============================================================
# 21. PER-CLASS
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "SELECTED MODEL — VALIDATION PER-CLASS METRICS"
)

print(
    "=" * 120
)

print(
    class_df.to_string(
        index=False
    )
)


# ============================================================
# 22. CONFUSION MATRIX
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "SELECTED MODEL — VALIDATION CONFUSION MATRIX"
)

print(
    "=" * 120
)

print(
    cm_df.to_string()
)


# ============================================================
# 23. FINAL
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "CYBER FEATURE-SELECTION STAGE COMPLETE"
)

print(
    "=" * 120
)

print(
    "✅ Comparison:",
    results_file
)

print(
    "✅ Selected config:",
    config_file
)

print(
    "✅ Selected model:",
    selected_model_file
)

print(
    "✅ Per-class metrics:",
    class_file
)

print(
    "✅ Confusion matrix:",
    cm_file
)

print(
    "\n🔒 LOCKED TEST WAS NOT LOADED."
)

print(
    "Next step = OOF probability calibration "
    "on TRAIN only, then one validation check."
)

In [ ]:
# ============================================================
# UAV PROJECT
# CORRECT FEATURE-SELECTION BASELINE
# + HIERARCHICAL CYBER MODEL
#
# Stage 1:
#   Benign vs Attack
#
# Stage 2:
#   DoS vs Replay
#
# DEVELOPMENT ONLY:
# - TRAIN for fitting
# - VALIDATION for evaluation
# - LOCKED TEST IS NEVER LOADED
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    confusion_matrix,
    classification_report,
)

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
    / "development"
)

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


EXPERIMENT = "cyber_schema_A_3class"


# ============================================================
# 2. LOAD TRAIN + VALIDATION ONLY
# ============================================================

train_df = pd.read_csv(
    SPLITS_DIR
    / f"{EXPERIMENT}_train.csv",
    low_memory=False
)

val_df = pd.read_csv(
    SPLITS_DIR
    / f"{EXPERIMENT}_validation.csv",
    low_memory=False
)


all_features = [
    c
    for c in train_df.columns
    if c != "attack_class"
]


# ============================================================
# 3. CORRECT FEATURE-SELECTION BASELINE
# ============================================================

reduction_file = (
    TABLES_DIR
    / "cyber_source_family_reduction.csv"
)

baseline_file = (
    TABLES_DIR
    / "development_validation_model_results.csv"
)

gain_file = (
    TABLES_DIR
    / "development_xgboost_source_gain.csv"
)


reduction_df = pd.read_csv(
    reduction_file
)

baseline_df = pd.read_csv(
    baseline_file
)

gain_df = pd.read_csv(
    gain_file
)

gain_df = (
    gain_df[
        gain_df["experiment"]
        == EXPERIMENT
    ]
    .sort_values(
        "gain_fraction",
        ascending=False
    )
    .reset_index(drop=True)
)


full_baseline = (
    baseline_df[
        (
            baseline_df["experiment"]
            == EXPERIMENT
        )
        &
        (
            baseline_df["model"]
            == "XGBoost"
        )
    ]
    .iloc[0]
)

true_full_macro_f1 = float(
    full_baseline["macro_f1"]
)

true_threshold = (
    0.99
    * true_full_macro_f1
)


def source_name(feature):
    return feature.split(
        "__",
        1
    )[0]


all_source_families = sorted(
    {
        source_name(f)
        for f in all_features
    }
)

gain_sources = (
    gain_df["source_feature"]
    .tolist()
)

zero_gain_sources = [
    source
    for source in all_source_families
    if source not in gain_sources
]


reduction_df[
    "macro_f1_retention_corrected"
] = (
    reduction_df["macro_f1"]
    / true_full_macro_f1
)

eligible = (
    reduction_df[
        reduction_df["macro_f1"]
        >= true_threshold
    ]
    .sort_values(
        "source_families"
    )
)

if eligible.empty:
    raise RuntimeError(
        "No reduced subset satisfies "
        "the corrected 99% criterion."
    )

selected_row = (
    eligible.iloc[0]
)

selected_k = int(
    selected_row[
        "source_families"
    ]
)

selected_sources = (
    gain_sources[:selected_k]
)

selected_features = [
    f
    for f in all_features
    if source_name(f)
    in selected_sources
]


print("=" * 110)
print("CORRECTED FEATURE-SELECTION AUDIT")
print("=" * 110)

print(
    "Total source families        :",
    len(all_source_families)
)

print(
    "Non-zero-gain source families:",
    len(gain_sources)
)

print(
    "Zero-gain source families    :",
    len(zero_gain_sources)
)

print(
    "\nZero-gain sources:"
)

for source in zero_gain_sources:
    print("  -", source)

print(
    "\nTrue full model Macro F1     :",
    f"{true_full_macro_f1:.6f}"
)

print(
    "True 99% retention threshold:",
    f"{true_threshold:.6f}"
)

print(
    "Selected source families     :",
    selected_k
)

print(
    "Selected Macro F1            :",
    f"{selected_row['macro_f1']:.6f}"
)

print(
    "Corrected retention          :",
    f"{selected_row['macro_f1'] / true_full_macro_f1:.6f}"
)

print(
    "Aggregated selected features :",
    len(selected_features)
)

print(
    "\nSelected sources:"
)

for source in selected_sources:
    print("  -", source)


# ============================================================
# 4. SAVE CORRECTED FEATURE CONFIG
# ============================================================

corrected_config = {
    "experiment":
        EXPERIMENT,

    "full_source_family_count":
        len(all_source_families),

    "nonzero_gain_source_family_count":
        len(gain_sources),

    "zero_gain_sources":
        zero_gain_sources,

    "true_full_aggregated_feature_count":
        len(all_features),

    "true_full_validation_macro_f1":
        true_full_macro_f1,

    "selection_rule":
        (
            "smallest predefined source-family "
            "subset retaining at least 99% of "
            "the true 161-feature full-model "
            "validation Macro F1"
        ),

    "retention_threshold":
        true_threshold,

    "selected_source_family_count":
        selected_k,

    "selected_sources":
        selected_sources,

    "selected_aggregated_feature_count":
        len(selected_features),

    "selected_features":
        selected_features,

    "selected_validation_macro_f1":
        float(
            selected_row["macro_f1"]
        ),

    "corrected_macro_f1_retention":
        float(
            selected_row["macro_f1"]
            / true_full_macro_f1
        ),
}


corrected_config_file = (
    PROCESSED_DIR
    / "cyber_selected_feature_configuration_corrected.json"
)

with corrected_config_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        corrected_config,
        f,
        indent=2
    )


corrected_reduction_file = (
    TABLES_DIR
    / "cyber_source_family_reduction_corrected.csv"
)

reduction_df.to_csv(
    corrected_reduction_file,
    index=False
)


# ============================================================
# 5. NUMERIC MATRICES
# ============================================================

X_train = (
    train_df[
        selected_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

X_val = (
    val_df[
        selected_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


# ============================================================
# 6. MODEL FACTORY
# ============================================================

def binary_xgb():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                n_estimators=400,
                learning_rate=0.05,
                max_depth=5,
                min_child_weight=1,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.0,
                reg_lambda=1.0,
                tree_method="hist",
                eval_metric="logloss",
                n_jobs=-1,
                random_state=42,
            )
        ),
    ])


# ============================================================
# 7. ECE
# ============================================================

def binary_ece(
    y_true,
    p_attack,
    n_bins=15
):

    pred = (
        p_attack >= 0.5
    ).astype(int)

    confidence = np.where(
        pred == 1,
        p_attack,
        1.0 - p_attack
    )

    correct = (
        pred == y_true
    ).astype(float)

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:

            mask = (
                (confidence >= edges[i])
                &
                (confidence <= edges[i + 1])
            )

        else:

            mask = (
                (confidence >= edges[i])
                &
                (confidence < edges[i + 1])
            )

        if not mask.any():
            continue

        ece += (
            mask.mean()
            *
            abs(
                correct[mask].mean()
                -
                confidence[mask].mean()
            )
        )

    return float(ece)


# ============================================================
# 8. STAGE 1 — BENIGN VS ATTACK
# ============================================================

y_train_detection = (
    train_df["attack_class"]
    != "Benign"
).astype(int).to_numpy()

y_val_detection = (
    val_df["attack_class"]
    != "Benign"
).astype(int).to_numpy()


detection_weights = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_train_detection
    )
)

stage1 = binary_xgb()

stage1.fit(
    X_train,
    y_train_detection,
    model__sample_weight=
        detection_weights
)

p_attack = (
    stage1.predict_proba(
        X_val
    )[:, 1]
)

pred_detection = (
    p_attack >= 0.5
).astype(int)


stage1_metrics = {
    "stage":
        "Stage 1 - Attack Detection",

    "accuracy":
        accuracy_score(
            y_val_detection,
            pred_detection
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_val_detection,
            pred_detection
        ),

    "precision_attack":
        precision_score(
            y_val_detection,
            pred_detection,
            zero_division=0
        ),

    "recall_attack":
        recall_score(
            y_val_detection,
            pred_detection,
            zero_division=0
        ),

    "f1_attack":
        f1_score(
            y_val_detection,
            pred_detection,
            zero_division=0
        ),

    "roc_auc":
        roc_auc_score(
            y_val_detection,
            p_attack
        ),

    "pr_auc":
        average_precision_score(
            y_val_detection,
            p_attack
        ),

    "log_loss":
        log_loss(
            y_val_detection,
            np.column_stack(
                [
                    1 - p_attack,
                    p_attack
                ]
            )
        ),

    "brier":
        float(
            np.mean(
                (
                    p_attack
                    - y_val_detection
                ) ** 2
            )
        ),

    "ece_15":
        binary_ece(
            y_val_detection,
            p_attack,
            n_bins=15
        ),
}


stage1_cm = pd.DataFrame(
    confusion_matrix(
        y_val_detection,
        pred_detection,
        labels=[0, 1]
    ),
    index=[
        "True Benign",
        "True Attack"
    ],
    columns=[
        "Pred Benign",
        "Pred Attack"
    ]
)


# ============================================================
# 9. STAGE 2 — DoS VS REPLAY
# ============================================================

attack_train_mask = (
    train_df["attack_class"]
    .isin(
        [
            "DoS",
            "Replay"
        ]
    )
)

attack_val_mask = (
    val_df["attack_class"]
    .isin(
        [
            "DoS",
            "Replay"
        ]
    )
)


X_train_attack = (
    X_train.loc[
        attack_train_mask
    ]
)

X_val_attack = (
    X_val.loc[
        attack_val_mask
    ]
)


# DoS = 0
# Replay = 1

y_train_attr = (
    train_df.loc[
        attack_train_mask,
        "attack_class"
    ]
    == "Replay"
).astype(int).to_numpy()

y_val_attr = (
    val_df.loc[
        attack_val_mask,
        "attack_class"
    ]
    == "Replay"
).astype(int).to_numpy()


attr_weights = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_train_attr
    )
)

stage2 = binary_xgb()

stage2.fit(
    X_train_attack,
    y_train_attr,
    model__sample_weight=
        attr_weights
)

p_replay = (
    stage2.predict_proba(
        X_val_attack
    )[:, 1]
)

pred_attr = (
    p_replay >= 0.5
).astype(int)


stage2_metrics = {
    "stage":
        "Stage 2 - DoS vs Replay",

    "accuracy":
        accuracy_score(
            y_val_attr,
            pred_attr
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_val_attr,
            pred_attr
        ),

    "macro_f1":
        f1_score(
            y_val_attr,
            pred_attr,
            average="macro",
            zero_division=0
        ),

    "roc_auc":
        roc_auc_score(
            y_val_attr,
            p_replay
        ),

    "log_loss":
        log_loss(
            y_val_attr,
            np.column_stack(
                [
                    1 - p_replay,
                    p_replay
                ]
            )
        ),
}


stage2_cm = pd.DataFrame(
    confusion_matrix(
        y_val_attr,
        pred_attr,
        labels=[0, 1]
    ),
    index=[
        "True DoS",
        "True Replay"
    ],
    columns=[
        "Pred DoS",
        "Pred Replay"
    ]
)


# ============================================================
# 10. END-TO-END HIERARCHICAL PREDICTIONS
# ============================================================

hierarchical_pred = np.array(
    ["Benign"] * len(val_df),
    dtype=object
)

predicted_attack_indices = np.where(
    pred_detection == 1
)[0]


if len(
    predicted_attack_indices
) > 0:

    X_stage2_live = (
        X_val.iloc[
            predicted_attack_indices
        ]
    )

    live_p_replay = (
        stage2.predict_proba(
            X_stage2_live
        )[:, 1]
    )

    live_labels = np.where(
        live_p_replay >= 0.5,
        "Replay",
        "DoS"
    )

    hierarchical_pred[
        predicted_attack_indices
    ] = live_labels


y_val_three = (
    val_df["attack_class"]
    .to_numpy()
)


hierarchical_metrics = {
    "accuracy":
        accuracy_score(
            y_val_three,
            hierarchical_pred
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_val_three,
            hierarchical_pred
        ),

    "macro_f1":
        f1_score(
            y_val_three,
            hierarchical_pred,
            average="macro",
            zero_division=0
        ),

    "weighted_f1":
        f1_score(
            y_val_three,
            hierarchical_pred,
            average="weighted",
            zero_division=0
        ),
}


class_order = [
    "Benign",
    "DoS",
    "Replay"
]

hierarchical_cm = pd.DataFrame(
    confusion_matrix(
        y_val_three,
        hierarchical_pred,
        labels=class_order
    ),
    index=class_order,
    columns=class_order
)


# ============================================================
# 11. SAVE MODELS
# ============================================================

stage1_file = (
    MODELS_DIR
    / "cyber_stage1_attack_detector.joblib"
)

stage2_file = (
    MODELS_DIR
    / "cyber_stage2_attack_attribution.joblib"
)

joblib.dump(
    stage1,
    stage1_file
)

joblib.dump(
    stage2,
    stage2_file
)


# ============================================================
# 12. SAVE METRICS
# ============================================================

stage1_metrics_df = pd.DataFrame(
    [stage1_metrics]
)

stage2_metrics_df = pd.DataFrame(
    [stage2_metrics]
)

hierarchical_metrics_df = pd.DataFrame(
    [hierarchical_metrics]
)


stage1_metrics_file = (
    TABLES_DIR
    / "cyber_stage1_validation_metrics.csv"
)

stage2_metrics_file = (
    TABLES_DIR
    / "cyber_stage2_validation_metrics.csv"
)

hierarchical_metrics_file = (
    TABLES_DIR
    / "cyber_hierarchical_validation_metrics.csv"
)

stage1_cm_file = (
    TABLES_DIR
    / "cyber_stage1_validation_confusion_matrix.csv"
)

stage2_cm_file = (
    TABLES_DIR
    / "cyber_stage2_validation_confusion_matrix.csv"
)

hierarchical_cm_file = (
    TABLES_DIR
    / "cyber_hierarchical_validation_confusion_matrix.csv"
)


stage1_metrics_df.to_csv(
    stage1_metrics_file,
    index=False
)

stage2_metrics_df.to_csv(
    stage2_metrics_file,
    index=False
)

hierarchical_metrics_df.to_csv(
    hierarchical_metrics_file,
    index=False
)

stage1_cm.to_csv(
    stage1_cm_file
)

stage2_cm.to_csv(
    stage2_cm_file
)

hierarchical_cm.to_csv(
    hierarchical_cm_file
)


# ============================================================
# 13. PRINT RESULTS
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "STAGE 1 — BENIGN VS ATTACK"
)

print(
    "=" * 110
)

for key, value in stage1_metrics.items():

    if key == "stage":
        continue

    print(
        f"{key:20s}: "
        f"{value:.6f}"
    )

print(
    "\nConfusion matrix:"
)

print(
    stage1_cm.to_string()
)


print(
    "\n"
    + "=" * 110
)

print(
    "STAGE 2 — DoS VS REPLAY"
)

print(
    "=" * 110
)

for key, value in stage2_metrics.items():

    if key == "stage":
        continue

    print(
        f"{key:20s}: "
        f"{value:.6f}"
    )

print(
    "\nConfusion matrix:"
)

print(
    stage2_cm.to_string()
)


print(
    "\n"
    + "=" * 110
)

print(
    "END-TO-END HIERARCHICAL 3-CLASS RESULT"
)

print(
    "=" * 110
)

for key, value in (
    hierarchical_metrics.items()
):

    print(
        f"{key:20s}: "
        f"{value:.6f}"
    )

print(
    "\nConfusion matrix:"
)

print(
    hierarchical_cm.to_string()
)


# ============================================================
# 14. FINAL
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "HIERARCHICAL DEVELOPMENT STAGE COMPLETE"
)

print(
    "=" * 110
)

print(
    "✅ Corrected feature config:",
    corrected_config_file
)

print(
    "✅ Stage-1 model:",
    stage1_file
)

print(
    "✅ Stage-2 model:",
    stage2_file
)

print(
    "\n🔒 LOCKED TEST WAS NOT LOADED."
)

print(
    "\nNext step:"
)

print(
    "If Stage 1 remains strong, "
    "freeze it as the primary detector "
    "and perform grouped OOF probability calibration."
)

In [ ]:
# ============================================================
# UAV PROJECT
# STAGE-1 GROUPED OOF TEMPERATURE CALIBRATION
#
# PRIMARY TASK:
#   Benign vs Attack
#
# Calibration:
#   Temperature scaling fitted ONLY on grouped OOF
#   predictions from TRAIN.
#
# Validation:
#   Used once to decide whether calibrated probabilities
#   improve predictive log loss.
#
# LOCKED TEST FEATURES ARE NEVER LOADED.
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import joblib

from scipy.optimize import minimize_scalar

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
    / "development"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPERIMENT = "cyber_schema_A_3class"


# ============================================================
# 2. LOAD TRAIN + VALIDATION FEATURE TABLES
#
# No locked-test feature table is loaded.
# ============================================================

train_df = pd.read_csv(
    SPLITS_DIR
    / f"{EXPERIMENT}_train.csv",
    low_memory=False
)

val_df = pd.read_csv(
    SPLITS_DIR
    / f"{EXPERIMENT}_validation.csv",
    low_memory=False
)


# ============================================================
# 3. LOAD TRAIN GROUP METADATA
#
# The split-map contains provenance/group metadata only.
# Test FEATURE values are not present in this file.
#
# The train.csv was written from the same ordered dataframe,
# so filtered train split-map rows should align exactly.
# ============================================================

split_map = pd.read_csv(
    SPLITS_DIR
    / f"{EXPERIMENT}_window_split_map.csv",
    low_memory=False
)

train_meta = (
    split_map[
        split_map["split"] == "train"
    ]
    .reset_index(drop=True)
)

assert len(train_meta) == len(train_df), (
    "Train metadata row count does not match train feature table."
)

assert (
    train_meta["attack_class"]
    .astype(str)
    .to_numpy()
    ==
    train_df["attack_class"]
    .astype(str)
    .to_numpy()
).all(), (
    "Train metadata ordering does not match train feature table."
)

groups = (
    train_meta[
        "group_id"
    ]
    .astype(str)
    .to_numpy()
)

print("✅ Train rows   :", len(train_df))
print("✅ Train groups :", len(np.unique(groups)))
print("✅ Validation   :", len(val_df))


# ============================================================
# 4. LOAD SELECTED FEATURE CONFIGURATION
# ============================================================

config_file = (
    PROCESSED_DIR
    / "cyber_selected_feature_configuration_corrected.json"
)

with config_file.open(
    "r",
    encoding="utf-8"
) as f:

    feature_config = json.load(f)

selected_features = (
    feature_config[
        "selected_features"
    ]
)

print(
    "✅ Selected features:",
    len(selected_features)
)

print(
    "✅ Selected source families:",
    feature_config[
        "selected_sources"
    ]
)


# ============================================================
# 5. MATRICES
# ============================================================

X_train = (
    train_df[
        selected_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

X_val = (
    val_df[
        selected_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

y_train = (
    train_df["attack_class"]
    != "Benign"
).astype(int).to_numpy()

y_val = (
    val_df["attack_class"]
    != "Benign"
).astype(int).to_numpy()


# ============================================================
# 6. LOAD FINAL DEVELOPMENT STAGE-1 PIPELINE
# ============================================================

stage1_file = (
    MODELS_DIR
    / "cyber_stage1_attack_detector.joblib"
)

stage1_model = joblib.load(
    stage1_file
)


# ============================================================
# 7. GROUPED OOF CONFIGURATION
# ============================================================

N_SPLITS = 3

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)

oof_probability = np.full(
    len(train_df),
    np.nan,
    dtype=float
)

fold_records = []


print(
    "\n"
    + "=" * 110
)

print(
    "GROUPED OOF CALIBRATION PREDICTIONS"
)

print(
    "=" * 110
)


# ============================================================
# 8. CREATE GROUPED OOF PREDICTIONS
# ============================================================

for fold, (
    fit_idx,
    holdout_idx
) in enumerate(
    cv.split(
        X_train,
        y_train,
        groups=groups
    ),
    start=1
):

    train_groups_fold = set(
        groups[fit_idx]
    )

    holdout_groups_fold = set(
        groups[holdout_idx]
    )

    overlap = (
        train_groups_fold
        &
        holdout_groups_fold
    )

    assert len(overlap) == 0

    fold_model = clone(
        stage1_model
    )

    fold_weights = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train[fit_idx]
        )
    )

    fold_model.fit(
        X_train.iloc[
            fit_idx
        ],
        y_train[fit_idx],
        model__sample_weight=
            fold_weights
    )

    fold_probability = (
        fold_model.predict_proba(
            X_train.iloc[
                holdout_idx
            ]
        )[:, 1]
    )

    oof_probability[
        holdout_idx
    ] = fold_probability

    fold_pred = (
        fold_probability
        >= 0.5
    ).astype(int)

    fold_records.append({
        "fold": fold,

        "fit_rows":
            len(fit_idx),

        "holdout_rows":
            len(holdout_idx),

        "fit_groups":
            len(
                train_groups_fold
            ),

        "holdout_groups":
            len(
                holdout_groups_fold
            ),

        "group_overlap":
            len(overlap),

        "holdout_attack_rate":
            float(
                y_train[
                    holdout_idx
                ].mean()
            ),

        "holdout_accuracy":
            accuracy_score(
                y_train[
                    holdout_idx
                ],
                fold_pred
            ),

        "holdout_balanced_accuracy":
            balanced_accuracy_score(
                y_train[
                    holdout_idx
                ],
                fold_pred
            ),

        "holdout_attack_f1":
            f1_score(
                y_train[
                    holdout_idx
                ],
                fold_pred,
                zero_division=0
            ),
    })

    print(
        f"\nFold {fold}"
    )

    print(
        "  Fit rows             :",
        len(fit_idx)
    )

    print(
        "  Holdout rows         :",
        len(holdout_idx)
    )

    print(
        "  Fit groups           :",
        len(
            train_groups_fold
        )
    )

    print(
        "  Holdout groups       :",
        len(
            holdout_groups_fold
        )
    )

    print(
        "  Group overlap        :",
        len(overlap)
    )

    print(
        "  Holdout Attack F1    :",
        f"{fold_records[-1]['holdout_attack_f1']:.6f}"
    )


assert not np.isnan(
    oof_probability
).any(), (
    "Some train rows did not receive OOF probabilities."
)


# ============================================================
# 9. LOGIT / TEMPERATURE FUNCTIONS
# ============================================================

EPS = 1e-12


def probability_to_logit(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        EPS,
        1.0 - EPS
    )

    return np.log(
        p / (1.0 - p)
    )


def sigmoid(z):

    z = np.clip(
        z,
        -50.0,
        50.0
    )

    return (
        1.0
        /
        (
            1.0
            + np.exp(-z)
        )
    )


def apply_temperature(
    probability,
    temperature
):

    logits = (
        probability_to_logit(
            probability
        )
    )

    calibrated_logits = (
        logits
        / temperature
    )

    return sigmoid(
        calibrated_logits
    )


# ============================================================
# 10. FIT TEMPERATURE ON TRAIN OOF ONLY
# ============================================================

def temperature_objective(
    temperature
):

    calibrated = (
        apply_temperature(
            oof_probability,
            temperature
        )
    )

    probability_matrix = (
        np.column_stack(
            [
                1.0 - calibrated,
                calibrated
            ]
        )
    )

    return log_loss(
        y_train,
        probability_matrix,
        labels=[0, 1]
    )


optimization = minimize_scalar(
    temperature_objective,
    bounds=(
        0.05,
        10.0
    ),
    method="bounded",
    options={
        "xatol": 1e-8
    }
)

if not optimization.success:
    raise RuntimeError(
        "Temperature optimization failed."
    )

temperature = float(
    optimization.x
)


# ============================================================
# 11. TRAIN-OOF BEFORE/AFTER CALIBRATION
# ============================================================

oof_calibrated = (
    apply_temperature(
        oof_probability,
        temperature
    )
)

oof_logloss_before = (
    log_loss(
        y_train,
        np.column_stack(
            [
                1 - oof_probability,
                oof_probability
            ]
        ),
        labels=[0, 1]
    )
)

oof_logloss_after = (
    log_loss(
        y_train,
        np.column_stack(
            [
                1 - oof_calibrated,
                oof_calibrated
            ]
        ),
        labels=[0, 1]
    )
)


# ============================================================
# 12. CALIBRATION METRICS
# ============================================================

def brier_score(
    y_true,
    probability
):

    return float(
        np.mean(
            (
                probability
                - y_true
            ) ** 2
        )
    )


def equal_width_ece(
    y_true,
    probability,
    n_bins=15
):

    pred = (
        probability >= 0.5
    ).astype(int)

    confidence = np.where(
        pred == 1,
        probability,
        1.0 - probability
    )

    correct = (
        pred == y_true
    ).astype(float)

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ece = 0.0

    for i in range(
        n_bins
    ):

        if i == n_bins - 1:

            mask = (
                (confidence >= edges[i])
                &
                (confidence <= edges[i + 1])
            )

        else:

            mask = (
                (confidence >= edges[i])
                &
                (confidence < edges[i + 1])
            )

        if not mask.any():
            continue

        ece += (
            mask.mean()
            *
            abs(
                correct[mask].mean()
                -
                confidence[mask].mean()
            )
        )

    return float(ece)


def adaptive_ece(
    y_true,
    probability,
    n_bins=15
):

    pred = (
        probability >= 0.5
    ).astype(int)

    confidence = np.where(
        pred == 1,
        probability,
        1.0 - probability
    )

    correct = (
        pred == y_true
    ).astype(float)

    order = np.argsort(
        confidence
    )

    bins = np.array_split(
        order,
        n_bins
    )

    ece = 0.0

    n = len(
        confidence
    )

    for idx in bins:

        if len(idx) == 0:
            continue

        bin_accuracy = (
            correct[idx]
            .mean()
        )

        bin_confidence = (
            confidence[idx]
            .mean()
        )

        ece += (
            len(idx)
            / n
            *
            abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(ece)


def calibration_summary(
    y_true,
    probability
):

    pred = (
        probability >= 0.5
    ).astype(int)

    probability_matrix = (
        np.column_stack(
            [
                1.0 - probability,
                probability
            ]
        )
    )

    confidence = np.maximum(
        probability,
        1.0 - probability
    )

    accuracy = (
        accuracy_score(
            y_true,
            pred
        )
    )

    return {
        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred
            ),

        "attack_precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "attack_recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "attack_f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probability
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                probability
            ),

        "log_loss":
            log_loss(
                y_true,
                probability_matrix,
                labels=[0, 1]
            ),

        "brier":
            brier_score(
                y_true,
                probability
            ),

        "equal_width_ece_15":
            equal_width_ece(
                y_true,
                probability,
                n_bins=15
            ),

        "adaptive_ece_15":
            adaptive_ece(
                y_true,
                probability,
                n_bins=15
            ),

        "mean_confidence":
            float(
                confidence.mean()
            ),

        "confidence_accuracy_gap":
            float(
                confidence.mean()
                - accuracy
            ),
    }


# ============================================================
# 13. FINAL STAGE-1 VALIDATION PROBABILITIES
# ============================================================

val_probability_raw = (
    stage1_model.predict_proba(
        X_val
    )[:, 1]
)

val_probability_calibrated = (
    apply_temperature(
        val_probability_raw,
        temperature
    )
)

before = (
    calibration_summary(
        y_val,
        val_probability_raw
    )
)

after = (
    calibration_summary(
        y_val,
        val_probability_calibrated
    )
)


# ============================================================
# 14. VERIFY LABELS ARE UNCHANGED
#
# Positive temperature scaling cannot change
# the 0.5 decision boundary.
# ============================================================

raw_predictions = (
    val_probability_raw
    >= 0.5
).astype(int)

cal_predictions = (
    val_probability_calibrated
    >= 0.5
).astype(int)

prediction_changes = int(
    np.sum(
        raw_predictions
        != cal_predictions
    )
)

assert prediction_changes == 0


# ============================================================
# 15. PREDEFINED ADOPTION RULE
#
# Adopt calibrated probabilities only if validation
# log loss is strictly lower than raw log loss.
# ============================================================

adopt_calibration = bool(
    after["log_loss"]
    <
    before["log_loss"]
)


# ============================================================
# 16. SAVE ARTIFACTS
# ============================================================

fold_df = pd.DataFrame(
    fold_records
)

fold_file = (
    TABLES_DIR
    / "cyber_stage1_calibration_oof_folds.csv"
)

fold_df.to_csv(
    fold_file,
    index=False
)


comparison_rows = []

for state, metrics in [
    ("uncalibrated", before),
    ("temperature_scaled", after),
]:

    comparison_rows.append({
        "state": state,
        **metrics
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_file = (
    TABLES_DIR
    / "cyber_stage1_validation_calibration_comparison.csv"
)

comparison_df.to_csv(
    comparison_file,
    index=False
)


calibration_config = {
    "method":
        "binary temperature scaling",

    "calibration_fit_source":
        (
            "3-fold StratifiedGroupKFold "
            "OOF predictions from frozen TRAIN only"
        ),

    "group_definition":
        "section + temporal segment_id",

    "n_splits":
        N_SPLITS,

    "temperature":
        temperature,

    "train_oof_log_loss_before":
        float(
            oof_logloss_before
        ),

    "train_oof_log_loss_after":
        float(
            oof_logloss_after
        ),

    "validation_log_loss_before":
        float(
            before["log_loss"]
        ),

    "validation_log_loss_after":
        float(
            after["log_loss"]
        ),

    "prediction_changes_at_0.5":
        prediction_changes,

    "adoption_rule":
        (
            "adopt temperature scaling only "
            "if validation log loss is lower"
        ),

    "calibration_adopted":
        adopt_calibration,
}


calibration_config_file = (
    PROCESSED_DIR
    / "cyber_stage1_temperature_calibration.json"
)

with calibration_config_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        calibration_config,
        f,
        indent=2
    )


# ============================================================
# 17. PRINT RESULTS
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TRAIN OOF TEMPERATURE FIT"
)

print(
    "=" * 110
)

print(
    "Temperature:",
    f"{temperature:.8f}"
)

print(
    "OOF Log Loss before:",
    f"{oof_logloss_before:.6f}"
)

print(
    "OOF Log Loss after :",
    f"{oof_logloss_after:.6f}"
)


print(
    "\n"
    + "=" * 125
)

print(
    "VALIDATION CALIBRATION COMPARISON"
)

print(
    "=" * 125
)

print(
    comparison_df[
        [
            "state",
            "accuracy",
            "balanced_accuracy",
            "attack_precision",
            "attack_recall",
            "attack_f1",
            "roc_auc",
            "pr_auc",
            "log_loss",
            "brier",
            "equal_width_ece_15",
            "adaptive_ece_15",
            "mean_confidence",
            "confidence_accuracy_gap",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "CALIBRATION DECISION"
)

print(
    "=" * 110
)

print(
    "Prediction changes:",
    prediction_changes
)

print(
    "Raw validation Log Loss:",
    f"{before['log_loss']:.6f}"
)

print(
    "Calibrated Log Loss      :",
    f"{after['log_loss']:.6f}"
)

print(
    "Calibration adopted      :",
    adopt_calibration
)

if adopt_calibration:

    print(
        "\n✅ Temperature scaling becomes "
        "the frozen Stage-1 probability mapping."
    )

else:

    print(
        "\n✅ Retain original Stage-1 probabilities."
    )

    print(
        "Temperature scaling did not improve "
        "the predefined validation Log-Loss criterion."
    )


print(
    "\n"
    + "=" * 110
)

print(
    "CALIBRATION STAGE COMPLETE"
)

print(
    "=" * 110
)

print(
    "✅ Calibration config:",
    calibration_config_file
)

print(
    "✅ Validation comparison:",
    comparison_file
)

print(
    "✅ OOF fold audit:",
    fold_file
)

print(
    "\n🔒 LOCKED TEST FEATURES WERE NOT LOADED."
)

In [ ]:
# ============================================================
# UAV PROJECT
# VALIDATION-ONLY RISK-AWARE ROUTING POLICY
#
# Primary Stage-1 task:
#   Benign vs Attack
#
# Policy:
#   High-confidence Benign -> continue monitoring
#   High-confidence Attack -> generic protective response
#   Low confidence          -> verification / escalation
#
# Threshold-selection rule:
#   Capture >=75% of validation prediction errors
#   while minimizing the fraction routed to verification.
#
# LOCKED TEST FEATURES ARE NEVER LOADED.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
    / "development"
)

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

EXPERIMENT = "cyber_schema_A_3class"

TARGET_ERROR_CAPTURE = 0.75


# ============================================================
# 2. LOAD VALIDATION ONLY
# ============================================================

val_df = pd.read_csv(
    SPLITS_DIR
    / f"{EXPERIMENT}_validation.csv",
    low_memory=False
)

print("✅ Validation windows:", len(val_df))


# ============================================================
# 3. LOAD FROZEN FEATURE CONFIG
# ============================================================

feature_config_file = (
    PROCESSED_DIR
    / "cyber_selected_feature_configuration_corrected.json"
)

with feature_config_file.open(
    "r",
    encoding="utf-8"
) as f:
    feature_config = json.load(f)

selected_features = (
    feature_config[
        "selected_features"
    ]
)


X_val = (
    val_df[
        selected_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

y_val = (
    val_df["attack_class"]
    != "Benign"
).astype(int).to_numpy()


# ============================================================
# 4. LOAD FROZEN STAGE-1 MODEL
# ============================================================

stage1_model = joblib.load(
    MODELS_DIR
    / "cyber_stage1_attack_detector.joblib"
)


# ============================================================
# 5. LOAD ADOPTED TEMPERATURE
# ============================================================

calibration_file = (
    PROCESSED_DIR
    / "cyber_stage1_temperature_calibration.json"
)

with calibration_file.open(
    "r",
    encoding="utf-8"
) as f:
    calibration = json.load(f)

if not calibration[
    "calibration_adopted"
]:
    raise RuntimeError(
        "Expected adopted temperature calibration."
    )

temperature = float(
    calibration[
        "temperature"
    ]
)

print(
    "✅ Frozen temperature:",
    temperature
)


# ============================================================
# 6. TEMPERATURE SCALING
# ============================================================

EPS = 1e-12


def apply_temperature(
    probability,
    temperature
):

    p = np.clip(
        np.asarray(
            probability,
            dtype=float
        ),
        EPS,
        1.0 - EPS
    )

    logits = np.log(
        p / (1.0 - p)
    )

    scaled_logits = (
        logits
        / temperature
    )

    scaled_logits = np.clip(
        scaled_logits,
        -50,
        50
    )

    return (
        1.0
        /
        (
            1.0
            + np.exp(
                -scaled_logits
            )
        )
    )


# ============================================================
# 7. VALIDATION PROBABILITIES
# ============================================================

raw_p_attack = (
    stage1_model.predict_proba(
        X_val
    )[:, 1]
)

p_attack = apply_temperature(
    raw_p_attack,
    temperature
)

base_pred = (
    p_attack >= 0.5
).astype(int)


# ============================================================
# 8. UNCERTAINTY DEFINITION
#
# confidence = max(P(Benign), P(Attack))
#
# uncertainty = 1 - confidence
#
# Range:
#   0.0 -> very confident
#   0.5 -> maximally uncertain
# ============================================================

confidence = np.maximum(
    p_attack,
    1.0 - p_attack
)

uncertainty = (
    1.0
    - confidence
)

errors = (
    base_pred != y_val
)

n_errors = int(
    errors.sum()
)

print(
    "\nBase validation errors:",
    n_errors
)

print(
    "Base validation accuracy:",
    f"{accuracy_score(y_val, base_pred):.6f}"
)


# ============================================================
# 9. SEARCH ALL POSSIBLE UNCERTAINTY THRESHOLDS
#
# review if:
# uncertainty >= threshold
#
# Predefined selection:
# 1. error capture >= 75%
# 2. minimum review burden
# 3. if tied, choose higher threshold
# ============================================================

candidate_thresholds = np.unique(
    uncertainty
)

policy_rows = []


for threshold in candidate_thresholds:

    review = (
        uncertainty
        >= threshold
    )

    automated = ~review

    flagged_count = int(
        review.sum()
    )

    flagged_errors = int(
        (
            review
            &
            errors
        ).sum()
    )

    error_capture = (
        flagged_errors
        / n_errors
        if n_errors > 0
        else 0.0
    )

    review_fraction = (
        flagged_count
        / len(y_val)
    )

    if automated.sum() > 0:

        automated_accuracy = (
            accuracy_score(
                y_val[
                    automated
                ],
                base_pred[
                    automated
                ]
            )
        )

    else:

        automated_accuracy = np.nan

    policy_rows.append({
        "uncertainty_threshold":
            float(
                threshold
            ),

        "confidence_threshold":
            float(
                1.0 - threshold
            ),

        "flagged_windows":
            flagged_count,

        "review_fraction":
            review_fraction,

        "flagged_errors":
            flagged_errors,

        "total_errors":
            n_errors,

        "error_capture":
            error_capture,

        "automated_windows":
            int(
                automated.sum()
            ),

        "automated_fraction":
            float(
                automated.mean()
            ),

        "automated_accuracy":
            automated_accuracy,
    })


policy_df = pd.DataFrame(
    policy_rows
)


# ============================================================
# 10. SELECT POLICY
# ============================================================

eligible = (
    policy_df[
        policy_df[
            "error_capture"
        ]
        >= TARGET_ERROR_CAPTURE
    ]
    .sort_values(
        [
            "review_fraction",
            "uncertainty_threshold",
        ],
        ascending=[
            True,
            False,
        ]
    )
)

if eligible.empty:
    raise RuntimeError(
        "No threshold satisfies "
        "the target error capture."
    )

selected = (
    eligible.iloc[0]
)

selected_threshold = float(
    selected[
        "uncertainty_threshold"
    ]
)

selected_confidence = float(
    selected[
        "confidence_threshold"
    ]
)


# ============================================================
# 11. APPLY SELECTED POLICY
# ============================================================

review_mask = (
    uncertainty
    >= selected_threshold
)

automated_mask = (
    ~review_mask
)

high_conf_attack = (
    automated_mask
    &
    (base_pred == 1)
)

high_conf_benign = (
    automated_mask
    &
    (base_pred == 0)
)


# ============================================================
# 12. ROUTING ACTION LABELS
# ============================================================

routing_action = np.full(
    len(y_val),
    "verification_required",
    dtype=object
)

routing_action[
    high_conf_benign
] = (
    "continue_monitoring"
)

routing_action[
    high_conf_attack
] = (
    "generic_protective_response"
)


# ============================================================
# 13. AUTOMATED-STREAM METRICS
# ============================================================

auto_y = (
    y_val[
        automated_mask
    ]
)

auto_pred = (
    base_pred[
        automated_mask
    ]
)

automated_metrics = {
    "automated_windows":
        int(
            automated_mask.sum()
        ),

    "automated_fraction":
        float(
            automated_mask.mean()
        ),

    "automated_accuracy":
        accuracy_score(
            auto_y,
            auto_pred
        ),

    "automated_attack_precision":
        precision_score(
            auto_y,
            auto_pred,
            zero_division=0
        ),

    "automated_attack_recall":
        recall_score(
            auto_y,
            auto_pred,
            zero_division=0
        ),

    "automated_attack_f1":
        f1_score(
            auto_y,
            auto_pred,
            zero_division=0
        ),
}


# ============================================================
# 14. REVIEW-STREAM ERROR CAPTURE
# ============================================================

flagged_errors = int(
    (
        review_mask
        &
        errors
    ).sum()
)

error_capture = (
    flagged_errors
    / n_errors
)

review_metrics = {
    "review_windows":
        int(
            review_mask.sum()
        ),

    "review_fraction":
        float(
            review_mask.mean()
        ),

    "errors_captured":
        flagged_errors,

    "total_validation_errors":
        n_errors,

    "error_capture":
        float(
            error_capture
        ),
}


# ============================================================
# 15. ROUTING DISTRIBUTION
# ============================================================

routing_counts = (
    pd.Series(
        routing_action,
        name="action"
    )
    .value_counts()
    .rename_axis(
        "action"
    )
    .reset_index(
        name="windows"
    )
)

routing_counts[
    "fraction"
] = (
    routing_counts[
        "windows"
    ]
    / len(y_val)
)


# ============================================================
# 16. SAVE VALIDATION ROUTING TABLE
#
# This is validation only.
# ============================================================

routing_df = pd.DataFrame({
    "true_attack":
        y_val,

    "calibrated_p_attack":
        p_attack,

    "base_prediction":
        base_pred,

    "confidence":
        confidence,

    "uncertainty":
        uncertainty,

    "prediction_error":
        errors.astype(int),

    "routed_for_verification":
        review_mask.astype(int),

    "routing_action":
        routing_action,
})


routing_file = (
    TABLES_DIR
    / "cyber_stage1_validation_routing.csv"
)

routing_df.to_csv(
    routing_file,
    index=False
)


# ============================================================
# 17. SAVE POLICY CURVE
# ============================================================

policy_curve_file = (
    TABLES_DIR
    / "cyber_stage1_validation_routing_threshold_curve.csv"
)

policy_df.to_csv(
    policy_curve_file,
    index=False
)


# ============================================================
# 18. SAVE FROZEN POLICY CONFIG
# ============================================================

policy_config = {
    "primary_task":
        "Benign vs Attack",

    "probability_source":
        (
            "Stage-1 XGBoost followed by "
            "adopted temperature scaling"
        ),

    "temperature":
        temperature,

    "base_attack_threshold":
        0.5,

    "uncertainty_definition":
        (
            "1 - max(P(Benign), P(Attack))"
        ),

    "target_validation_error_capture":
        TARGET_ERROR_CAPTURE,

    "threshold_selection_rule":
        (
            "among thresholds capturing at least "
            "75% of validation prediction errors, "
            "select the threshold with minimum "
            "verification burden; ties resolved "
            "toward the higher uncertainty threshold"
        ),

    "selected_uncertainty_threshold":
        selected_threshold,

    "equivalent_confidence_threshold":
        selected_confidence,

    "high_confidence_benign_action":
        "continue_monitoring",

    "high_confidence_attack_action":
        "generic_protective_response",

    "low_confidence_action":
        "verification_required",

    "attack_attribution_policy":
        (
            "DoS-vs-Replay attribution is not "
            "trusted for autonomous response because "
            "development discrimination was weak."
        ),

    **review_metrics,

    **automated_metrics,
}


policy_config_file = (
    PROCESSED_DIR
    / "cyber_stage1_frozen_routing_policy.json"
)

with policy_config_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        policy_config,
        f,
        indent=2
    )


# ============================================================
# 19. PRINT SELECTED POLICY
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "SELECTED RISK-AWARE ROUTING POLICY"
)

print(
    "=" * 110
)

print(
    "Target error capture       :",
    f"{TARGET_ERROR_CAPTURE:.2%}"
)

print(
    "Uncertainty threshold      :",
    f"{selected_threshold:.6f}"
)

print(
    "Equivalent confidence      :",
    f"{selected_confidence:.6f}"
)

print(
    "Review windows             :",
    review_metrics[
        "review_windows"
    ]
)

print(
    "Review fraction            :",
    f"{review_metrics['review_fraction']:.2%}"
)

print(
    "Errors captured            :",
    (
        f"{review_metrics['errors_captured']}"
        f"/"
        f"{review_metrics['total_validation_errors']}"
    )
)

print(
    "Error capture              :",
    f"{review_metrics['error_capture']:.2%}"
)


# ============================================================
# 20. AUTOMATED STREAM
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "AUTOMATED HIGH-CONFIDENCE STREAM"
)

print(
    "=" * 110
)

for key, value in (
    automated_metrics.items()
):

    if isinstance(
        value,
        float
    ):

        print(
            f"{key:28s}: "
            f"{value:.6f}"
        )

    else:

        print(
            f"{key:28s}: "
            f"{value}"
        )


# ============================================================
# 21. ROUTING DISTRIBUTION
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "ROUTING ACTION DISTRIBUTION"
)

print(
    "=" * 110
)

print(
    routing_counts.to_string(
        index=False
    )
)


# ============================================================
# 22. POLICY QA
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "POLICY QA"
)

print(
    "=" * 110
)

print(
    "Validation windows accounted for:",
    (
        automated_metrics[
            "automated_windows"
        ]
        +
        review_metrics[
            "review_windows"
        ]
    )
)

assert (
    automated_metrics[
        "automated_windows"
    ]
    +
    review_metrics[
        "review_windows"
    ]
    ==
    len(y_val)
)

assert (
    review_metrics[
        "error_capture"
    ]
    >= TARGET_ERROR_CAPTURE
)

print(
    "✅ Every validation window assigned exactly one route."
)

print(
    "✅ Target error-capture criterion satisfied."
)


# ============================================================
# 23. FINAL
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "RISK-AWARE POLICY FROZEN"
)

print(
    "=" * 110
)

print(
    "✅ Policy config:",
    policy_config_file
)

print(
    "✅ Threshold curve:",
    policy_curve_file
)

print(
    "✅ Validation routing:",
    routing_file
)

print(
    "\n🔒 LOCKED TEST FEATURES WERE NOT LOADED."
)

print(
    "\nNext step:"
)

print(
    "Open the Locked Test exactly once and evaluate:"
)

print(
    "1. Stage-1 attack detection"
)

print(
    "2. Probability calibration"
)

print(
    "3. Frozen risk-aware routing policy"
)

print(
    "4. Physical confounding conclusion remains "
    "development-only and is NOT promoted as "
    "a general attack detector."
)

In [ ]:
# ============================================================
# UAV PROJECT
# ONE-TIME LOCKED TEST EVALUATION
#
# PRIMARY FROZEN TASK:
#   Benign vs Attack
#
# Evaluates ONCE:
#   1. Frozen Stage-1 attack detector
#   2. Frozen temperature calibration
#   3. Frozen risk-aware routing policy
#
# IMPORTANT:
# - No tuning.
# - No threshold changes.
# - No feature changes.
# - No calibration refitting.
# - Test results must NOT affect the pipeline.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    confusion_matrix,
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

WINDOWS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "windows"
)

SPLITS_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "splits"
)

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

MODELS_DIR = (
    PROJECT_DIR
    / "models"
    / "development"
)

TABLES_DIR = (
    PROJECT_DIR
    / "results"
    / "tables"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPERIMENT = "cyber_schema_A_3class"


# ============================================================
# 2. ONE-TIME EXECUTION GUARD
# ============================================================

CONSUMED_MARKER = (
    PROCESSED_DIR
    / "LOCKED_TEST_CONSUMED.json"
)

if CONSUMED_MARKER.exists():

    raise RuntimeError(
        "\nLOCKED TEST HAS ALREADY BEEN CONSUMED.\n"
        f"Marker exists:\n{CONSUMED_MARKER}\n\n"
        "Do NOT rerun final test evaluation."
    )


# ============================================================
# 3. FROZEN ARTIFACTS
# ============================================================

STAGE1_MODEL_FILE = (
    MODELS_DIR
    / "cyber_stage1_attack_detector.joblib"
)

FEATURE_CONFIG_FILE = (
    PROCESSED_DIR
    / "cyber_selected_feature_configuration_corrected.json"
)

CALIBRATION_FILE = (
    PROCESSED_DIR
    / "cyber_stage1_temperature_calibration.json"
)

ROUTING_POLICY_FILE = (
    PROCESSED_DIR
    / "cyber_stage1_frozen_routing_policy.json"
)

SPLIT_CONFIG_FILE = (
    PROCESSED_DIR
    / "frozen_split_configuration.json"
)

WINDOWS_FILE = (
    WINDOWS_DIR
    / f"{EXPERIMENT}_windows.csv"
)

SPLIT_MAP_FILE = (
    SPLITS_DIR
    / f"{EXPERIMENT}_window_split_map.csv"
)


required_files = [
    STAGE1_MODEL_FILE,
    FEATURE_CONFIG_FILE,
    CALIBRATION_FILE,
    ROUTING_POLICY_FILE,
    SPLIT_CONFIG_FILE,
    WINDOWS_FILE,
    SPLIT_MAP_FILE,
]

for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


# ============================================================
# 4. HASH FROZEN DEVELOPMENT ARTIFACTS
#
# Done BEFORE locked-test features are extracted.
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


frozen_artifact_paths = {
    "stage1_model":
        STAGE1_MODEL_FILE,

    "feature_config":
        FEATURE_CONFIG_FILE,

    "temperature_calibration":
        CALIBRATION_FILE,

    "routing_policy":
        ROUTING_POLICY_FILE,

    "split_configuration":
        SPLIT_CONFIG_FILE,
}


artifact_hashes = {
    name: sha256_file(path)
    for name, path
    in frozen_artifact_paths.items()
}


freeze_manifest = {
    "created_before_locked_test_access":
        True,

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "experiment":
        EXPERIMENT,

    "artifacts":
        {
            name: {
                "path":
                    str(path),

                "sha256":
                    artifact_hashes[
                        name
                    ],
            }
            for name, path
            in frozen_artifact_paths.items()
        },
}


FREEZE_MANIFEST_FILE = (
    PROCESSED_DIR
    / "pretest_freeze_manifest.json"
)

with FREEZE_MANIFEST_FILE.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        freeze_manifest,
        f,
        indent=2
    )


print("=" * 115)
print("PRE-TEST FREEZE")
print("=" * 115)

print(
    "✅ Frozen artifacts hashed BEFORE test extraction."
)

for name, value in (
    artifact_hashes.items()
):

    print(
        f"{name:26s}: {value}"
    )


# ============================================================
# 5. LOAD FROZEN CONFIGURATION
# ============================================================

with FEATURE_CONFIG_FILE.open(
    "r",
    encoding="utf-8"
) as f:

    feature_config = json.load(f)


with CALIBRATION_FILE.open(
    "r",
    encoding="utf-8"
) as f:

    calibration = json.load(f)


with ROUTING_POLICY_FILE.open(
    "r",
    encoding="utf-8"
) as f:

    routing_policy = json.load(f)


selected_features = (
    feature_config[
        "selected_features"
    ]
)

temperature = float(
    calibration[
        "temperature"
    ]
)

calibration_adopted = bool(
    calibration[
        "calibration_adopted"
    ]
)

uncertainty_threshold = float(
    routing_policy[
        "selected_uncertainty_threshold"
    ]
)

confidence_threshold = float(
    routing_policy[
        "equivalent_confidence_threshold"
    ]
)


assert calibration_adopted is True


print(
    "\n✅ Frozen selected features:",
    len(selected_features)
)

print(
    "✅ Frozen temperature:",
    temperature
)

print(
    "✅ Frozen uncertainty threshold:",
    uncertainty_threshold
)

print(
    "✅ Frozen confidence threshold:",
    confidence_threshold
)


# ============================================================
# 6. OPEN LOCKED TEST
#
# This is the FIRST and FINAL test-feature access.
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "OPENING LOCKED TEST — FINAL EVALUATION"
)

print(
    "=" * 115
)


split_map = pd.read_csv(
    SPLIT_MAP_FILE,
    low_memory=False
)

windows_df = pd.read_csv(
    WINDOWS_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# Validate IDs
# ------------------------------------------------------------

assert (
    split_map[
        "window_id"
    ].is_unique
)

assert (
    windows_df[
        "window_id"
    ].is_unique
)


test_map = (
    split_map[
        split_map[
            "split"
        ] == "test"
    ]
    .copy()
)


test_ids = set(
    test_map[
        "window_id"
    ]
)


test_df = (
    windows_df[
        windows_df[
            "window_id"
        ].isin(
            test_ids
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# Make deterministic order
# ------------------------------------------------------------

test_df = (
    test_df
    .sort_values(
        "window_id"
    )
    .reset_index(
        drop=True
    )
)

test_map = (
    test_map
    .sort_values(
        "window_id"
    )
    .reset_index(
        drop=True
    )
)


assert len(test_df) == len(
    test_map
)

assert (
    test_df[
        "window_id"
    ].to_numpy()
    ==
    test_map[
        "window_id"
    ].to_numpy()
).all()

assert (
    test_df[
        "attack_class"
    ].astype(str).to_numpy()
    ==
    test_map[
        "attack_class"
    ].astype(str).to_numpy()
).all()


print(
    "✅ Locked test windows:",
    len(test_df)
)

print(
    "\nLocked test class distribution:"
)

print(
    test_df[
        "attack_class"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 7. BUILD FROZEN TEST MATRIX
# ============================================================

missing_features = (
    set(
        selected_features
    )
    -
    set(
        test_df.columns
    )
)

if missing_features:

    raise RuntimeError(
        f"Missing frozen test features: "
        f"{missing_features}"
    )


X_test = (
    test_df[
        selected_features
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


# Binary target:
#
# Benign = 0
# DoS / Replay = 1
# ------------------------------------------------------------

y_test = (
    test_df[
        "attack_class"
    ]
    != "Benign"
).astype(int).to_numpy()


# ============================================================
# 8. LOAD FROZEN STAGE-1 MODEL
# ============================================================

stage1_model = joblib.load(
    STAGE1_MODEL_FILE
)


# ============================================================
# 9. TEMPERATURE SCALING
# ============================================================

EPS = 1e-12


def apply_temperature(
    probability,
    temperature
):

    p = np.clip(
        np.asarray(
            probability,
            dtype=float
        ),
        EPS,
        1.0 - EPS
    )

    logits = np.log(
        p / (1.0 - p)
    )

    calibrated_logits = (
        logits
        / temperature
    )

    calibrated_logits = np.clip(
        calibrated_logits,
        -50,
        50
    )

    return (
        1.0
        /
        (
            1.0
            + np.exp(
                -calibrated_logits
            )
        )
    )


# ============================================================
# 10. FROZEN RAW + CALIBRATED PROBABILITIES
# ============================================================

p_attack_raw = (
    stage1_model
    .predict_proba(
        X_test
    )[:, 1]
)


p_attack_calibrated = (
    apply_temperature(
        p_attack_raw,
        temperature
    )
)


pred_raw = (
    p_attack_raw >= 0.5
).astype(int)

pred_calibrated = (
    p_attack_calibrated
    >= 0.5
).astype(int)


prediction_changes = int(
    np.sum(
        pred_raw
        != pred_calibrated
    )
)


assert prediction_changes == 0


# ============================================================
# 11. CALIBRATION METRICS
# ============================================================

def brier_score(
    y_true,
    probability
):

    return float(
        np.mean(
            (
                probability
                - y_true
            ) ** 2
        )
    )


def equal_width_ece(
    y_true,
    probability,
    n_bins=15
):

    prediction = (
        probability >= 0.5
    ).astype(int)

    confidence = np.where(
        prediction == 1,
        probability,
        1.0 - probability
    )

    correct = (
        prediction
        == y_true
    ).astype(float)

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ece = 0.0

    for i in range(
        n_bins
    ):

        if i == n_bins - 1:

            mask = (
                (confidence >= edges[i])
                &
                (confidence <= edges[i + 1])
            )

        else:

            mask = (
                (confidence >= edges[i])
                &
                (confidence < edges[i + 1])
            )

        if not mask.any():
            continue

        ece += (
            mask.mean()
            *
            abs(
                correct[
                    mask
                ].mean()
                -
                confidence[
                    mask
                ].mean()
            )
        )

    return float(ece)


def adaptive_ece(
    y_true,
    probability,
    n_bins=15
):

    prediction = (
        probability >= 0.5
    ).astype(int)

    confidence = np.where(
        prediction == 1,
        probability,
        1.0 - probability
    )

    correct = (
        prediction
        == y_true
    ).astype(float)

    order = np.argsort(
        confidence
    )

    bins = np.array_split(
        order,
        n_bins
    )

    n = len(
        y_true
    )

    ece = 0.0

    for idx in bins:

        if len(idx) == 0:
            continue

        ece += (
            len(idx)
            / n
            *
            abs(
                correct[
                    idx
                ].mean()
                -
                confidence[
                    idx
                ].mean()
            )
        )

    return float(ece)


def evaluate_probability(
    y_true,
    probability
):

    pred = (
        probability >= 0.5
    ).astype(int)

    probability_matrix = (
        np.column_stack(
            [
                1.0 - probability,
                probability
            ]
        )
    )

    confidence = np.maximum(
        probability,
        1.0 - probability
    )

    accuracy = (
        accuracy_score(
            y_true,
            pred
        )
    )

    return {
        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred
            ),

        "attack_precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "attack_recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "attack_f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probability
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                probability
            ),

        "log_loss":
            log_loss(
                y_true,
                probability_matrix,
                labels=[
                    0,
                    1
                ]
            ),

        "brier":
            brier_score(
                y_true,
                probability
            ),

        "equal_width_ece_15":
            equal_width_ece(
                y_true,
                probability,
                n_bins=15
            ),

        "adaptive_ece_15":
            adaptive_ece(
                y_true,
                probability,
                n_bins=15
            ),

        "mean_confidence":
            float(
                confidence.mean()
            ),

        "confidence_accuracy_gap":
            float(
                confidence.mean()
                - accuracy
            ),
    }


raw_metrics = (
    evaluate_probability(
        y_test,
        p_attack_raw
    )
)

calibrated_metrics = (
    evaluate_probability(
        y_test,
        p_attack_calibrated
    )
)


# ============================================================
# 12. LOCKED TEST CONFUSION MATRIX
# ============================================================

test_cm = pd.DataFrame(
    confusion_matrix(
        y_test,
        pred_calibrated,
        labels=[
            0,
            1
        ]
    ),
    index=[
        "True Benign",
        "True Attack"
    ],
    columns=[
        "Pred Benign",
        "Pred Attack"
    ]
)


# ============================================================
# 13. APPLY FROZEN RISK-AWARE ROUTING POLICY
# ============================================================

confidence = np.maximum(
    p_attack_calibrated,
    1.0 - p_attack_calibrated
)

uncertainty = (
    1.0
    - confidence
)

errors = (
    pred_calibrated
    != y_test
)


review_mask = (
    uncertainty
    >= uncertainty_threshold
)

automated_mask = (
    ~review_mask
)


high_conf_benign = (
    automated_mask
    &
    (
        pred_calibrated == 0
    )
)

high_conf_attack = (
    automated_mask
    &
    (
        pred_calibrated == 1
    )
)


routing_action = np.full(
    len(test_df),
    "verification_required",
    dtype=object
)

routing_action[
    high_conf_benign
] = (
    "continue_monitoring"
)

routing_action[
    high_conf_attack
] = (
    "generic_protective_response"
)


# ============================================================
# 14. ROUTING PERFORMANCE
# ============================================================

total_errors = int(
    errors.sum()
)

review_windows = int(
    review_mask.sum()
)

captured_errors = int(
    (
        review_mask
        &
        errors
    ).sum()
)

error_capture = (
    captured_errors
    / total_errors
    if total_errors > 0
    else 1.0
)


auto_y = (
    y_test[
        automated_mask
    ]
)

auto_pred = (
    pred_calibrated[
        automated_mask
    ]
)


if len(auto_y) == 0:

    raise RuntimeError(
        "Frozen policy routed every "
        "test window to verification."
    )


routing_metrics = {
    "test_windows":
        len(test_df),

    "base_errors":
        total_errors,

    "review_windows":
        review_windows,

    "review_fraction":
        float(
            review_mask.mean()
        ),

    "captured_errors":
        captured_errors,

    "error_capture":
        float(
            error_capture
        ),

    "automated_windows":
        int(
            automated_mask.sum()
        ),

    "automated_fraction":
        float(
            automated_mask.mean()
        ),

    "automated_accuracy":
        accuracy_score(
            auto_y,
            auto_pred
        ),

    "automated_attack_precision":
        precision_score(
            auto_y,
            auto_pred,
            zero_division=0
        ),

    "automated_attack_recall":
        recall_score(
            auto_y,
            auto_pred,
            zero_division=0
        ),

    "automated_attack_f1":
        f1_score(
            auto_y,
            auto_pred,
            zero_division=0
        ),
}


routing_counts = (
    pd.Series(
        routing_action,
        name="action"
    )
    .value_counts()
    .rename_axis(
        "action"
    )
    .reset_index(
        name="windows"
    )
)

routing_counts[
    "fraction"
] = (
    routing_counts[
        "windows"
    ]
    /
    len(test_df)
)


# ============================================================
# 15. SAVE FINAL PREDICTION RECORD
# ============================================================

prediction_df = pd.DataFrame({
    "window_id":
        test_df[
            "window_id"
        ].to_numpy(),

    "true_class":
        test_df[
            "attack_class"
        ].to_numpy(),

    "true_attack":
        y_test,

    "raw_p_attack":
        p_attack_raw,

    "calibrated_p_attack":
        p_attack_calibrated,

    "prediction":
        pred_calibrated,

    "confidence":
        confidence,

    "uncertainty":
        uncertainty,

    "prediction_error":
        errors.astype(int),

    "routed_for_verification":
        review_mask.astype(int),

    "routing_action":
        routing_action,
})


predictions_file = (
    TABLES_DIR
    / "cyber_stage1_locked_test_predictions.csv"
)

prediction_df.to_csv(
    predictions_file,
    index=False
)


# ============================================================
# 16. SAVE FINAL METRICS
# ============================================================

comparison_df = pd.DataFrame([
    {
        "state":
            "uncalibrated",
        **raw_metrics
    },
    {
        "state":
            "temperature_scaled",
        **calibrated_metrics
    },
])


comparison_file = (
    TABLES_DIR
    / "cyber_stage1_locked_test_metrics.csv"
)

comparison_df.to_csv(
    comparison_file,
    index=False
)


cm_file = (
    TABLES_DIR
    / "cyber_stage1_locked_test_confusion_matrix.csv"
)

test_cm.to_csv(
    cm_file
)


routing_metrics_file = (
    TABLES_DIR
    / "cyber_stage1_locked_test_routing_metrics.csv"
)

pd.DataFrame([
    routing_metrics
]).to_csv(
    routing_metrics_file,
    index=False
)


routing_counts_file = (
    TABLES_DIR
    / "cyber_stage1_locked_test_routing_distribution.csv"
)

routing_counts.to_csv(
    routing_counts_file,
    index=False
)


# ============================================================
# 17. FINAL TEST SUMMARY JSON
# ============================================================

final_summary = {
    "experiment":
        EXPERIMENT,

    "evaluation_policy":
        "one-time locked test",

    "test_windows":
        len(test_df),

    "selected_source_families":
        feature_config[
            "selected_sources"
        ],

    "selected_aggregated_features":
        len(
            selected_features
        ),

    "temperature":
        temperature,

    "uncertainty_threshold":
        uncertainty_threshold,

    "confidence_threshold":
        confidence_threshold,

    "uncalibrated_metrics":
        raw_metrics,

    "temperature_scaled_metrics":
        calibrated_metrics,

    "prediction_changes_from_temperature":
        prediction_changes,

    "routing_metrics":
        routing_metrics,

    "routing_distribution":
        routing_counts.to_dict(
            orient="records"
        ),

    "physical_branch_interpretation":
        (
            "Physical Schema-A perfect validation "
            "performance is not promoted as general "
            "attack-detection evidence because "
            "barometer absolute level was identified "
            "as a strong session/scenario confound."
        ),
}


final_summary_file = (
    TABLES_DIR
    / "final_locked_test_summary.json"
)

with final_summary_file.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_summary,
        f,
        indent=2
    )


# ============================================================
# 18. CREATE TEST-CONSUMED MARKER
#
# Created ONLY after successful final evaluation.
# ============================================================

consumed_record = {
    "locked_test_consumed":
        True,

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "experiment":
        EXPERIMENT,

    "test_windows":
        len(test_df),

    "pretest_freeze_manifest":
        str(
            FREEZE_MANIFEST_FILE
        ),

    "final_summary":
        str(
            final_summary_file
        ),

    "policy":
        (
            "No further model, feature, calibration, "
            "or routing-policy changes are permitted "
            "based on locked-test performance."
        ),
}


with CONSUMED_MARKER.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        consumed_record,
        f,
        indent=2
    )


# ============================================================
# 19. PRINT FINAL DETECTION RESULTS
# ============================================================

print(
    "\n"
    + "=" * 130
)

print(
    "FINAL LOCKED TEST — STAGE-1 ATTACK DETECTION"
)

print(
    "=" * 130
)

print(
    comparison_df[
        [
            "state",
            "accuracy",
            "balanced_accuracy",
            "attack_precision",
            "attack_recall",
            "attack_f1",
            "roc_auc",
            "pr_auc",
            "log_loss",
            "brier",
            "equal_width_ece_15",
            "adaptive_ece_15",
            "mean_confidence",
            "confidence_accuracy_gap",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 20. CONFUSION MATRIX
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "FINAL LOCKED TEST CONFUSION MATRIX"
)

print(
    "=" * 115
)

print(
    test_cm.to_string()
)


# ============================================================
# 21. CALIBRATION GENERALIZATION
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "FINAL CALIBRATION GENERALIZATION"
)

print(
    "=" * 115
)

print(
    "Prediction changes:",
    prediction_changes
)

print(
    "Raw Log Loss:",
    f"{raw_metrics['log_loss']:.6f}"
)

print(
    "Calibrated Log Loss:",
    f"{calibrated_metrics['log_loss']:.6f}"
)

print(
    "Raw Brier:",
    f"{raw_metrics['brier']:.6f}"
)

print(
    "Calibrated Brier:",
    f"{calibrated_metrics['brier']:.6f}"
)

print(
    "Raw Equal-width ECE:",
    f"{raw_metrics['equal_width_ece_15']:.6f}"
)

print(
    "Calibrated Equal-width ECE:",
    f"{calibrated_metrics['equal_width_ece_15']:.6f}"
)

print(
    "Raw Adaptive ECE:",
    f"{raw_metrics['adaptive_ece_15']:.6f}"
)

print(
    "Calibrated Adaptive ECE:",
    f"{calibrated_metrics['adaptive_ece_15']:.6f}"
)


# ============================================================
# 22. FROZEN ROUTING POLICY ON TEST
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "FINAL LOCKED TEST — FROZEN RISK-AWARE ROUTING"
)

print(
    "=" * 115
)

print(
    "Frozen uncertainty threshold:",
    f"{uncertainty_threshold:.6f}"
)

print(
    "Frozen confidence threshold :",
    f"{confidence_threshold:.6f}"
)

print(
    "\nBase errors:",
    routing_metrics[
        "base_errors"
    ]
)

print(
    "Review windows:",
    routing_metrics[
        "review_windows"
    ]
)

print(
    "Review fraction:",
    f"{routing_metrics['review_fraction']:.2%}"
)

print(
    "Errors captured:",
    (
        f"{routing_metrics['captured_errors']}"
        f"/"
        f"{routing_metrics['base_errors']}"
    )
)

print(
    "Error capture:",
    f"{routing_metrics['error_capture']:.2%}"
)

print(
    "\nAutomated windows:",
    routing_metrics[
        "automated_windows"
    ]
)

print(
    "Automated fraction:",
    f"{routing_metrics['automated_fraction']:.2%}"
)

print(
    "Automated accuracy:",
    f"{routing_metrics['automated_accuracy']:.6f}"
)

print(
    "Automated Attack precision:",
    f"{routing_metrics['automated_attack_precision']:.6f}"
)

print(
    "Automated Attack recall:",
    f"{routing_metrics['automated_attack_recall']:.6f}"
)

print(
    "Automated Attack F1:",
    f"{routing_metrics['automated_attack_f1']:.6f}"
)


# ============================================================
# 23. ROUTING DISTRIBUTION
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "FINAL ROUTING ACTION DISTRIBUTION"
)

print(
    "=" * 115
)

print(
    routing_counts.to_string(
        index=False
    )
)


# ============================================================
# 24. FINAL LOCK
# ============================================================

print(
    "\n"
    + "=" * 115
)

print(
    "LOCKED TEST CONSUMED — PROJECT EVALUATION FROZEN"
)

print(
    "=" * 115
)

print(
    "✅ Final metrics:",
    comparison_file
)

print(
    "✅ Confusion matrix:",
    cm_file
)

print(
    "✅ Routing metrics:",
    routing_metrics_file
)

print(
    "✅ Final summary:",
    final_summary_file
)

print(
    "✅ Test-consumed marker:",
    CONSUMED_MARKER
)

print(
    "\n🔒 DO NOT tune anything after these results."
)

In [ ]:
# ============================================================
# PART 1/5
# FINAL PROJECT PACKAGE — LOAD FROZEN RESULTS
#
# NO TRAINING
# NO TUNING
# NO TEST MODIFICATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

RESULTS_DIR = PROJECT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

MODELS_DIR = PROJECT_DIR / "models"

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. REQUIRED FROZEN FILES
# ============================================================

FILES = {

    "parsed_manifest":
        TABLES_DIR
        / "parsed_section_manifest.csv",

    "split_manifest":
        TABLES_DIR
        / "frozen_split_manifest.csv",

    "stage1_validation":
        TABLES_DIR
        / "cyber_stage1_validation_metrics.csv",

    "stage2_validation":
        TABLES_DIR
        / "cyber_stage2_validation_metrics.csv",

    "hierarchical_validation":
        TABLES_DIR
        / "cyber_hierarchical_validation_metrics.csv",

    "locked_test_metrics":
        TABLES_DIR
        / "cyber_stage1_locked_test_metrics.csv",

    "locked_test_cm":
        TABLES_DIR
        / "cyber_stage1_locked_test_confusion_matrix.csv",

    "locked_test_routing":
        TABLES_DIR
        / "cyber_stage1_locked_test_routing_metrics.csv",

    "physical_sensitivity":
        TABLES_DIR
        / "physical_barometer_confounding_sensitivity.csv",

    "physical_distributions":
        TABLES_DIR
        / "physical_barometer_class_distributions.csv",

    "feature_reduction":
        TABLES_DIR
        / "cyber_source_family_reduction_corrected.csv",

    "feature_config":
        PROCESSED_DIR
        / "cyber_selected_feature_configuration_corrected.json",

    "calibration_config":
        PROCESSED_DIR
        / "cyber_stage1_temperature_calibration.json",

    "routing_config":
        PROCESSED_DIR
        / "cyber_stage1_frozen_routing_policy.json",

    "window_config":
        PROCESSED_DIR
        / "window_feature_configuration.json",

    "consumed_marker":
        PROCESSED_DIR
        / "LOCKED_TEST_CONSUMED.json",
}


for name, path in FILES.items():

    if not path.exists():

        raise FileNotFoundError(
            f"Missing artifact:\n"
            f"{name}\n{path}"
        )


# ============================================================
# 3. VERIFY TEST IS ALREADY CONSUMED
# ============================================================

with FILES["consumed_marker"].open(
    "r",
    encoding="utf-8"
) as f:

    consumed = json.load(f)


assert consumed.get(
    "locked_test_consumed",
    False
)


print(
    "✅ Locked Test already consumed and frozen."
)

print(
    "✅ No model will be retrained."
)


# ============================================================
# 4. LOAD TABLES
# ============================================================

parsed_manifest = pd.read_csv(
    FILES["parsed_manifest"]
)

split_manifest = pd.read_csv(
    FILES["split_manifest"]
)

stage1_val = pd.read_csv(
    FILES["stage1_validation"]
).iloc[0]

stage2_val = pd.read_csv(
    FILES["stage2_validation"]
).iloc[0]

hierarchical_val = pd.read_csv(
    FILES["hierarchical_validation"]
).iloc[0]

locked_metrics = pd.read_csv(
    FILES["locked_test_metrics"]
)

raw_test = (
    locked_metrics[
        locked_metrics["state"]
        == "uncalibrated"
    ]
    .iloc[0]
)

cal_test = (
    locked_metrics[
        locked_metrics["state"]
        == "temperature_scaled"
    ]
    .iloc[0]
)

locked_cm = pd.read_csv(
    FILES["locked_test_cm"],
    index_col=0
)

routing_test = pd.read_csv(
    FILES["locked_test_routing"]
).iloc[0]

physical_sensitivity = pd.read_csv(
    FILES["physical_sensitivity"]
)

physical_distributions = pd.read_csv(
    FILES["physical_distributions"]
)

feature_reduction = pd.read_csv(
    FILES["feature_reduction"]
)


# ============================================================
# 5. LOAD JSON CONFIGS
# ============================================================

with FILES["feature_config"].open(
    "r",
    encoding="utf-8"
) as f:

    feature_config = json.load(f)


with FILES["calibration_config"].open(
    "r",
    encoding="utf-8"
) as f:

    calibration_config = json.load(f)


with FILES["routing_config"].open(
    "r",
    encoding="utf-8"
) as f:

    routing_config = json.load(f)


with FILES["window_config"].open(
    "r",
    encoding="utf-8"
) as f:

    window_config = json.load(f)


# ============================================================
# 6. HELPERS
# ============================================================

def f4(value):
    return f"{float(value):.4f}"


def pct(value):
    return f"{100 * float(value):.2f}%"


def markdown_table(df):

    try:

        return df.to_markdown(
            index=False
        )

    except Exception:

        cols = list(
            df.columns
        )

        lines = [
            "| "
            + " | ".join(
                str(c)
                for c in cols
            )
            + " |",

            "| "
            + " | ".join(
                ["---"] * len(cols)
            )
            + " |",
        ]

        for _, row in df.iterrows():

            lines.append(
                "| "
                + " | ".join(
                    str(row[c])
                    for c in cols
                )
                + " |"
            )

        return "\n".join(
            lines
        )


# ============================================================
# 7. IMPORTANT FINAL NUMBERS
# ============================================================

temperature = float(
    calibration_config["temperature"]
)

uncertainty_threshold = float(
    routing_config[
        "selected_uncertainty_threshold"
    ]
)

confidence_threshold = float(
    routing_config[
        "equivalent_confidence_threshold"
    ]
)


selected_sources = (
    feature_config[
        "selected_sources"
    ]
)

selected_feature_count = int(
    feature_config[
        "selected_aggregated_feature_count"
    ]
)

full_feature_count = int(
    feature_config[
        "true_full_aggregated_feature_count"
    ]
)

feature_retention = float(
    feature_config[
        "corrected_macro_f1_retention"
    ]
)


# ============================================================
# 8. DOCUMENTATION TABLES
# ============================================================

section_doc = (
    parsed_manifest[
        [
            "section",
            "attack_class",
            "modality",
            "schema_id",
            "rows",
            "predictor_columns",
        ]
    ]
    .copy()
)

section_doc.columns = [
    "Section",
    "Class",
    "Modality",
    "Schema",
    "Rows",
    "Predictors",
]


cyber_split = (
    split_manifest[
        split_manifest["experiment"]
        == "cyber_schema_A_3class"
    ]
)

split_summary = (
    cyber_split
    .groupby(
        "split",
        as_index=False
    )["windows"]
    .sum()
)

order_map = {
    "train": 0,
    "validation": 1,
    "test": 2,
}

split_summary["_order"] = (
    split_summary["split"]
    .map(order_map)
)

split_summary = (
    split_summary
    .sort_values("_order")
    .drop(
        columns="_order"
    )
)

split_summary.columns = [
    "Partition",
    "Windows",
]


barometer_ranges = (
    physical_distributions[
        (
            physical_distributions["split"]
            == "validation"
        )
        &
        (
            physical_distributions["feature"]
            == "barometer__mean"
        )
    ][
        [
            "attack_class",
            "min",
            "max",
            "median",
        ]
    ]
    .copy()
)

barometer_ranges.columns = [
    "Class",
    "Minimum",
    "Maximum",
    "Median",
]


print(
    "\n"
    + "=" * 100
)

print(
    "PART 1 COMPLETE"
)

print(
    "=" * 100
)

print(
    "Selected sources:",
    selected_sources
)

print(
    "Selected features:",
    selected_feature_count
)

print(
    "Temperature:",
    temperature
)

print(
    "Locked Test Attack F1:",
    f4(
        cal_test["attack_f1"]
    )
)

In [ ]:
# ============================================================
# PART 2/5
# FINAL FIGURES
# ============================================================


# ============================================================
# FIGURE 1
# VALIDATION VS LOCKED TEST
# ============================================================

metric_names = [
    "Accuracy",
    "Balanced\nAccuracy",
    "Attack\nRecall",
    "Attack\nF1",
    "ROC-AUC",
    "PR-AUC",
]

validation_values = [
    float(stage1_val["accuracy"]),
    float(stage1_val["balanced_accuracy"]),
    float(stage1_val["recall_attack"]),
    float(stage1_val["f1_attack"]),
    float(stage1_val["roc_auc"]),
    float(stage1_val["pr_auc"]),
]

test_values = [
    float(cal_test["accuracy"]),
    float(cal_test["balanced_accuracy"]),
    float(cal_test["attack_recall"]),
    float(cal_test["attack_f1"]),
    float(cal_test["roc_auc"]),
    float(cal_test["pr_auc"]),
]


x = np.arange(
    len(metric_names)
)

width = 0.36


fig, ax = plt.subplots(
    figsize=(10, 5.5)
)

bars1 = ax.bar(
    x - width / 2,
    validation_values,
    width,
    label="Validation"
)

bars2 = ax.bar(
    x + width / 2,
    test_values,
    width,
    label="Locked Test"
)

ax.set_ylim(
    0,
    1.05
)

ax.set_ylabel(
    "Score"
)

ax.set_title(
    "Stage-1 UAV Attack Detection: "
    "Validation vs Locked Temporal Test"
)

ax.set_xticks(x)

ax.set_xticklabels(
    metric_names
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)


for bars in [
    bars1,
    bars2
]:

    for bar in bars:

        value = (
            bar.get_height()
        )

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            value + 0.015,
            f"{value:.2f}",
            ha="center",
            fontsize=8
        )


plt.tight_layout()

FIG_VALIDATION_TEST = (
    FIGURES_DIR
    / "validation_vs_locked_test.png"
)

plt.savefig(
    FIG_VALIDATION_TEST,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# FIGURE 2
# LOCKED TEST CONFUSION MATRIX
# ============================================================

cm_values = (
    locked_cm.to_numpy()
)


fig, ax = plt.subplots(
    figsize=(6, 5)
)

image = ax.imshow(
    cm_values
)

ax.set_title(
    "Locked Test Confusion Matrix\n"
    "Stage-1 Benign vs Attack"
)

ax.set_xlabel(
    "Predicted Class"
)

ax.set_ylabel(
    "True Class"
)

ax.set_xticks(
    np.arange(
        len(
            locked_cm.columns
        )
    )
)

ax.set_xticklabels(
    locked_cm.columns
)

ax.set_yticks(
    np.arange(
        len(
            locked_cm.index
        )
    )
)

ax.set_yticklabels(
    locked_cm.index
)


for i in range(
    cm_values.shape[0]
):

    for j in range(
        cm_values.shape[1]
    ):

        ax.text(
            j,
            i,
            str(
                int(
                    cm_values[
                        i,
                        j
                    ]
                )
            ),
            ha="center",
            va="center",
            fontsize=14
        )


fig.colorbar(
    image,
    ax=ax
)

plt.tight_layout()

FIG_CM = (
    FIGURES_DIR
    / "locked_test_confusion_matrix.png"
)

plt.savefig(
    FIG_CM,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# FIGURE 3
# CALIBRATION
# ============================================================

cal_names = [
    "Log Loss",
    "Brier",
    "Equal-width\nECE",
    "Adaptive\nECE",
]

raw_values = [
    float(raw_test["log_loss"]),
    float(raw_test["brier"]),
    float(
        raw_test[
            "equal_width_ece_15"
        ]
    ),
    float(
        raw_test[
            "adaptive_ece_15"
        ]
    ),
]

scaled_values = [
    float(cal_test["log_loss"]),
    float(cal_test["brier"]),
    float(
        cal_test[
            "equal_width_ece_15"
        ]
    ),
    float(
        cal_test[
            "adaptive_ece_15"
        ]
    ),
]


x = np.arange(
    len(cal_names)
)

fig, ax = plt.subplots(
    figsize=(8.5, 5.2)
)

ax.bar(
    x - width / 2,
    raw_values,
    width,
    label="Uncalibrated"
)

ax.bar(
    x + width / 2,
    scaled_values,
    width,
    label="Temperature Scaled"
)

ax.set_xticks(x)

ax.set_xticklabels(
    cal_names
)

ax.set_ylabel(
    "Error / Calibration Metric"
)

ax.set_title(
    "Calibration Generalization "
    "on the Locked Test"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

FIG_CALIBRATION = (
    FIGURES_DIR
    / "locked_test_calibration_generalization.png"
)

plt.savefig(
    FIG_CALIBRATION,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# FIGURE 4
# RISK-AWARE ROUTING
# ============================================================

routing_names = [
    "Review\nFraction",
    "Automated\nFraction",
    "Error\nCapture",
    "Automated\nAccuracy",
    "Automated\nAttack F1",
]

routing_values = [
    float(
        routing_test[
            "review_fraction"
        ]
    ),
    float(
        routing_test[
            "automated_fraction"
        ]
    ),
    float(
        routing_test[
            "error_capture"
        ]
    ),
    float(
        routing_test[
            "automated_accuracy"
        ]
    ),
    float(
        routing_test[
            "automated_attack_f1"
        ]
    ),
]


fig, ax = plt.subplots(
    figsize=(8.5, 5.2)
)

bars = ax.bar(
    routing_names,
    routing_values
)

ax.set_ylim(
    0,
    1.05
)

ax.set_ylabel(
    "Fraction / Score"
)

ax.set_title(
    "Frozen Risk-Aware Routing "
    "on the Locked Test"
)

ax.grid(
    axis="y",
    alpha=0.25
)


for bar in bars:

    value = (
        bar.get_height()
    )

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,
        value + 0.015,
        f"{value:.2f}",
        ha="center",
        fontsize=9
    )


plt.tight_layout()

FIG_ROUTING = (
    FIGURES_DIR
    / "locked_test_risk_aware_routing.png"
)

plt.savefig(
    FIG_ROUTING,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# FIGURE 5
# PHYSICAL BAROMETER CONFOUNDING
# ============================================================

physical_order = [
    "all_features",
    "remove_all_barometer",
    "remove_barometer_absolute_level",
    "all_sources_dynamic_only",
]


physical_plot = (
    physical_sensitivity
    .set_index(
        "configuration"
    )
    .loc[
        physical_order
    ]
    .reset_index()
)


labels = [
    "All\nFeatures",
    "Remove\nBarometer",
    "Remove Absolute\nBarometer",
    "Dynamic\nOnly",
]


fig, ax = plt.subplots(
    figsize=(8.5, 5.2)
)

bars = ax.bar(
    labels,
    physical_plot[
        "macro_f1"
    ]
)

ax.set_ylim(
    0,
    1.05
)

ax.set_ylabel(
    "Validation Macro F1"
)

ax.set_title(
    "Physical Branch Sensitivity "
    "to Barometer Features"
)

ax.grid(
    axis="y",
    alpha=0.25
)


for bar in bars:

    value = (
        bar.get_height()
    )

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        fontsize=9
    )


plt.tight_layout()

FIG_PHYSICAL = (
    FIGURES_DIR
    / "physical_barometer_confounding.png"
)

plt.savefig(
    FIG_PHYSICAL,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# FIGURE 6
# FEATURE REDUCTION
# ============================================================

reduction_plot = (
    feature_reduction
    .sort_values(
        "source_families"
    )
)


full_f1 = float(
    feature_config[
        "true_full_validation_macro_f1"
    ]
)

retention_threshold = float(
    feature_config[
        "retention_threshold"
    ]
)


fig, ax = plt.subplots(
    figsize=(8.5, 5.2)
)

ax.plot(
    reduction_plot[
        "source_families"
    ],
    reduction_plot[
        "macro_f1"
    ],
    marker="o",
    label="Reduced Model"
)

ax.axhline(
    full_f1,
    linestyle="--",
    label="Full Model"
)

ax.axhline(
    retention_threshold,
    linestyle=":",
    label="99% Retention Threshold"
)

ax.scatter(
    [
        feature_config[
            "selected_source_family_count"
        ]
    ],
    [
        feature_config[
            "selected_validation_macro_f1"
        ]
    ],
    s=90,
    label="Selected"
)

ax.set_xlabel(
    "Source Feature Families"
)

ax.set_ylabel(
    "Validation Macro F1"
)

ax.set_title(
    "Cyber Source-Family Feature Reduction"
)

ax.grid(
    alpha=0.25
)

ax.legend()

plt.tight_layout()

FIG_REDUCTION = (
    FIGURES_DIR
    / "cyber_feature_reduction.png"
)

plt.savefig(
    FIG_REDUCTION,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


print(
    "=" * 100
)

print(
    "PART 2 COMPLETE — FIGURES"
)

print(
    "=" * 100
)


for path in [
    FIG_VALIDATION_TEST,
    FIG_CM,
    FIG_CALIBRATION,
    FIG_ROUTING,
    FIG_PHYSICAL,
    FIG_REDUCTION,
]:

    print(
        "✅",
        path.name
    )

In [ ]:
# ============================================================
# PART 3/5
# README + METHODOLOGY
# ============================================================


# ============================================================
# README
# ============================================================

readme = f"""
# Reliable UAV Cyber-Physical Security

Leakage-aware UAV attack detection, probability calibration,
and risk-aware routing under temporal distribution shift.

## Overview

This project studies UAV cybersecurity with emphasis on
reliability rather than headline classification accuracy.

The workflow includes:

- multi-schema dataset auditing,
- leakage-aware parsing,
- temporal window construction,
- chronological group-held-out evaluation,
- feature reduction,
- probability calibration,
- uncertainty-aware routing,
- and explicit confounding analysis.

## Dataset structure

The original CSV is not one homogeneous machine-learning table.

It contains ten cyber and physical sections with different schemas.

{markdown_table(section_doc)}

A naïve global classifier could learn schema identity instead of
attack behavior.

Therefore, the primary comparable experiment uses Schema A:

- Benign
- DoS
- Replay

## Primary cyber task

The final Stage-1 task is:

**Benign vs Attack**

where:

`Attack = DoS OR Replay`

The model uses windows of 20 cyber observations.

Temporal segments are separated when timestamps reset or when
a sufficiently large temporal gap occurs.

## Frozen split

The experiment uses a chronological segment-grouped split.

{markdown_table(split_summary)}

No temporal segment appears in more than one partition.

The Locked Test was evaluated only after the model, features,
calibration, and routing policy were frozen.

## Final feature set

The full temporal representation contained:

**{full_feature_count} aggregated features**

The final lightweight detector uses only:

- `time_since_last_packet`
- `wlan.fc.type`

Final aggregated feature count:

**{selected_feature_count}**

Validation Macro-F1 retention relative to the full model:

**{pct(feature_retention)}**

![Feature reduction](results/figures/cyber_feature_reduction.png)

## Stage-1 results

| Metric | Validation | Locked Test |
|---|---:|---:|
| Accuracy | {f4(stage1_val["accuracy"])} | {f4(cal_test["accuracy"])} |
| Balanced Accuracy | {f4(stage1_val["balanced_accuracy"])} | {f4(cal_test["balanced_accuracy"])} |
| Attack Precision | {f4(stage1_val["precision_attack"])} | {f4(cal_test["attack_precision"])} |
| Attack Recall | {f4(stage1_val["recall_attack"])} | {f4(cal_test["attack_recall"])} |
| Attack F1 | {f4(stage1_val["f1_attack"])} | {f4(cal_test["attack_f1"])} |
| ROC-AUC | {f4(stage1_val["roc_auc"])} | {f4(cal_test["roc_auc"])} |
| PR-AUC | {f4(stage1_val["pr_auc"])} | {f4(cal_test["pr_auc"])} |

![Validation vs Locked Test](results/figures/validation_vs_locked_test.png)

The locked temporal test exposed a substantial operating-point
generalization gap.

At the frozen 0.5 threshold:

- all 96 benign windows were classified correctly,
- 151 attack windows were detected,
- 108 attack windows were missed.

![Locked Test Confusion Matrix](results/figures/locked_test_confusion_matrix.png)

Despite the recall degradation, ranking remained strong:

- ROC-AUC = **{f4(cal_test["roc_auc"])}**
- PR-AUC = **{f4(cal_test["pr_auc"])}**

## Attack attribution

The secondary DoS-vs-Replay model was intentionally evaluated
separately.

Validation:

- Accuracy = **{f4(stage2_val["accuracy"])}**
- Balanced Accuracy = **{f4(stage2_val["balanced_accuracy"])}**
- Macro F1 = **{f4(stage2_val["macro_f1"])}**
- ROC-AUC = **{f4(stage2_val["roc_auc"])}**

This is not strong enough for autonomous attack-specific response.

## Probability calibration

Temperature scaling was fitted using grouped out-of-fold predictions
from the frozen training partition only.

Frozen temperature:

**T = {temperature:.8f}**

Locked Test:

| Metric | Raw | Temperature Scaled |
|---|---:|---:|
| Log Loss | {f4(raw_test["log_loss"])} | {f4(cal_test["log_loss"])} |
| Brier | {f4(raw_test["brier"])} | {f4(cal_test["brier"])} |
| Equal-width ECE | {f4(raw_test["equal_width_ece_15"])} | {f4(cal_test["equal_width_ece_15"])} |
| Adaptive ECE | {f4(raw_test["adaptive_ece_15"])} | {f4(cal_test["adaptive_ece_15"])} |

Temperature scaling changed no labels but improved all reported
probability-quality metrics on the Locked Test.

![Calibration](results/figures/locked_test_calibration_generalization.png)

## Risk-aware routing

Uncertainty is defined as:

`1 - max(P(Benign), P(Attack))`

Frozen uncertainty threshold:

**{uncertainty_threshold:.6f}**

Equivalent confidence threshold:

**{confidence_threshold:.6f}**

Locked Test:

- Verification fraction = **{pct(routing_test["review_fraction"])}**
- Error capture = **{pct(routing_test["error_capture"])}**
- Automated fraction = **{pct(routing_test["automated_fraction"])}**
- Automated accuracy = **{pct(routing_test["automated_accuracy"])}**
- Automated Attack F1 = **{f4(routing_test["automated_attack_f1"])}**

![Risk-aware Routing](results/figures/locked_test_risk_aware_routing.png)

Routing actions:

- high-confidence Benign → continue monitoring
- high-confidence Attack → generic protective response
- low-confidence prediction → verification required

## Physical-domain finding

The physical Schema-A XGBoost initially achieved perfect validation
classification.

However, barometer level dominated the result.

Removing the barometer reduced Macro F1 to:

**{f4(float(physical_sensitivity.loc[
    physical_sensitivity["configuration"] == "remove_all_barometer",
    "macro_f1"
].iloc[0]))}**

Removing absolute barometer-level statistics reduced Macro F1 to:

**{f4(float(physical_sensitivity.loc[
    physical_sensitivity["configuration"] == "remove_barometer_absolute_level",
    "macro_f1"
].iloc[0]))}**

![Physical Confounding](results/figures/physical_barometer_confounding.png)

The physical branch is therefore reported as a confounding case study,
not as evidence of perfect general UAV attack detection.

## Main methodological contributions

1. Reconstruction of a heterogeneous multi-schema UAV dataset.
2. Schema-aware experimental design.
3. Leakage-resistant temporal segmentation and splitting.
4. Reduction from {full_feature_count} to {selected_feature_count}
   aggregated cyber features.
5. Separation of attack detection from weak attack attribution.
6. Grouped OOF temperature calibration.
7. Frozen uncertainty-based routing.
8. Explicit reporting of temporal generalization failure and
   physical sensor confounding.

## Repository structure

```text
reliable-uav-cyber-physical-security/
├── data/
│   ├── README.md
│   ├── raw/
│   └── processed/
├── models/
├── notebooks/
├── results/
│   ├── figures/
│   └── tables/
├── README.md
├── METHODOLOGY.md
├── RESULTS.md
├── LIMITATIONS.md
├── MODEL_CARD.md
├── requirements.txt
└── .gitignore ```

##Intended use

Cybersecurity research and methodological demonstration.

This repository is not a certified UAV safety system or
production autonomous response controller.
"""

(PROJECT_DIR / "README.md").write_text(
readme.strip() + "\n",
encoding="utf-8"
)

#============================================================
#METHODOLOGY
#============================================================

cyber_window = (
window_config[
"cyber_schema_A_3class"
]
)

physical_window = (
window_config[
"physical_schema_A_3class"
]
)

methodology = f"""

Methodology
##1. Objective

The objective is to evaluate UAV cyber-physical attack detection
while controlling for schema leakage, temporal overlap, probability
miscalibration, and uncertain predictions.

##2. Raw dataset reconstruction

The raw CSV contains:

54,774 data rows
10 embedded section headers
54,784 total physical lines

The ten reconstructed sections are shown below.

{markdown_table(section_doc)}

Different classes do not always use the same schema.

A single naïve five-class table was therefore rejected.

##3. Schema-aware experiments

The primary comparable Schema-A classes are:

Benign
DoS
Replay

Separate Cyber and Physical Schema-A experiments were created.

Evil Twin and FDI were retained only in secondary within-schema
analyses.

##4. Feature governance

Likely identity and collection variables were excluded before model
development.

Cyber exclusions included variables such as:

timestamps,
frame identifiers,
MAC/IP endpoint identity,
sequence identifiers,
raw payload-like fields.

Timestamps were retained only for segmentation.

##5. Temporal windowing

Cyber Schema A:

Window size = {cyber_window["window_size"]} observations
Gap threshold = {float(cyber_window["gap_threshold_seconds"]):.6f} seconds
Total windows = {cyber_window["total_windows"]}
Exact duplicate windows = {cyber_window["duplicate_windows"]}
Conflicting window groups = {cyber_window["conflicting_feature_groups"]}

Physical Schema A:

Window size = {physical_window["window_size"]} observations
Gap threshold = {float(physical_window["gap_threshold_seconds"]):.6f} seconds
Total windows = {physical_window["total_windows"]}

Windows never cross timestamp resets or long temporal gaps.

##6. Frozen temporal split

Entire temporal segments were assigned to only one partition.

{markdown_table(split_summary)}

The split was chronological within each class.

The test partition remained locked during:

model selection,
feature selection,
calibration fitting,
routing threshold selection.
##7. Baseline modeling

Development baselines:

Logistic Regression
Random Forest
XGBoost

Macro F1 was the primary multiclass development metric.

##8. Physical confounding audit

Physical XGBoost initially achieved a validation Macro F1 of 1.0000.

Barometer validation ranges showed strong separation:

{markdown_table(barometer_ranges)}

Sensitivity results:

{markdown_table(
physical_sensitivity[
[
"configuration",
"features",
"accuracy",
"balanced_accuracy",
"macro_f1",
"log_loss",
]
]
)}

The perfect physical result was therefore not promoted as
general attack-detection evidence.

##9. Cyber feature reduction

Full temporal cyber representation:

{full_feature_count} aggregated features

Predefined selection rule:

Select the smallest tested source-family subset retaining at least
99% of full-model validation Macro F1.

Selected source families:

time_since_last_packet
wlan.fc.type

Selected aggregated features:

{selected_feature_count}

Retention:

{pct(feature_retention)}

##10. Hierarchical formulation

Stage 1:

Benign vs Attack

Stage 2:

DoS vs Replay

Stage-1 validation Attack F1:

{f4(stage1_val["f1_attack"])}

Stage-2 validation Macro F1:

{f4(stage2_val["macro_f1"])}

The weak Stage-2 result prevented attack-specific autonomous response.

##11. Calibration

Temperature scaling was fitted using grouped OOF probabilities from
the frozen Train set.

Temperature:

{temperature:.8f}

The calibration mapping was adopted because validation log loss
improved according to the predefined rule.

##12. Risk-aware routing

Uncertainty:

1 - max(P(Benign), P(Attack))

The validation threshold was selected to capture at least 75% of
prediction errors while minimizing verification burden.

Frozen uncertainty threshold:

{uncertainty_threshold:.6f}

##13. Locked Test policy

Before test evaluation, the model and all decision artifacts were frozen.

The test was then consumed once.

No post-test tuning is permitted.
"""

(PROJECT_DIR / "METHODOLOGY.md").write_text(
methodology.strip() + "\n",
encoding="utf-8"
)

print(
"=" * 100
)

print(
"PART 3 COMPLETE"
)

print(
    "=" * 100
)

print(
    "PART 3 COMPLETE"
)

print(
    "=" * 100
)

print(
    "✅ README.md"
)

print(
    "✅ METHODOLOGY.md"
)

In [ ]:
# ============================================================
# PART 4/5
# RESULTS + LIMITATIONS + MODEL CARD
# ============================================================


# ============================================================
# RESULTS.md
# ============================================================

results_md = f"""
# Results

## Primary Stage-1 Attack Detector

The final primary task is:

**Benign vs Attack**

### Validation vs Locked Test

| Metric | Validation | Locked Test |
|---|---:|---:|
| Accuracy | {f4(stage1_val["accuracy"])} | {f4(cal_test["accuracy"])} |
| Balanced Accuracy | {f4(stage1_val["balanced_accuracy"])} | {f4(cal_test["balanced_accuracy"])} |
| Attack Precision | {f4(stage1_val["precision_attack"])} | {f4(cal_test["attack_precision"])} |
| Attack Recall | {f4(stage1_val["recall_attack"])} | {f4(cal_test["attack_recall"])} |
| Attack F1 | {f4(stage1_val["f1_attack"])} | {f4(cal_test["attack_f1"])} |
| ROC-AUC | {f4(stage1_val["roc_auc"])} | {f4(cal_test["roc_auc"])} |
| PR-AUC | {f4(stage1_val["pr_auc"])} | {f4(cal_test["pr_auc"])} |

The Locked Test exposed a substantial temporal generalization gap.

Despite lower threshold-level recall, ranking ability remained stronger:

- ROC-AUC = **{f4(cal_test["roc_auc"])}**
- PR-AUC = **{f4(cal_test["pr_auc"])}**

![Validation vs Locked Test](results/figures/validation_vs_locked_test.png)

## Locked Test Confusion Matrix

{markdown_table(locked_cm.reset_index())}

Interpretation:

- 96 benign windows correctly classified
- 0 false-positive attacks
- 151 attacks correctly detected
- 108 attacks missed

![Confusion Matrix](results/figures/locked_test_confusion_matrix.png)

The detector therefore became highly conservative at the frozen
0.5 operating threshold.

## Calibration

Temperature scaling changed zero class predictions.

| Metric | Raw | Temperature Scaled |
|---|---:|---:|
| Log Loss | {f4(raw_test["log_loss"])} | {f4(cal_test["log_loss"])} |
| Brier | {f4(raw_test["brier"])} | {f4(cal_test["brier"])} |
| Equal-width ECE | {f4(raw_test["equal_width_ece_15"])} | {f4(cal_test["equal_width_ece_15"])} |
| Adaptive ECE | {f4(raw_test["adaptive_ece_15"])} | {f4(cal_test["adaptive_ece_15"])} |

All reported calibration-quality metrics improved on the Locked Test.

![Calibration](results/figures/locked_test_calibration_generalization.png)

## Risk-Aware Routing

Frozen test routing:

| Metric | Value |
|---|---:|
| Test windows | {int(routing_test["test_windows"])} |
| Base errors | {int(routing_test["base_errors"])} |
| Verification windows | {int(routing_test["review_windows"])} |
| Verification fraction | {pct(routing_test["review_fraction"])} |
| Errors captured | {int(routing_test["captured_errors"])} |
| Error capture | {pct(routing_test["error_capture"])} |
| Automated windows | {int(routing_test["automated_windows"])} |
| Automated fraction | {pct(routing_test["automated_fraction"])} |
| Automated accuracy | {pct(routing_test["automated_accuracy"])} |
| Automated Attack precision | {f4(routing_test["automated_attack_precision"])} |
| Automated Attack recall | {f4(routing_test["automated_attack_recall"])} |
| Automated Attack F1 | {f4(routing_test["automated_attack_f1"])} |

![Risk-Aware Routing](results/figures/locked_test_risk_aware_routing.png)

## Attack Attribution

Stage-2 DoS-vs-Replay validation:

- Accuracy = **{f4(stage2_val["accuracy"])}**
- Balanced Accuracy = **{f4(stage2_val["balanced_accuracy"])}**
- Macro F1 = **{f4(stage2_val["macro_f1"])}**
- ROC-AUC = **{f4(stage2_val["roc_auc"])}**

This result is insufficient for reliable autonomous attack-specific
response.

## Physical Branch

Sensitivity analysis:

{markdown_table(
    physical_sensitivity[
        [
            "configuration",
            "features",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "log_loss",
        ]
    ]
)}

The apparent perfect physical validation result was dominated by
barometer baseline information.

![Physical Confounding](results/figures/physical_barometer_confounding.png)

The physical branch is reported as a confounding case study.
"""


(PROJECT_DIR / "RESULTS.md").write_text(
    results_md.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# LIMITATIONS.md
# ============================================================

limitations = f"""
# Limitations

## Single Dataset

The study uses one UAV cyber-physical dataset.

The findings should not automatically be generalized to other UAV
platforms, environments, radios, or attack implementations.

## Heterogeneous Schemas

The raw dataset contains multiple incompatible cyber and physical
schemas.

A fair global five-class experiment was therefore not possible without
substantial schema-leakage risk.

## Temporal Generalization Gap

Stage-1 Attack Recall changed from:

**{pct(stage1_val["recall_attack"])} on validation**

to:

**{pct(cal_test["attack_recall"])} on the Locked Test**

This is the most important limitation of the final detector.

The frozen 0.5 operating point did not generalize reliably across
later temporal segments.

## Limited Independent Temporal Groups

The number of independent temporal segments is substantially smaller
than the number of windows.

Window-level sample counts should therefore not be interpreted as fully
independent experimental repetitions.

## Physical Confounding

The perfect physical validation result depended strongly on absolute
barometer level.

Removing the barometer caused performance to collapse.

This prevents interpretation of the physical model as a general UAV
attack detector.

## No Causal Barometer Claim

The analysis identifies a strong session/scenario confound.

It does not prove that barometer behavior can never be affected by
cyberattacks.

Independent controlled flight experiments would be required.

## Weak Attack Attribution

DoS-vs-Replay validation Macro F1:

**{f4(stage2_val["macro_f1"])}**

ROC-AUC:

**{f4(stage2_val["roc_auc"])}**

This is not sufficient for reliable autonomous attack-specific action.

## Confidence Is Not Full Epistemic Uncertainty

The routing policy uses calibrated predictive confidence.

It is not a complete Bayesian or epistemic uncertainty estimate.

## Review Burden

On the Locked Test, the frozen policy routed:

**{pct(routing_test["review_fraction"])}**

of windows for additional verification.

This may be operationally expensive.

## No Human Study

The verification workflow is simulated.

No analyst workload, response-time, or human accuracy experiment was
performed.

## No Live UAV Deployment

No autonomous intervention was executed on a real UAV.

The protective-response state is conceptual.

## No Post-Test Tuning

The Locked Test has already been consumed.

Any future methodological improvement must use a new external dataset
or newly collected test set.
"""


(PROJECT_DIR / "LIMITATIONS.md").write_text(
    limitations.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# MODEL_CARD.md
# ============================================================

model_card = f"""
# Model Card

## Model

Stage-1 UAV Cyber Attack Detector

Model family:

**XGBoost**

Task:

**Benign vs Attack**

where:

`Attack = DoS or Replay`

## Input

Fixed windows of 20 cyber observations.

Final source families:

- `time_since_last_packet`
- `wlan.fc.type`

Aggregated features:

**{selected_feature_count}**

## Probability Output

The detector produces:

`P(Attack)`

Temperature scaling:

**T = {temperature:.8f}**

## Frozen Operating Point

Attack if:

`P(Attack) >= 0.5`

Locked-Test performance:

| Metric | Value |
|---|---:|
| Accuracy | {f4(cal_test["accuracy"])} |
| Balanced Accuracy | {f4(cal_test["balanced_accuracy"])} |
| Attack Precision | {f4(cal_test["attack_precision"])} |
| Attack Recall | {f4(cal_test["attack_recall"])} |
| Attack F1 | {f4(cal_test["attack_f1"])} |
| ROC-AUC | {f4(cal_test["roc_auc"])} |
| PR-AUC | {f4(cal_test["pr_auc"])} |

## Risk-Aware Routing

Uncertainty:

`1 - max(P(Benign), P(Attack))`

Frozen threshold:

**{uncertainty_threshold:.6f}**

Equivalent confidence:

**{confidence_threshold:.6f}**

Actions:

| Condition | Action |
|---|---|
| High-confidence Benign | Continue monitoring |
| High-confidence Attack | Generic protective response |
| Low confidence | Verification required |

Locked-Test automated accuracy:

**{pct(routing_test["automated_accuracy"])}**

Locked-Test error capture:

**{pct(routing_test["error_capture"])}**

## Intended Use

Research on:

- UAV cybersecurity
- temporal distribution shift
- intrusion-detection reliability
- probability calibration
- uncertainty-aware routing

## Out-of-Scope Use

Do not treat this model as:

- a certified flight-safety system,
- a universal UAV IDS,
- an autonomous attack-specific controller,
- or a reliable DoS-vs-Replay attribution engine.

## Major Risk

The Locked Test showed substantial Attack Recall degradation.

A fixed 0.5 decision threshold can miss attacks under temporal shift.

## Evaluation Integrity

The Locked Test was evaluated once after freezing:

- selected features,
- model,
- temperature,
- routing threshold,
- response policy.

No post-test tuning is permitted.
"""


(PROJECT_DIR / "MODEL_CARD.md").write_text(
    model_card.strip() + "\n",
    encoding="utf-8"
)


print(
    "=" * 100
)

print(
    "PART 4 COMPLETE"
)

print(
    "=" * 100
)

print(
    "✅ RESULTS.md"
)

print(
    "✅ LIMITATIONS.md"
)

print(
    "✅ MODEL_CARD.md"
)

In [ ]:
# ============================================================
# PART 5/5
# REPOSITORY CLEANUP + FINAL QA
# ============================================================


# ============================================================
# 1. data/README.md
# ============================================================

data_readme = """
# Data

The raw UAV dataset is not committed to this repository.

Expected local location:

`data/raw/Dataset_T-ITS.csv`

## Important Structure

The source CSV contains several embedded cyber and physical sections
with different schemas.

The reconstructed sections are:

- cyber_benign
- physical_benign
- cyber_dos
- physical_dos
- cyber_replay
- physical_replay
- cyber_evil_twin
- physical_evil_twin
- cyber_fdi
- physical_fdi

The raw CSV must not be treated as one homogeneous machine-learning
table.

## Git Policy

Do not commit:

- raw datasets
- parsed row-level datasets
- window-level datasets
- split-level feature matrices
- large fitted models
- row-level prediction dumps

Commit:

- lightweight JSON configuration files
- audit summaries
- aggregate result tables
- figures
- research documentation
"""

(DATA_DIR / "README.md").write_text(
    data_readme.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 2. requirements.txt
# ============================================================

requirements = """
numpy>=1.24
pandas>=2.0
scipy>=1.10
scikit-learn>=1.3
xgboost>=2.0
matplotlib>=3.7
joblib>=1.3
"""

(PROJECT_DIR / "requirements.txt").write_text(
    requirements.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 3. .gitignore
# ============================================================

gitignore = """
# Raw data
data/raw/*
!data/raw/.gitkeep

# Large derived datasets
data/processed/sections/
data/processed/experiments/
data/processed/windows/
data/processed/splits/

# Large fitted models
models/development/*.joblib

# Row-level outputs
results/tables/*predictions.csv
results/tables/*routing.csv
results/tables/*threshold_curve.csv

# Notebook / Python cache
.ipynb_checkpoints/
__pycache__/
*.pyc
*.pyo

# OS / editors
.DS_Store
Thumbs.db
.vscode/
.idea/

# Temporary
*.tmp
*.temp
"""

(PROJECT_DIR / ".gitignore").write_text(
    gitignore.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 4. RAW PLACEHOLDER
# ============================================================

RAW_DIR = DATA_DIR / "raw"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

(RAW_DIR / ".gitkeep").touch()


# ============================================================
# 5. FINAL LIGHTWEIGHT RESEARCH SUMMARY
# ============================================================

final_research_summary = {

    "project":
        "Reliable UAV Cyber-Physical Security",

    "primary_task":
        "Benign vs Attack",

    "attack_classes":
        [
            "DoS",
            "Replay"
        ],

    "selected_source_features":
        selected_sources,

    "selected_aggregated_features":
        selected_feature_count,

    "temperature":
        temperature,

    "uncertainty_threshold":
        uncertainty_threshold,

    "confidence_threshold":
        confidence_threshold,

    "locked_test": {

        "accuracy":
            float(
                cal_test["accuracy"]
            ),

        "balanced_accuracy":
            float(
                cal_test["balanced_accuracy"]
            ),

        "attack_precision":
            float(
                cal_test["attack_precision"]
            ),

        "attack_recall":
            float(
                cal_test["attack_recall"]
            ),

        "attack_f1":
            float(
                cal_test["attack_f1"]
            ),

        "roc_auc":
            float(
                cal_test["roc_auc"]
            ),

        "pr_auc":
            float(
                cal_test["pr_auc"]
            ),
    },

    "calibration": {

        "raw_log_loss":
            float(
                raw_test["log_loss"]
            ),

        "scaled_log_loss":
            float(
                cal_test["log_loss"]
            ),

        "raw_brier":
            float(
                raw_test["brier"]
            ),

        "scaled_brier":
            float(
                cal_test["brier"]
            ),
    },

    "routing": {

        "review_fraction":
            float(
                routing_test[
                    "review_fraction"
                ]
            ),

        "error_capture":
            float(
                routing_test[
                    "error_capture"
                ]
            ),

        "automated_fraction":
            float(
                routing_test[
                    "automated_fraction"
                ]
            ),

        "automated_accuracy":
            float(
                routing_test[
                    "automated_accuracy"
                ]
            ),

        "automated_attack_f1":
            float(
                routing_test[
                    "automated_attack_f1"
                ]
            ),
    },

    "physical_branch":
        (
            "Confounding case study; "
            "not a general attack detector."
        ),

    "post_test_tuning":
        False,
}


SUMMARY_FILE = (
    TABLES_DIR
    / "final_research_summary.json"
)

with SUMMARY_FILE.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_research_summary,
        f,
        indent=2
    )


# ============================================================
# 6. HASH HELPER
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ============================================================
# 7. FINAL ARTIFACT MANIFEST
# ============================================================

important_files = [

    PROJECT_DIR / "README.md",
    PROJECT_DIR / "METHODOLOGY.md",
    PROJECT_DIR / "RESULTS.md",
    PROJECT_DIR / "LIMITATIONS.md",
    PROJECT_DIR / "MODEL_CARD.md",
    PROJECT_DIR / "requirements.txt",
    PROJECT_DIR / ".gitignore",

    DATA_DIR / "README.md",

    SUMMARY_FILE,

    FIG_VALIDATION_TEST,
    FIG_CM,
    FIG_CALIBRATION,
    FIG_ROUTING,
    FIG_PHYSICAL,
    FIG_REDUCTION,
]


artifact_manifest = {

    "generated_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "locked_test_consumed":
        True,

    "post_test_model_changes":
        False,

    "artifacts":
        [],
}


for path in important_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing final artifact:\n{path}"
        )

    artifact_manifest[
        "artifacts"
    ].append({

        "path":
            str(
                path.relative_to(
                    PROJECT_DIR
                )
            ),

        "sha256":
            sha256_file(path),

        "bytes":
            path.stat().st_size,
    })


MANIFEST_FILE = (
    RESULTS_DIR
    / "final_research_artifact_manifest.json"
)

with MANIFEST_FILE.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact_manifest,
        f,
        indent=2
    )


# ============================================================
# 8. FINAL DOCUMENTATION QA
# ============================================================

docs = [

    PROJECT_DIR / "README.md",
    PROJECT_DIR / "METHODOLOGY.md",
    PROJECT_DIR / "RESULTS.md",
    PROJECT_DIR / "LIMITATIONS.md",
    PROJECT_DIR / "MODEL_CARD.md",
    PROJECT_DIR / "requirements.txt",
    PROJECT_DIR / ".gitignore",
    DATA_DIR / "README.md",
]


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL DOCUMENTATION QA"
)

print(
    "=" * 100
)


for path in docs:

    assert path.exists()

    print(
        "✅",
        path.relative_to(
            PROJECT_DIR
        )
    )


# ============================================================
# 9. FINAL FIGURE QA
# ============================================================

figures = [

    FIG_VALIDATION_TEST,
    FIG_CM,
    FIG_CALIBRATION,
    FIG_ROUTING,
    FIG_PHYSICAL,
    FIG_REDUCTION,
]


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL FIGURE QA"
)

print(
    "=" * 100
)


for path in figures:

    assert path.exists()

    print(
        "✅",
        path.name
    )


# ============================================================
# 10. FINAL RESEARCH SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL RESEARCH PACKAGE COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nPRIMARY TASK"
)

print(
    "Benign vs Attack"
)


print(
    "\nSELECTED SOURCES"
)

for source in selected_sources:

    print(
        "-",
        source
    )


print(
    "\nSELECTED AGGREGATED FEATURES:"
)

print(
    selected_feature_count
)


print(
    "\nLOCKED TEST"
)

print(
    "Accuracy        :",
    f4(
        cal_test["accuracy"]
    )
)

print(
    "Balanced Acc.   :",
    f4(
        cal_test[
            "balanced_accuracy"
        ]
    )
)

print(
    "Attack Precision:",
    f4(
        cal_test[
            "attack_precision"
        ]
    )
)

print(
    "Attack Recall   :",
    f4(
        cal_test[
            "attack_recall"
        ]
    )
)

print(
    "Attack F1       :",
    f4(
        cal_test[
            "attack_f1"
        ]
    )
)

print(
    "ROC-AUC         :",
    f4(
        cal_test[
            "roc_auc"
        ]
    )
)

print(
    "PR-AUC          :",
    f4(
        cal_test[
            "pr_auc"
        ]
    )
)


print(
    "\nCALIBRATION"
)

print(
    "Log Loss:",
    f4(
        raw_test[
            "log_loss"
        ]
    ),
    "→",
    f4(
        cal_test[
            "log_loss"
        ]
    )
)

print(
    "Brier:",
    f4(
        raw_test[
            "brier"
        ]
    ),
    "→",
    f4(
        cal_test[
            "brier"
        ]
    )
)


print(
    "\nRISK-AWARE ROUTING"
)

print(
    "Review fraction    :",
    pct(
        routing_test[
            "review_fraction"
        ]
    )
)

print(
    "Error capture      :",
    pct(
        routing_test[
            "error_capture"
        ]
    )
)

print(
    "Automated fraction :",
    pct(
        routing_test[
            "automated_fraction"
        ]
    )
)

print(
    "Automated accuracy :",
    pct(
        routing_test[
            "automated_accuracy"
        ]
    )
)

print(
    "Automated Attack F1:",
    f4(
        routing_test[
            "automated_attack_f1"
        ]
    )
)


print(
    "\nPHYSICAL BRANCH"
)

print(
    "✅ Preserved as confounding case study."
)

print(
    "❌ Not reported as a perfect general detector."
)


print(
    "\nFINAL FILES"
)

print(
    "Summary :",
    SUMMARY_FILE
)

print(
    "Manifest:",
    MANIFEST_FILE
)


print(
    "\n🔒 NO MODEL RETRAINING"
)

print(
    "🔒 NO POST-TEST TUNING"
)

print(
    "🔒 LOCKED TEST REMAINS FINAL"
)

In [ ]:
# ============================================================
# GITHUB PRE-PUSH AUDIT
# Step 1: Notebook discovery + repository safety check
# ============================================================

from pathlib import Path
import os
import subprocess


PROJECT_DIR = Path(
    "/content/drive/MyDrive/uav project/"
    "reliable-uav-cyber-physical-security"
)

NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"

NOTEBOOKS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. BASIC PROJECT CHECK
# ============================================================

print("=" * 100)
print("PROJECT CHECK")
print("=" * 100)

print("Project:")
print(PROJECT_DIR)

print("\nExists:", PROJECT_DIR.exists())

if not PROJECT_DIR.exists():
    raise FileNotFoundError(PROJECT_DIR)


# ============================================================
# 2. FIND NOTEBOOKS NEAR THIS PROJECT
# ============================================================

SEARCH_ROOT = Path(
    "/content/drive/MyDrive/uav project"
)

notebooks = sorted(
    SEARCH_ROOT.rglob("*.ipynb"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

print("\n" + "=" * 100)
print("NOTEBOOK CANDIDATES — NEWEST FIRST")
print("=" * 100)

if not notebooks:

    print("⚠️ No .ipynb files found under:")
    print(SEARCH_ROOT)

else:

    for i, path in enumerate(
        notebooks[:20],
        start=1
    ):

        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{i:02d}. "
            f"{path} "
            f"({size_mb:.2f} MB)"
        )


# ============================================================
# 3. CHECK WHETHER PROJECT IS A GIT REPOSITORY
# ============================================================

def run_git(*args):

    result = subprocess.run(
        ["git", *args],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )

    return (
        result.returncode,
        result.stdout.strip(),
        result.stderr.strip()
    )


code, out, err = run_git(
    "rev-parse",
    "--is-inside-work-tree"
)

print("\n" + "=" * 100)
print("GIT REPOSITORY CHECK")
print("=" * 100)

if code == 0:

    print("✅ Git repository:", out)

else:

    print("⚠️ Not initialized as Git repository yet.")


# ============================================================
# 4. CURRENT GIT STATUS
# ============================================================

if code == 0:

    _, status, _ = run_git(
        "status",
        "--short"
    )

    print("\n" + "=" * 100)
    print("CURRENT GIT STATUS")
    print("=" * 100)

    if status:

        print(status)

    else:

        print("✅ Working tree currently clean.")


# ============================================================
# 5. FILE SAFETY AUDIT
# ============================================================

print("\n" + "=" * 100)
print("FILES THAT REQUIRE ATTENTION")
print("=" * 100)

risky_extensions = {
    ".csv",
    ".joblib",
    ".pkl",
    ".pickle",
    ".parquet",
    ".zip",
}

risky_files = []

for path in PROJECT_DIR.rglob("*"):

    if not path.is_file():
        continue

    relative = path.relative_to(
        PROJECT_DIR
    )

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    if (
        path.suffix.lower()
        in risky_extensions
        or size_mb >= 10
    ):

        risky_files.append(
            (
                str(relative),
                size_mb
            )
        )


if risky_files:

    for relative, size_mb in sorted(
        risky_files,
        key=lambda x: x[1],
        reverse=True
    ):

        print(
            f"{size_mb:9.2f} MB  "
            f"{relative}"
        )

else:

    print(
        "✅ No obvious large/risky files found."
    )


# ============================================================
# 6. CHECK IGNORE STATUS OF IMPORTANT PRIVATE/LARGE PATHS
# ============================================================

if code == 0:

    test_paths = [
        "data/raw/Dataset_T-ITS.csv",
        "data/processed/windows/cyber_schema_A_3class_windows.csv",
        "data/processed/splits/cyber_schema_A_3class_train.csv",
        "models/development/cyber_stage1_attack_detector.joblib",
        "results/tables/cyber_stage1_locked_test_predictions.csv",
    ]

    print("\n" + "=" * 100)
    print("GITIGNORE SAFETY CHECK")
    print("=" * 100)

    for rel in test_paths:

        result = subprocess.run(
            [
                "git",
                "check-ignore",
                "-v",
                rel
            ],
            cwd=PROJECT_DIR,
            capture_output=True,
            text=True
        )

        if result.returncode == 0:

            print(
                "✅ IGNORED:",
                rel
            )

        else:

            print(
                "⚠️ NOT IGNORED:",
                rel
            )


# ============================================================
# 7. TOP-LEVEL PROJECT STRUCTURE
# ============================================================

print("\n" + "=" * 100)
print("TOP-LEVEL PROJECT STRUCTURE")
print("=" * 100)

for path in sorted(
    PROJECT_DIR.iterdir()
):

    if path.name == ".git":
        continue

    if path.is_dir():
        print(
            "📁",
            path.name + "/"
        )

    else:
        print(
            "📄",
            path.name
        )


print("\n" + "=" * 100)
print("PRE-PUSH AUDIT COMPLETE")
print("=" * 100)

print(
    "\n🔒 Nothing was staged or pushed."
)